In [1]:
from __future__ import annotations
from pygam import PoissonGAM, s, f
import warnings
import os
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from patsy.builtins import bs
from scipy.special import gammaln

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
np.set_printoptions(suppress=True, linewidth=140)

try:
    import duckdb
    HAS_DUCKDB = True
except Exception:
    HAS_DUCKDB = False

try:
    import pyarrow  # noqa: F401
    HAS_PYARROW = True
except Exception:
    HAS_PYARROW = False

try:
    from sklearn.ensemble import (
        ExtraTreesRegressor,
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )
    from sklearn.base import clone
    from sklearn.inspection import permutation_importance
    from sklearn.linear_model import PoissonRegressor
    from sklearn.model_selection import KFold, train_test_split
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False

print("duckdb available :", HAS_DUCKDB)
print("pyarrow available:", HAS_PYARROW)
print("sklearn available:", HAS_SKLEARN)

duckdb available : True
pyarrow available: True
sklearn available: True


---

# Session 1 · Data overview and scope

In [2]:
# SESSION 1.1 SHARED CONTROL PANEL

DATA_PATH = Path("script/data/parquet/full.parquet")


# Core scope filters
OBSERVATION_YEARS = None          # Example: [2018, 2019]; use None to keep all years
ISSUE_AGE_MIN = 0
ISSUE_AGE_MAX = 17
ATTAINED_AGE_MIN = 1
ATTAINED_AGE_MAX = 67
DURATION_MIN = 1
DURATION_CAP = 119

AGE_IND_KEEP = None               # Example: ["0"] or ["1"]; None keeps all age bases
SEX_KEEP = None                   # Example: ["M"] or ["F"]; None keeps all sexes
SMOKER_STATUS_KEEP = ["U"]        # Example: ["U"]; set to None to keep all
INSURANCE_PLAN_DROP = ["Other"]   # Set to [] to keep "Other"

EXCLUDE_POST_LEVEL_TERM = True
POST_LEVEL_TERM_EXCLUDE_VALUES = ["PLT"]


# 1M face-amount cap option
CAP_FACE_AT_1M = True
FACE_CAP_VALUE = 1_000_000
FACE_CAP_SOURCE_BANDS = [
    "08: 1,000,000 - 2,499,999",
    "09: 2,500,000 - 4,999,999",
    "10: 5,000,000 - 9,999,999",
    "11: 10,000,000+",
]
FACE_CAP_TARGET_BAND = "08: 1,000,000+"


# Age-basis helper columns used for testing / modeling
ISSUE_AGE_MOD_COL = "Issue_Age_mod"
ATTAINED_AGE_MOD_COL = "Attained_Age_mod"
ISSUE_AGE_MOD_MAX = ISSUE_AGE_MAX + 0.5
ATTAINED_AGE_MOD_MAX = ATTAINED_AGE_MAX + 0.5

# Shared output folder
EXPORT_DIR = Path("script/model_outputs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# SESSION 1.2 OPTIONAL CONTROL PANEL

# Optional for summary work only.
COLLAPSE_DURATION_26_PLUS = False
# Optional maturity-normalized exposure scaling for later issue years for descriptive work.
APPLY_ISSUE_YEAR_EXPOSURE_SCALING = True
ISSUE_YEAR_EXPOSURE_TARGET_YEARS = 7
ISSUE_YEAR_EXPOSURE_FULL_CREDIT_START = 2013
ISSUE_YEAR_EXPOSURE_SCALE_START = 2014
ISSUE_YEAR_EXPOSURE_SCALE_END = 2019


In [4]:
# SESSION 1.3 NUMERIC AND CATEGORICAL COLUMNS

ACTUAL_CNT_COL = "Death_Count"
EXPECTED_CNT_COL = "ExpDth_VBT2015_Cnt"
ACTUAL_AMT_COL = "Death_Claim_Amount"
EXPECTED_AMT_COL = "ExpDth_VBT2015_Amt"

NUMERIC_COLS = [
    "Observation_Year", "Issue_Age", ISSUE_AGE_MOD_COL, "Duration", "Issue_Year", "Attained_Age", ATTAINED_AGE_MOD_COL,
    "Amount_Exposed", "Policies_Exposed", ACTUAL_AMT_COL, ACTUAL_CNT_COL,
    EXPECTED_CNT_COL, EXPECTED_AMT_COL
]

RAW_CATEGORICAL_COLS = [
    "Age_Ind", "Sex", "Smoker_Status", "Insurance_Plan", "Face_Amount_Band",
    "SOA_Antp_Lvl_TP", "SOA_Guar_Lvl_TP", "SOA_Post_Lvl_Ind", "Slct_Ult_Ind",
    "Preferred_Indicator", "Number_of_Pfd_Classes", "Preferred_Class"
]

DERIVED_CATEGORICAL_COLS = [
    "Insurance_Plan_Group",
    "Face_Band_Group",
    "Issue_Age_Band",
    "Attained_Age_Band",
    "Duration_Band",
]

CATEGORICAL_COLS = RAW_CATEGORICAL_COLS + DERIVED_CATEGORICAL_COLS
BASE_COLUMNS = [
    "Observation_Year",
    "Issue_Age",
    "Duration",
    "Issue_Year",
    "Attained_Age",
    "Amount_Exposed",
    "Policies_Exposed",
    ACTUAL_AMT_COL,
    ACTUAL_CNT_COL,
    EXPECTED_CNT_COL,
    EXPECTED_AMT_COL,
] + RAW_CATEGORICAL_COLS


In [5]:
# SESSION 1.4 SAMPLE CONTROLS AND GROUPING

# Development sampling controls
DEVELOPMENT_SAMPLE_FRACTION = None     # Example: 0.30 for a 30% sample
DEVELOPMENT_SAMPLE_STRATA = ["Observation_Year", "Age_Ind", "Sex", "Issue_Age_Band", "Duration_Band"]
DEVELOPMENT_SAMPLE_RANDOM_STATE = 42


# Grouping choices for challenger diagnostics
PLAN_GROUP_MAP = {
    "Perm": "Perm",
    "Term": "Term",
    "UL": "Perm",
    "ULSG": "Perm",
    "VL": "Perm",
    "VLSG": "Perm",
    "Other": "Other/Unknown",
    "Missing": "Missing",
}

FACE_BAND_GROUP_MAP = {
    "01: 0 - 9,999": "01-04: <100k",
    "02: 10,000 - 24,999": "01-04: <100k",
    "03: 25,000 - 49,999": "01-04: <100k",
    "04: 50,000 - 99,999": "01-04: <100k",
    "05: 100,000 - 249,999": "05-011: >100k",
    "06: 250,000 - 499,999": "05-011: >100k",
    "07: 500,000 - 999,999": "05-011: >100k",
    "08: 1,000,000 - 2,499,999": "05-011: >100k",
    "08: 1,000,000+": "05-011: >100k",
    "09: 2,500,000 - 4,999,999": "05-011: >100k",
    "10: 5,000,000 - 9,999,999": "05-011: >100k",
    "11: 10,000,000+": "05-011: >100k",
    "Missing": "Missing",
}


In [22]:
# SESSION 1.5 HELPER FUNCTIONS

# SESSION 1.5.1 DEPENDENCY CHECK HELPERS
# Checks that packages are available before any parquet file read and running model code.
def require_parquet_support():
    if HAS_DUCKDB or HAS_PYARROW:
        return
    raise ImportError(
        "This notebook needs either duckdb or pyarrow to read parquet. "
        "Install one of them locally, for example: pip install duckdb or pip install pyarrow"
    )
def require_sklearn():
    if HAS_SKLEARN:
        return
    raise ImportError(
        "This notebook needs scikit-learn for Sessions 2 and 3. "
        "Install it locally, for example: pip install scikit-learn"
    )


# SESSION 1.5.2 DATA READ AND EXPORT HELPERS
# Reads selected columns from the parquet source.
def read_parquet_data(path: Path, columns: list[str] | None = None) -> pd.DataFrame:
    require_parquet_support()
    if not path.exists():
        raise FileNotFoundError(f"Parquet file not found: {path}")
    if HAS_DUCKDB:
        cols_sql = "*" if columns is None else ", ".join([f'"{c}"' for c in columns])
        query = f"SELECT {cols_sql} FROM read_parquet('{path.as_posix()}')"
        return duckdb.sql(query).df()
    return pd.read_parquet(path, columns=columns)

# Writes the tables to EXPORT_DIR and prints the saved path.
def export_csv(df: pd.DataFrame, filename: str):
    path = EXPORT_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved: {path}")


# SESSION 1.5.3 FORMAT, MATH, AND METRIC HELPERS
# Forces matplotlib axes to show regular decimal labels instead of scientific notation.
def force_plain_axis(ax, axis: str = "both"):
    formatter = mtick.ScalarFormatter(useOffset=False)
    formatter.set_scientific(False)
    if axis in ("x", "both"):
        ax.xaxis.set_major_formatter(formatter)
    if axis in ("y", "both"):
        ax.yaxis.set_major_formatter(formatter)

# Divides scalars or arrays while returning NaN instead of inf when the denominator is effectively zero.
def safe_divide(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    out = np.divide(
        numerator,
        denominator,
        out=np.full_like(numerator, np.nan, dtype=float),
        where=np.abs(denominator) > 1e-12,
    )
    if np.ndim(out) == 0:
        return float(out)
    if out.size == 1:
        return float(np.ravel(out)[0])
    return out

# Computes Poisson deviance with predictions clipped above zero.
def safe_poisson_deviance(y_true, y_pred) -> float:
    y = np.asarray(y_true, dtype=float)
    mu = np.clip(np.asarray(y_pred, dtype=float), 1e-12, None)
    term = np.where(y == 0, 0.0, y * np.log(np.clip(y, 1e-12, None) / mu))
    return float(2.0 * np.sum(term - (y - mu)))

# Computes expected-weighted absolute error between actual A/E and predicted A/E.
def weighted_ae_error(actual, predicted, expected) -> float:
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    expected = np.asarray(expected, dtype=float)

    actual_ae = actual / np.clip(expected, 1e-12, None)
    pred_ae = predicted / np.clip(expected, 1e-12, None)
    return float(np.average(np.abs(actual_ae - pred_ae), weights=np.clip(expected, 1e-12, None)))

# Computes a positive weighted mean and returns NaN when no credible weights remain.
def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    keep = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if keep.sum() == 0:
        return np.nan
    return float(np.sum(x[keep] * w[keep]) / np.sum(w[keep]))

# Computes a positive weighted Pearson correlation.
def weighted_corr(x, y, w):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    keep = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0)
    if keep.sum() <= 1:
        return np.nan

    x = x[keep]
    y = y[keep]
    w = w[keep]
    mx = np.sum(w * x) / np.sum(w)
    my = np.sum(w * y) / np.sum(w)
    cov_xy = np.sum(w * (x - mx) * (y - my)) / np.sum(w)
    var_x = np.sum(w * (x - mx) ** 2) / np.sum(w)
    var_y = np.sum(w * (y - my) ** 2) / np.sum(w)

    if (var_x <= 0) or (var_y <= 0):
        return np.nan
    return float(cov_xy / np.sqrt(var_x * var_y))


# SESSION 1.5.4 DATA SCOPE AND FEATURE ENGINEERING HELPERS

# Standardaizes configured numeric and categorical columns into consistent analysis dtypes.
def standardize_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in NUMERIC_COLS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    for col in CATEGORICAL_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("Missing")
    return out

# Creates Issue_Age_mod and Attained_Age_mod so ALB records use raw age + 0.5 and ANB records use raw age.
def derive_age_basis_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    age_ind = out["Age_Ind"].astype("string").fillna("Missing")
    out[ISSUE_AGE_MOD_COL] = pd.to_numeric(out["Issue_Age"], errors="coerce")
    out[ATTAINED_AGE_MOD_COL] = pd.to_numeric(out["Attained_Age"], errors="coerce")

    alb_mask = age_ind == "1"   # ALB
    anb_mask = age_ind == "0"   # ANB
    out.loc[alb_mask, ISSUE_AGE_MOD_COL] = out.loc[alb_mask, "Issue_Age"] + 0.5
    out.loc[alb_mask, ATTAINED_AGE_MOD_COL] = out.loc[alb_mask, "Attained_Age"] + 0.5
    out.loc[anb_mask, ISSUE_AGE_MOD_COL] = out.loc[anb_mask, "Issue_Age"]
    out.loc[anb_mask, ATTAINED_AGE_MOD_COL] = out.loc[anb_mask, "Attained_Age"]
    return out

# Optionally scales exposure for later issue years.
def apply_issue_year_exposure_scaling(df: pd.DataFrame) -> pd.DataFrame:
    if not APPLY_ISSUE_YEAR_EXPOSURE_SCALING:
        return df.copy()
    out = df.copy()
    issue_year = pd.to_numeric(out["Issue_Year"], errors="coerce")
    eligible = issue_year.between(ISSUE_YEAR_EXPOSURE_SCALE_START, ISSUE_YEAR_EXPOSURE_SCALE_END, inclusive="both")

    observed_years = ISSUE_YEAR_EXPOSURE_SCALE_END - issue_year + 1
    scale = np.where(
        eligible,
        ISSUE_YEAR_EXPOSURE_TARGET_YEARS / np.clip(observed_years, 1, None),
        1.0,
    )
    out["Policies_Exposed"] = pd.to_numeric(out["Policies_Exposed"], errors="coerce") * scale
    out["Amount_Exposed"] = pd.to_numeric(out["Amount_Exposed"], errors="coerce") * scale
    return out

# Caps selected face amount bands at 1M and adjusts amount-based actual, exposed, and expected values.
def apply_face_amount_cap_1m(df: pd.DataFrame) -> pd.DataFrame:
    if not CAP_FACE_AT_1M:
        return df.copy()
    out = df.copy()
    mask = out["Face_Amount_Band"].isin(FACE_CAP_SOURCE_BANDS)

    out.loc[mask, "Face_Amount_Band"] = FACE_CAP_TARGET_BAND
    out.loc[mask, ACTUAL_AMT_COL] = FACE_CAP_VALUE * out.loc[mask, ACTUAL_CNT_COL]
    out.loc[mask, "Amount_Exposed"] = FACE_CAP_VALUE * out.loc[mask, "Policies_Exposed"]
    out.loc[mask, EXPECTED_AMT_COL] = FACE_CAP_VALUE * out.loc[mask, EXPECTED_CNT_COL]
    return out

# Adds grouped plan and face-band variables.
def add_grouped_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "Insurance_Plan" in out.columns:
        out["Insurance_Plan_Group"] = (
            out["Insurance_Plan"]
            .astype("string")
            .fillna("Missing")
            .map(PLAN_GROUP_MAP)
            .fillna(out["Insurance_Plan"].astype("string").fillna("Missing"))
            .astype("string")
        )

    if "Face_Amount_Band" in out.columns:
        out["Face_Band_Group"] = (
            out["Face_Amount_Band"]
            .astype("string")
            .fillna("Missing")
            .map(FACE_BAND_GROUP_MAP)
            .fillna(out["Face_Amount_Band"].astype("string").fillna("Missing"))
            .astype("string")
        )
    return out

# Adds issue-age, attained-age, and duration bands used for validation and review tables.
def add_validation_bands(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    issue_age_series = out[ISSUE_AGE_MOD_COL] if ISSUE_AGE_MOD_COL in out.columns else out["Issue_Age"]
    attained_age_series = out[ATTAINED_AGE_MOD_COL] if ATTAINED_AGE_MOD_COL in out.columns else out["Attained_Age"]
    out["Issue_Age_Band"] = pd.cut(
        issue_age_series,
        bins=[-0.1, 0.5, 5.5, 10.5, 15.5, 17.5],
        labels=["0", "1-5", "6-10", "11-15", "16-17"],
        include_lowest=True,
        right=True,
    )
    attained_min = int(np.floor(attained_age_series.min())) if len(out) else 0
    attained_floor = max(0, (attained_min // 5) * 5)
    attained_bins = list(range(attained_floor, 101, 5))
    if attained_bins[-1] < 100:
        attained_bins.append(100)
    out["Attained_Age_Band"] = pd.cut(
        attained_age_series,
        bins=attained_bins,
        include_lowest=True,
        right=False,
    )
    out["Duration_Band"] = pd.cut(
        np.where(out["Duration"] >= 26, 26, out["Duration"]),
        bins=[-0.1, 1, 5, 10, 15, 20, 26],
        labels=["0-1", "2-5", "6-10", "11-15", "16-20", "21-26+"],
        include_lowest=True,
        right=True,
    )
    return out

# Applies all data filters and derived rules, while recording an audit log.
def apply_scope_filters(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = standardize_frame(df)
    steps = [{"step": "raw_input", "rows": len(out)}]

    if OBSERVATION_YEARS is not None:
        out = out[out["Observation_Year"].isin(OBSERVATION_YEARS)].copy()
        steps.append({"step": f"Observation_Year in {OBSERVATION_YEARS}", "rows": len(out)})

    out = out[
        out["Issue_Age"].between(ISSUE_AGE_MIN, ISSUE_AGE_MAX, inclusive="both")
    ].copy()
    steps.append({"step": f"Issue_Age between {ISSUE_AGE_MIN} and {ISSUE_AGE_MAX}", "rows": len(out)})

    if ATTAINED_AGE_MIN is not None:
        out = out[out["Attained_Age"] >= ATTAINED_AGE_MIN].copy()
        steps.append({"step": f"Attained_Age >= {ATTAINED_AGE_MIN}", "rows": len(out)})

    if ATTAINED_AGE_MAX is not None:
        out = out[out["Attained_Age"] <= ATTAINED_AGE_MAX].copy()
        steps.append({"step": f"Attained_Age <= {ATTAINED_AGE_MAX}", "rows": len(out)})

    if AGE_IND_KEEP is not None:
        out = out[out["Age_Ind"].isin(AGE_IND_KEEP)].copy()
        steps.append({"step": f"Age_Ind in {AGE_IND_KEEP}", "rows": len(out)})

    if SEX_KEEP is not None:
        out = out[out["Sex"].isin(SEX_KEEP)].copy()
        steps.append({"step": f"Sex in {SEX_KEEP}", "rows": len(out)})

    if SMOKER_STATUS_KEEP is not None:
        out = out[out["Smoker_Status"].isin(SMOKER_STATUS_KEEP)].copy()
        steps.append({"step": f"Smoker_Status in {SMOKER_STATUS_KEEP}", "rows": len(out)})

    if INSURANCE_PLAN_DROP:
        out = out[~out["Insurance_Plan"].isin(INSURANCE_PLAN_DROP)].copy()
        steps.append({"step": f"Insurance_Plan not in {INSURANCE_PLAN_DROP}", "rows": len(out)})

    if EXCLUDE_POST_LEVEL_TERM:
        out = out[~out["SOA_Post_Lvl_Ind"].isin(POST_LEVEL_TERM_EXCLUDE_VALUES)].copy()
        steps.append({"step": f"SOA_Post_Lvl_Ind not in {POST_LEVEL_TERM_EXCLUDE_VALUES}", "rows": len(out)})

    if COLLAPSE_DURATION_26_PLUS:
        out["Duration"] = np.where(out["Duration"] >= 26, 26, out["Duration"])
        steps.append({"step": "Duration >= 26 collapsed to 26", "rows": len(out)})

    out = out[out[EXPECTED_CNT_COL] > 0].copy()
    steps.append({"step": f"{EXPECTED_CNT_COL} > 0", "rows": len(out)})

    out = apply_face_amount_cap_1m(out)
    if CAP_FACE_AT_1M:
        steps.append({"step": "Applied 1M face cap rule", "rows": len(out)})

    out = derive_age_basis_columns(out)
    steps.append({"step": "Added Issue_Age_mod / Attained_Age_mod (ALB = raw + 0.5; ANB = raw)", "rows": len(out)})

    out = apply_issue_year_exposure_scaling(out)
    if APPLY_ISSUE_YEAR_EXPOSURE_SCALING:
        steps.append({"step": "Scaled Policies_Exposed / Amount_Exposed for later issue years to a 7-year maturity basis", "rows": len(out)})

    out = add_grouped_features(out)
    out = add_validation_bands(out)
    steps.append({"step": "Added grouped plan / face-band fields and validation bands", "rows": len(out)})
    return out.reset_index(drop=True), pd.DataFrame(steps)

# Takes a reproducible development sample with stratified sampling.
def apply_optional_development_sample(
    df: pd.DataFrame,
    fraction: float | None = None,
    strata: list[str] | None = None,
    random_state: int = 42,
) -> pd.DataFrame:
    if fraction is None:
        return df.copy()
    if not (0 < fraction <= 1):
        raise ValueError("DEVELOPMENT_SAMPLE_FRACTION must be in (0, 1].")
    if math.isclose(fraction, 1.0):
        return df.copy()
    work = df.copy()
    strata = [c for c in (strata or []) if c in work.columns]
    if not strata:
        return work.sample(frac=fraction, random_state=random_state).reset_index(drop=True)
    pieces = []
    for _, sub in work.groupby(strata, dropna=False, sort=False):
        n_take = max(1, int(round(len(sub) * fraction)))
        n_take = min(n_take, len(sub))
        pieces.append(sub.sample(n=n_take, random_state=random_state))
    sampled = pd.concat(pieces, ignore_index=True)
    return sampled.reset_index(drop=True)


# SESSION 1.5.5 TRAIN / HOLDOUT SPLIT HELPERS
# Builds buckets for numeric measures used in train/holdout split.
def make_balance_bucket(series: pd.Series, q: int = 5, zero_label: str | None = None) -> pd.Series:
    """Return stable, readable buckets for split stratification."""
    x = pd.to_numeric(series, errors="coerce")

    if zero_label is not None:
        out = pd.Series("positive", index=series.index, dtype="object")
        out[x.fillna(0) <= 0] = zero_label
        positive = x[x > 0]
        if positive.nunique(dropna=True) > 1:
            labels = pd.qcut(
                positive.rank(method="first"),
                q=min(q, positive.nunique(dropna=True)),
                labels=False,
                duplicates="drop",
            ).astype("Int64").astype(str)
            out.loc[positive.index] = "pos_q" + labels
        return out.fillna("Missing").astype(str)

    if x.nunique(dropna=True) <= 1:
        return pd.Series("all", index=series.index, dtype="object")

    labels = pd.qcut(
        x.rank(method="first"),
        q=min(q, x.nunique(dropna=True)),
        labels=False,
        duplicates="drop",
    ).astype("Int64").astype(str)
    return ("q" + labels).fillna("Missing").astype(str)

# Combines configured strata columns with exposure and death buckets, collapsing rare cells to a fallback label.
def build_split_strata(df: pd.DataFrame, strata_cols: list[str], min_count: int) -> pd.Series:
    """Build a practical split key and collapse rare cells so sklearn can stratify."""
    work = df.copy()
    present = [c for c in strata_cols if c in work.columns]
    parts = []
    for col in present:
        parts.append(work[col].astype("string").fillna("Missing").astype(str))
    parts.append(make_balance_bucket(work["Policies_Exposed"], q=5).rename("Exposure_Bucket"))
    parts.append(make_balance_bucket(work[ACTUAL_CNT_COL], q=4, zero_label="death_0").rename("Death_Bucket"))
    key = parts[0]
    for part in parts[1:]:
        key = key + " | " + part
    counts = key.value_counts(dropna=False)
    rare = counts[counts < min_count].index
    return key.where(~key.isin(rare), "Other / thin split cells")

# Compares train versus full-scope exposure and deaths by review segment and labels thin or imbalanced cells.
def split_balance_table(
    full_df: pd.DataFrame,
    train_df: pd.DataFrame,
    score_df: pd.DataFrame,
    review_cols: list[str],
    min_ratio: float = 0.75,
    max_ratio: float = 0.85,
    min_full_exposure: float = 100.0,
    min_full_deaths: float = 5.0,
) -> pd.DataFrame:
    """Check whether train holds roughly 80% of exposure and deaths by segment."""
    rows = []
    metric_cols = ["Policies_Exposed", ACTUAL_CNT_COL]

    for col in [c for c in review_cols if c in full_df.columns]:
        full = full_df.groupby(col, dropna=False)[metric_cols].sum().rename(
            columns={"Policies_Exposed": "Full_Exposure", ACTUAL_CNT_COL: "Full_Deaths"}
        )
        train = train_df.groupby(col, dropna=False)[metric_cols].sum().rename(
            columns={"Policies_Exposed": "Train_Exposure", ACTUAL_CNT_COL: "Train_Deaths"}
        )
        score = score_df.groupby(col, dropna=False)[metric_cols].sum().rename(
            columns={"Policies_Exposed": "Holdout_Exposure", ACTUAL_CNT_COL: "Holdout_Deaths"}
        )
        counts = pd.DataFrame({
            "Full_Rows": full_df.groupby(col, dropna=False).size(),
            "Train_Rows": train_df.groupby(col, dropna=False).size(),
            "Holdout_Rows": score_df.groupby(col, dropna=False).size(),
        })

        check = full.join(train, how="left").join(score, how="left").join(counts, how="left").fillna(0).reset_index()
        check = check.rename(columns={col: "Level"})
        check.insert(0, "Dimension", col)
        check["Train_Exposure_Ratio"] = safe_divide(check["Train_Exposure"], check["Full_Exposure"])
        check["Train_Death_Ratio"] = safe_divide(check["Train_Deaths"], check["Full_Deaths"])

        thin_exposure = check["Full_Exposure"] < min_full_exposure
        thin_deaths = check["Full_Deaths"] < min_full_deaths
        exposure_ok = check["Train_Exposure_Ratio"].between(min_ratio, max_ratio, inclusive="both")
        death_ok = check["Train_Death_Ratio"].between(min_ratio, max_ratio, inclusive="both")

        check["Exposure_Check"] = np.select(
            [thin_exposure, exposure_ok],
            ["Thin data", "OK"],
            default="Review",
        )
        check["Death_Check"] = np.select(
            [check["Full_Deaths"] == 0, thin_deaths, death_ok],
            ["No deaths", "Thin data", "OK"],
            default="Review",
        )
        rows.append(check)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    out["Any_Strict_Review"] = (out["Exposure_Check"] == "Review") | (out["Death_Check"] == "Review")
    return out.sort_values(["Any_Strict_Review", "Dimension", "Level"], ascending=[False, True, True]).reset_index(drop=True)

# Turns split-balance review flags into a numeric penalty.
def split_balance_penalty(balance: pd.DataFrame, min_ratio: float, max_ratio: float) -> float:
    """Small score is better; zero means all credible checks are inside range."""
    if balance.empty:
        return 0.0
    penalty = 0.0
    for ratio_col, check_col in [
        ("Train_Exposure_Ratio", "Exposure_Check"),
        ("Train_Death_Ratio", "Death_Check"),
    ]:
        strict = balance[check_col] == "Review"
        vals = pd.to_numeric(balance.loc[strict, ratio_col], errors="coerce").dropna()
        if len(vals):
            penalty += float(np.maximum(min_ratio - vals, 0).sum() + np.maximum(vals - max_ratio, 0).sum())
    return penalty

# Searches random seeds and returns the train and holdout split with the best segment exposure/death balance.
def make_balanced_train_score_split(
    df: pd.DataFrame,
    train_fraction: float,
    random_state: int,
    review_cols: list[str],
    strata_cols: list[str],
    min_ratio: float = 0.75,
    max_ratio: float = 0.85,
    min_full_exposure: float = 100.0,
    min_full_deaths: float = 5.0,
    max_seed_tries: int = 200,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    """Search seeds and keep the split with the best exposure/death balance."""
    require_sklearn()

    if not (0 < train_fraction < 1):
        raise ValueError("TRAIN_FRACTION must be between 0 and 1.")
    min_count = max(2, int(math.ceil(1 / min(train_fraction, 1 - train_fraction))))
    best = None
    tried = 0

    for seed in range(random_state, random_state + max_seed_tries):
        strat_key = build_split_strata(df, strata_cols=strata_cols, min_count=min_count)
        try:
            train_idx, score_idx = train_test_split(
                df.index,
                train_size=train_fraction,
                random_state=seed,
                stratify=strat_key,
            )
        except ValueError:
            train_idx, score_idx = train_test_split(
                df.index,
                train_size=train_fraction,
                random_state=seed,
                stratify=None,
            )
        train_df = df.loc[train_idx].copy().reset_index(drop=True)
        score_df = df.loc[score_idx].copy().reset_index(drop=True)
        balance = split_balance_table(
            df,
            train_df,
            score_df,
            review_cols=review_cols,
            min_ratio=min_ratio,
            max_ratio=max_ratio,
            min_full_exposure=min_full_exposure,
            min_full_deaths=min_full_deaths,
        )
        penalty = split_balance_penalty(balance, min_ratio=min_ratio, max_ratio=max_ratio)
        tried += 1

        if best is None or penalty < best[0]:
            best = (penalty, seed, train_df, score_df, balance)
        if penalty == 0:
            break

    penalty, seed, train_df, score_df, balance = best
    split_info = {
        "Selected_Seed": seed,
        "Seeds_Tried": tried,
        "Balance_Penalty": penalty,
        "Train_Fraction_Target": train_fraction,
        "Strict_Range": f"{min_ratio:.0%}-{max_ratio:.0%}",
        "Thin_Exposure_Threshold": min_full_exposure,
        "Thin_Death_Threshold": min_full_deaths,
    }
    return train_df, score_df, balance, split_info

# Creates a compact train/holdout/full summary of rows, exposure, deaths, A/E, and rates.
def split_overall_summary(train_df: pd.DataFrame, score_df: pd.DataFrame, full_df: pd.DataFrame) -> pd.DataFrame:
    """Simple overall split summary before the detailed segment balance table."""
    rows = []
    for label, sub in [("Train", train_df), ("Holdout", score_df), ("Full scope", full_df)]:
        deaths = sub[ACTUAL_CNT_COL].sum()
        expected = sub[EXPECTED_CNT_COL].sum()
        exposure = sub["Policies_Exposed"].sum()
        rows.append({
            "Split": label,
            "Rows": len(sub),
            "Death_Count": deaths,
            "Expected_Deaths": expected,
            "Policies_Exposed": exposure,
            "Share_of_Full_Exposure": safe_divide(exposure, full_df["Policies_Exposed"].sum()),
            "Share_of_Full_Deaths": safe_divide(deaths, full_df[ACTUAL_CNT_COL].sum()),
            "Actual_to_VBT": safe_divide(deaths, expected),
            "Actual_Rate_per_1000": safe_divide(deaths * 1000.0, exposure),
        })
    return pd.DataFrame(rows)


# SESSION 1.5.6 ACTUARIAL SUMMARY AND A/E HELPERS

# Calculates full-dataset exposure, actual, expected, A/E, and rate metrics in a display-friendly table.
def overall_metrics(df: pd.DataFrame) -> pd.DataFrame:
    actual_cnt = float(df[ACTUAL_CNT_COL].sum())
    expected_cnt = float(df[EXPECTED_CNT_COL].sum())
    actual_amt = float(df[ACTUAL_AMT_COL].sum())
    expected_amt = float(df[EXPECTED_AMT_COL].sum())
    policies = float(df["Policies_Exposed"].sum())
    amount_exposed = float(df["Amount_Exposed"].sum())
    return pd.DataFrame({
        "Metric": [
            "Rows",
            "Policies Exposed",
            "Amount Exposed",
            "Death Count",
            "Expected Death Count",
            "A/E Count",
            "Death Claim Amount",
            "Expected Death Amount",
            "A/E Amount",
            "Actual Rate per 1000",
            "Expected Rate per 1000",
        ],
        "Value": [
            len(df),
            policies,
            amount_exposed,
            actual_cnt,
            expected_cnt,
            safe_divide(actual_cnt, expected_cnt),
            actual_amt,
            expected_amt,
            safe_divide(actual_amt, expected_amt),
            safe_divide(actual_cnt * 1000.0, policies),
            safe_divide(expected_cnt * 1000.0, policies),
        ]
    })

# Aggregates actual, expected, exposure, A/E, and rates by grouping columns.
def ae_summary(df: pd.DataFrame, group_cols: str | list[str]) -> pd.DataFrame:
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    out = (
        df.groupby(group_cols, dropna=False)[
            [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"]
        ]
        .sum()
        .reset_index()
        .sort_values(group_cols)
        .reset_index(drop=True)
    )
    out["A/E_Count"] = safe_divide(out[ACTUAL_CNT_COL], out[EXPECTED_CNT_COL])
    out["A/E_Amount"] = safe_divide(out[ACTUAL_AMT_COL], out[EXPECTED_AMT_COL])
    out["Actual_Rate_per_1000"] = safe_divide(out[ACTUAL_CNT_COL] * 1000.0, out["Policies_Exposed"])
    out["Expected_Rate_per_1000"] = safe_divide(out[EXPECTED_CNT_COL] * 1000.0, out["Policies_Exposed"])
    return out

# Builds paired two-way pivot tables for Actual-to-VBT and expected death counts.
def two_way_ae_tables(df: pd.DataFrame, row_col: str, col_col: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    base = (
        df.groupby([row_col, col_col], dropna=False)[[ACTUAL_CNT_COL, EXPECTED_CNT_COL]]
        .sum()
        .reset_index()
    )
    base["Actual_to_VBT"] = safe_divide(base[ACTUAL_CNT_COL], base[EXPECTED_CNT_COL])

    ae_pivot = base.pivot(index=row_col, columns=col_col, values="Actual_to_VBT")
    exp_pivot = base.pivot(index=row_col, columns=col_col, values=EXPECTED_CNT_COL)

    return ae_pivot, exp_pivot


# SESSION 1.5.7 DEPENDENCE AND ASSOCIATION HELPERS

# Computes weighted Spearman correlation by ranking two numeric columns and applying weighted_corr().
def weighted_spearman(df: pd.DataFrame, x_col: str, y_col: str, weight_col: str = "Policies_Exposed") -> float:
    if x_col == y_col:
        return 1.0
    sub = df[[x_col, y_col, weight_col]].copy()
    sub[x_col] = pd.to_numeric(sub[x_col], errors="coerce")
    sub[y_col] = pd.to_numeric(sub[y_col], errors="coerce")
    sub[weight_col] = pd.to_numeric(sub[weight_col], errors="coerce")

    sub = sub.replace([np.inf, -np.inf], np.nan).dropna()
    if len(sub) <= 1:
        return np.nan
    x_rank = sub[x_col].rank(method="average")
    y_rank = sub[y_col].rank(method="average")
    return weighted_corr(x_rank, y_rank, sub[weight_col])

# Builds a pairwise weighted Spearman correlation matrix for numeric columns.
def weighted_corr_matrix(df: pd.DataFrame, columns: list[str], weight_col: str = "Policies_Exposed") -> pd.DataFrame:
    cols = [c for c in columns if c in df.columns]
    out = pd.DataFrame(index=cols, columns=cols, dtype=float)

    for c1 in cols:
        for c2 in cols:
            if c1 == c2:
                out.loc[c1, c2] = 1.0
            else:
                out.loc[c1, c2] = weighted_spearman(df, c1, c2, weight_col=weight_col)

    return out

# Creates an exposure-weighted cross-tabulation for two categorical variables.
def weighted_crosstab(df: pd.DataFrame, row_col: str, col_col: str, weight_col: str = "Policies_Exposed") -> pd.DataFrame:
    sub = df[[row_col, col_col, weight_col]].copy()
    sub[row_col] = sub[row_col].astype("string").fillna("Missing")
    sub[col_col] = sub[col_col].astype("string").fillna("Missing")
    sub[weight_col] = pd.to_numeric(sub[weight_col], errors="coerce").fillna(0.0)

    out = pd.pivot_table(
        sub,
        index=row_col,
        columns=col_col,
        values=weight_col,
        aggfunc="sum",
        fill_value=0.0,
        dropna=False,
    )
    return out

# Calculates bias-corrected Cramers V from an exposure-weighted contingency table.
def cramers_v_from_weighted_table(table: pd.DataFrame) -> float:
    observed = np.asarray(table, dtype=float)
    n = observed.sum()
    if n <= 0 or observed.shape[0] < 2 or observed.shape[1] < 2:
        return np.nan
    row_totals = observed.sum(axis=1, keepdims=True)
    col_totals = observed.sum(axis=0, keepdims=True)
    expected = row_totals @ col_totals / n
    keep = expected > 0
    chi2 = np.sum(((observed - expected) ** 2 / expected)[keep])
    phi2 = chi2 / n
    r, k = observed.shape
    if n <= 1:
        return np.nan
    phi2_corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)
    denom = min(k_corr - 1, r_corr - 1)
    if denom <= 0:
        return np.nan
    return float(np.sqrt(phi2_corr / denom))

# Builds a pairwise weighted Cramers V matrix for available categorical columns.
def cramers_v_matrix(df: pd.DataFrame, columns: list[str], weight_col: str = "Policies_Exposed") -> pd.DataFrame:
    cols = [c for c in columns if c in df.columns]
    out = pd.DataFrame(index=cols, columns=cols, dtype=float)
    for c1 in cols:
        for c2 in cols:
            if c1 == c2:
                out.loc[c1, c2] = 1.0
            else:
                table = weighted_crosstab(df, c1, c2, weight_col=weight_col)
                out.loc[c1, c2] = cramers_v_from_weighted_table(table)
    return out


# SESSION 1.5.8 MODEL FRAME AND ENCODING HELPERS

# Builds a clean modeling frame with required columns, derived age/spline fields, dtypes, and positive expected counts.
def prepare_model_frame(df: pd.DataFrame, required_cols: list[str]) -> pd.DataFrame:
    required_cols = list(dict.fromkeys(required_cols))
    out = df.copy()
    if ISSUE_AGE_MOD_COL not in out.columns or ATTAINED_AGE_MOD_COL not in out.columns:
        out = derive_age_basis_columns(out)
    out = add_spline_safe_columns(out)
    required_cols = [c for c in required_cols if c in out.columns]
    out = out[required_cols].copy()
    for col in required_cols:
        if col in CATEGORICAL_COLS or col in DERIVED_CATEGORICAL_COLS:
            out[col] = out[col].astype("string").fillna("Missing")
        else:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan)
    keep_cols = [c for c in required_cols if c not in [ACTUAL_AMT_COL, EXPECTED_AMT_COL]]
    out = out.dropna(subset=keep_cols).copy()
    out = out[out[EXPECTED_CNT_COL] > 0].copy()
    return out.reset_index(drop=True)

# Aggregates model-ready rows to the requested cell level, summing exposure and actual/expected metrics.
def aggregate_cells(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    metric_cols = [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed"
    ]
    metric_cols = [c for c in metric_cols if c in df.columns]
    if not features:
        agg = df[metric_cols].sum().to_frame().T
        return agg.reset_index(drop=True)
    agg = (
        df.groupby(features, dropna=False)[metric_cols]
        .sum()
        .reset_index()
        .sort_values(features)
        .reset_index(drop=True)
    )
    return agg

# Separates a feature list into categorical and numeric features.
def split_feature_types(features: list[str]) -> tuple[list[str], list[str]]:
    numeric_feature_cols = set(NUMERIC_COLS) | {
        ISSUE_AGE_MOD_COL,
        ATTAINED_AGE_MOD_COL,
        "Issue_Age_mod_spline",
        "Attained_Age_mod_spline",
        "Duration_spline",
    }
    cat_cols = [c for c in features if c in CATEGORICAL_COLS or c in DERIVED_CATEGORICAL_COLS]
    num_cols = [c for c in features if c in numeric_feature_cols]
    return cat_cols, num_cols

# Captures training-set category levels.
def build_category_levels(df: pd.DataFrame, features: list[str]) -> dict[str, list[str]]:
    cat_cols, _ = split_feature_types(features)
    out = {}
    for col in cat_cols:
        vals = df[col].astype("string").fillna("Missing")
        out[col] = vals.drop_duplicates().tolist()
    return out

# One-hot encodes categorical features, coerces numeric features, and aligns to saved design columns when scoring.
def encode_features(
    df: pd.DataFrame,
    features: list[str],
    category_levels: dict[str, list[str]] | None = None,
    design_columns: list[str] | None = None,
) -> pd.DataFrame:
    work = df[features].copy()
    cat_cols, num_cols = split_feature_types(features)

    for col in cat_cols:
        work[col] = work[col].astype("string").fillna("Missing")
        if category_levels is not None and col in category_levels:
            work[col] = pd.Categorical(work[col], categories=category_levels[col])
    enc = pd.get_dummies(work, columns=cat_cols, drop_first=False, dtype=float)

    for col in num_cols:
        enc[col] = pd.to_numeric(enc[col], errors="coerce")
    enc = enc.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    if design_columns is not None:
        for col in design_columns:
            if col not in enc.columns:
                enc[col] = 0.0
        enc = enc[design_columns].copy()
    return enc.astype(float)


# SESSION 1.5.9 FEATURE SCREENING HELPERS
def screen_formula_for_feature(feature: str) -> str:
    if feature == ISSUE_AGE_MOD_COL:
        return (
            f"{ACTUAL_CNT_COL} ~ bs({ISSUE_AGE_MOD_COL}, knots={tuple(GAM_ISSUE_AGE_KNOTS)}, degree=3, "
            f"include_intercept=False, lower_bound={ISSUE_AGE_MIN}, upper_bound={ISSUE_AGE_MOD_MAX})"
        )
    if feature == "Duration":
        return (
            f"{ACTUAL_CNT_COL} ~ bs(Duration, knots={tuple(GAM_DURATION_KNOTS)}, degree=3, "
            f"include_intercept=False, lower_bound=0, upper_bound={GAM_DURATION_UPPER_BOUND})"
        )
    if feature == ATTAINED_AGE_MOD_COL:
        return (
            f"{ACTUAL_CNT_COL} ~ bs({ATTAINED_AGE_MOD_COL}, knots={tuple(GAM_ATTAINED_AGE_KNOTS)}, degree=3, "
            f"include_intercept=False, lower_bound={ATTAINED_AGE_MIN}, upper_bound={ATTAINED_AGE_MOD_MAX})"
        )
    if feature in NUMERIC_SCREEN_FEATURES:
        return f"{ACTUAL_CNT_COL} ~ bs({feature}, df={SCREEN_SPLINE_DF}, degree=3, include_intercept=False)"
    return f"{ACTUAL_CNT_COL} ~ C({feature})"

def run_univariate_screen(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows = []
    for feature in features:
        needed = [feature, ACTUAL_CNT_COL, EXPECTED_CNT_COL]
        sub = prepare_model_frame(df, needed)
        try:
            sub = aggregate_cells(sub, [feature])

            base_model = smf.glm(
                formula=f"{ACTUAL_CNT_COL} ~ 1",
                data=sub,
                family=sm.families.Poisson(),
                offset=np.log(sub["Policies_Exposed"]),
            ).fit()
            model = smf.glm(
                formula=screen_formula_for_feature(feature),
                data=sub,
                family=sm.families.Poisson(),
                offset=np.log(sub["Policies_Exposed"]),
            ).fit()
            pred = model.predict(sub, offset=np.log(sub["Policies_Exposed"]))
            rows.append({
                "Feature": feature,
                "Levels_or_Unique": int(sub[feature].nunique(dropna=False)),
                "Base_Deviance": float(base_model.deviance),
                "Model_Deviance": float(model.deviance),
                "Deviance_Reduction": float(base_model.deviance - model.deviance),
                "Deviance_Reduction_%": 100 * (base_model.deviance - model.deviance) / base_model.deviance if base_model.deviance > 0 else np.nan,
                "AIC": float(model.aic),
                "Weighted_AE_Error": weighted_ae_error(sub[ACTUAL_CNT_COL], pred, sub[EXPECTED_CNT_COL]),
            })
        except Exception as exc:
            rows.append({
                "Feature": feature,
                "Levels_or_Unique": np.nan,
                "Base_Deviance": np.nan,
                "Model_Deviance": np.nan,
                "Deviance_Reduction": np.nan,
                "Deviance_Reduction_%": np.nan,
                "AIC": np.nan,
                "Weighted_AE_Error": np.nan,
                "Note": str(exc)[:200],
            })
    out = pd.DataFrame(rows)
    return out.sort_values(["Deviance_Reduction_%", "Deviance_Reduction"], ascending=False, na_position="last").reset_index(drop=True)

def run_tree_feature_importance(
    df: pd.DataFrame,
    features: list[str],
    estimator,
    smooth: float = 0.25,
    random_state: int = 42,
) -> pd.DataFrame:
    require_sklearn()
    needed = features + [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"]
    base = prepare_model_frame(df, needed)
    cell_df = aggregate_cells(base, features).copy()
    category_levels = build_category_levels(cell_df, features)
    X = encode_features(cell_df, features, category_levels=category_levels)
    y_log_ratio = np.log((cell_df[ACTUAL_CNT_COL] + smooth) / (cell_df["Policies_Exposed"] + smooth))
    w = np.clip(cell_df["Policies_Exposed"].to_numpy(dtype=float), 1e-12, None)
    estimator.fit(X, y_log_ratio, sample_weight=w)

    if hasattr(estimator, "feature_importances_"):
        raw_imp = pd.Series(estimator.feature_importances_, index=X.columns)
    else:
        perm = permutation_importance(
            estimator, X, y_log_ratio, n_repeats=10, random_state=random_state,
            scoring="neg_mean_absolute_error"
        )
        raw_imp = pd.Series(perm.importances_mean, index=X.columns)
    rows = []
    for feature in features:
        mask = raw_imp.index == feature
        mask = mask | raw_imp.index.str.startswith(f"{feature}_")
        rows.append({
            "Feature": feature,
            "Importance": float(raw_imp[mask].sum()),
        })
    out = pd.DataFrame(rows).sort_values("Importance", ascending=False).reset_index(drop=True)
    out["Rank"] = np.arange(1, len(out) + 1)
    return out

def combine_screen_ranks(
    univariate_df: pd.DataFrame | None = None,
    gbm_df: pd.DataFrame | None = None,
    extra_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    pieces = []
    if univariate_df is not None:
        uni = univariate_df[["Feature", "Deviance_Reduction_%"]].copy()
        uni["Univariate_Rank"] = uni["Deviance_Reduction_%"].rank(ascending=False, method="dense")
        pieces.append(uni)
    if gbm_df is not None:
        gbm = gbm_df[["Feature", "Importance"]].rename(columns={"Importance": "GBM_Importance"})
        gbm["GBM_Rank"] = gbm["GBM_Importance"].rank(ascending=False, method="dense")
        pieces.append(gbm)
    if extra_df is not None:
        extra = extra_df[["Feature", "Importance"]].rename(columns={"Importance": "ExtraTrees_Importance"})
        extra["ExtraTrees_Rank"] = extra["ExtraTrees_Importance"].rank(ascending=False, method="dense")
        pieces.append(extra)
    if not pieces:
        raise ValueError("No screening result tables were provided.")
    out = pieces[0].copy()
    for piece in pieces[1:]:
        out = out.merge(piece, on="Feature", how="outer")
    rank_cols = [c for c in out.columns if c.endswith("_Rank")]
    sort_anchor = "Deviance_Reduction_%" if "Deviance_Reduction_%" in out.columns else rank_cols[0]
    out["Average_Rank"] = out[rank_cols].mean(axis=1)
    out = out.sort_values(["Average_Rank", sort_anchor], ascending=[True, False], na_position="last").reset_index(drop=True)
    return out


# SESSION 1.5.10 SPLINE AND FORMULA CONSTRUCTION HELPERS
# Clips inputs inside spline bounds.
def add_spline_safe_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    age_lo = np.nextafter(float(ISSUE_AGE_MIN), np.inf)
    age_hi = np.nextafter(float(ISSUE_AGE_MOD_MAX), -np.inf)
    att_lo = np.nextafter(float(ATTAINED_AGE_MIN), np.inf)
    att_hi = np.nextafter(float(ATTAINED_AGE_MOD_MAX), -np.inf)
    dur_lo = np.nextafter(0.0, np.inf)
    dur_hi = np.nextafter(float(GAM_DURATION_UPPER_BOUND), -np.inf)

    out["Issue_Age_mod_spline"] = pd.to_numeric(
        out[ISSUE_AGE_MOD_COL], errors="coerce"
    ).clip(lower=age_lo, upper=age_hi)
    out["Attained_Age_mod_spline"] = pd.to_numeric(
        out[ATTAINED_AGE_MOD_COL], errors="coerce"
    ).clip(lower=att_lo, upper=att_hi)
    out["Duration_spline"] = pd.to_numeric(
        out["Duration"], errors="coerce"
    ).clip(lower=dur_lo, upper=dur_hi)
    # Add piecewise attained-age hinge columns when ENFORCE_CONTINUITY is False.
    out = add_piecewise_attained_age_cols_if_needed(out)
    return out

# Extracts notebook column names referenced in a statsmodels formula.
def extract_formula_features(formula: str) -> list[str]:
    tokens = set(re.findall(r"\b[A-Za-z_][A-Za-z0-9_]*\b", formula))
    metric_exclusions = {
        ACTUAL_CNT_COL,
        EXPECTED_CNT_COL,
        ACTUAL_AMT_COL,
        EXPECTED_AMT_COL,
        "Policies_Exposed",
        "Amount_Exposed",
    }
    derived_numeric_cols = [
        ISSUE_AGE_MOD_COL,
        ATTAINED_AGE_MOD_COL,
        "Issue_Age_mod_spline",
        "Attained_Age_mod_spline",
        "Duration_spline",
        "Duration",
    ]
    candidate_cols = list(dict.fromkeys(
        BASE_COLUMNS
        + NUMERIC_COLS
        + CATEGORICAL_COLS
        + DERIVED_CATEGORICAL_COLS
        + derived_numeric_cols
    ))
    return [c for c in candidate_cols if c in tokens and c not in metric_exclusions]

# What: Adds age segment columns for the non-continuous attained-age challenger basis.
def build_attained_age_piecewise_cols(df: pd.DataFrame, bp1: float, bp2: float) -> pd.DataFrame:
    out = df.copy()
    aa = pd.to_numeric(out["Attained_Age_mod_spline"], errors="coerce")
    out["AA_seg1"] = aa
    out["AA_seg2"] = np.maximum(0.0, aa - float(bp1))
    out["AA_seg3"] = np.maximum(0.0, aa - float(bp2))
    return out

# Builds the attained-age GAM formula.
def build_attained_age_gam_formula(
    enforce_continuity: bool,
    knots: list,
    attained_age_min: int,
    attained_age_mod_max: float,
    actual_cnt_col: str,
    duration_knots: list,
    duration_upper_bound: float,
) -> str:

    if enforce_continuity:
        attained_term = (
            f"bs(Attained_Age_mod_spline, knots={tuple(knots)}, degree=3, "
            f"include_intercept=False, lower_bound={attained_age_min}, "
            f"upper_bound={attained_age_mod_max})"
        )
    else:
        # Piecewise-linear hinge basis; no C² constraint.
        # AA_seg1/AA_seg2/AA_seg3 are added to the frame by
        # add_piecewise_attained_age_cols_if_needed() below.
        attained_term = "AA_seg1 + AA_seg2 + AA_seg3"
    duration_term = (
        f"bs(Duration_spline, knots={tuple(duration_knots)}, degree=3, "
        f"include_intercept=False, lower_bound=0, upper_bound={duration_upper_bound})"
    )
    return (
        f"{actual_cnt_col} ~ C(Age_Ind) + C(Sex) + C(Slct_Ult_Ind) "
        f"+ {attained_term} "
        f"+ {duration_term}"
    )

# Adds attained-age piecewise columns only when the continuity control is turned off.
def add_piecewise_attained_age_cols_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    if not ENFORCE_CONTINUITY:
        return build_attained_age_piecewise_cols(df, float(AA_BP1), float(AA_BP2))
    return df


# SESSION 1.5.11 MODEL BUNDLE FITTING AND SCORING HELPERS
# Returns a standardized metrics row comparing actual, expected, and predicted deaths for one dataset label.
def evaluate_predictions(actual, expected, predicted, label: str) -> dict:
    actual = np.asarray(actual, dtype=float)
    expected = np.asarray(expected, dtype=float)
    predicted = np.clip(np.asarray(predicted, dtype=float), 1e-12, None)
    return {
        "Dataset": label,
        "Rows": len(actual),
        "Actual_Deaths": float(actual.sum()),
        "Expected_VBT": float(expected.sum()),
        "Predicted_Deaths": float(predicted.sum()),
        "Actual_to_Predicted": safe_divide(actual.sum(), predicted.sum()),
        "Actual_to_VBT": safe_divide(actual.sum(), expected.sum()),
        "Poisson_Deviance": safe_poisson_deviance(actual, predicted),
        "Weighted_AE_Error": weighted_ae_error(actual, predicted, expected),
    }

# Scores a model bundle on new data and appends predicted deaths, A/P, A/E, and rate diagnostics.
def score_with_bundle(bundle: dict, score_df: pd.DataFrame) -> pd.DataFrame:
    validation_keep_cols = [
        "Observation_Year", "Age_Ind", "Sex", "Insurance_Plan", "Insurance_Plan_Group",
        "Face_Amount_Band", "Face_Band_Group",
        "Issue_Age", ISSUE_AGE_MOD_COL, "Attained_Age", ATTAINED_AGE_MOD_COL, "Duration", "Slct_Ult_Ind",
        "Issue_Age_Band", "Attained_Age_Band", "Duration_Band",
        "Policies_Exposed", "Amount_Exposed", ACTUAL_CNT_COL, EXPECTED_CNT_COL,
        ACTUAL_AMT_COL, EXPECTED_AMT_COL
    ]
    validation_keep_cols = [c for c in validation_keep_cols if c in score_df.columns]

    features = bundle["features"]
    needed = list(dict.fromkeys(features + validation_keep_cols))
    score_base = prepare_model_frame(score_df, needed)
    if bundle["model_type"] == "statsmodels_glm":
        pred = bundle["result"].predict(score_base, offset=np.log(score_base["Policies_Exposed"]))
    else:
        X_score = encode_features(
            score_base,
            features,
            category_levels=bundle["category_levels"],
            design_columns=bundle["design_columns"],
        )
        if bundle["model_type"] == "poisson_regressor":
            pred_rate = np.clip(bundle["result"].predict(X_score), 1e-12, None)
            pred = pred_rate * score_base["Policies_Exposed"].to_numpy(dtype=float)
        elif bundle["model_type"] == "log_ratio_model":
            pred_log = bundle["result"].predict(X_score)
            pred = np.exp(pred_log) * score_base["Policies_Exposed"].to_numpy(dtype=float)
        else:
            raise ValueError(f"Unsupported model_type: {bundle['model_type']}")
    out = score_base.copy()
    out["Predicted_Deaths"] = np.clip(np.asarray(pred, dtype=float), 1e-12, None)
    out["Predicted_to_VBT"] = safe_divide(out["Predicted_Deaths"], out[EXPECTED_CNT_COL])
    out["Actual_to_Predicted"] = safe_divide(out[ACTUAL_CNT_COL], out["Predicted_Deaths"])
    out["Actual_to_VBT"] = safe_divide(out[ACTUAL_CNT_COL], out[EXPECTED_CNT_COL])
    out["Actual_Rate_per_1000"] = safe_divide(out[ACTUAL_CNT_COL] * 1000.0, out["Policies_Exposed"])
    out["Expected_Rate_per_1000"] = safe_divide(out[EXPECTED_CNT_COL] * 1000.0, out["Policies_Exposed"])
    out["Predicted_Rate_per_1000"] = safe_divide(out["Predicted_Deaths"] * 1000.0, out["Policies_Exposed"])
    return out

# Computes BIC from a fitted statsmodels result.
def infer_glm_bic(result) -> float:
    nobs = getattr(result, "nobs", np.nan)
    k = len(getattr(result, "params", []))
    llf = getattr(result, "llf", np.nan)
    if not np.isfinite(nobs) or not np.isfinite(llf) or nobs <= 0:
        return np.nan
    return float(-2.0 * llf + k * np.log(nobs))

# Creates an intercept-only mortality-rate baseline and its training Poisson deviance.
def make_intercept_baseline(train_cells: pd.DataFrame) -> tuple[float, float]:
    baseline_ratio = safe_divide(train_cells[ACTUAL_CNT_COL].sum(), train_cells["Policies_Exposed"].sum())
    baseline_pred = baseline_ratio * train_cells["Policies_Exposed"].to_numpy(dtype=float)
    baseline_dev = safe_poisson_deviance(train_cells[ACTUAL_CNT_COL], baseline_pred)
    return float(baseline_ratio), float(baseline_dev)

# Converts a fitted model bundle into one standardized row for the model comparison table.
def bundle_comparison_row(bundle: dict) -> dict:
    metrics = bundle["metrics"].set_index("Dataset")
    row = {
        "Model_Key": bundle["model_key"],
        "Model_Type": bundle["model_type"],
        "Feature_Count": len(bundle["features"]),
        "Parameter_Count": bundle.get("parameter_count", np.nan),
        "Likelihood_Criteria_Comparable": bundle.get("likelihood_comparable", "No"),
        "Train_Deviance": float(metrics.loc["Train", "Poisson_Deviance"]),
        "Scored_Deviance": float(metrics.loc["Scored_Data", "Poisson_Deviance"]),
        "Train_A/P": float(metrics.loc["Train", "Actual_to_Predicted"]),
        "Scored_A/P": float(metrics.loc["Scored_Data", "Actual_to_Predicted"]),
        "Train_Weighted_AE_Error": float(metrics.loc["Train", "Weighted_AE_Error"]),
        "Scored_Weighted_AE_Error": float(metrics.loc["Scored_Data", "Weighted_AE_Error"]),
        "Train_Deviance_Explained_vs_Intercept": safe_divide(
            bundle.get("baseline_deviance", np.nan) - float(metrics.loc["Train", "Poisson_Deviance"]),
            bundle.get("baseline_deviance", np.nan),
        ),
        "AIC": bundle.get("aic", np.nan),
        "BIC": bundle.get("bic", np.nan),
        "Train_Null_Deviance": bundle.get("null_deviance", np.nan),
        "Train_Deviance_Gap_Scored_minus_Train": float(metrics.loc["Scored_Data", "Poisson_Deviance"]) - float(metrics.loc["Train", "Poisson_Deviance"]),
    }
    return row

# Fits a statsmodels Poisson GLM bundle from a formula and evaluates it on train and scored data.
def fit_formula_glm_bundle(model_key: str, formula: str, train_df: pd.DataFrame, score_df: pd.DataFrame) -> dict:
    features = extract_formula_features(formula)
    required_cols = features + [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"
    ]
    train_model_df = prepare_model_frame(train_df, required_cols)
    score_model_df = prepare_model_frame(score_df, required_cols)

    train_cells = aggregate_cells(train_model_df, features)
    baseline_ratio, baseline_dev = make_intercept_baseline(train_cells)
    result = smf.glm(
        formula=formula,
        data=train_cells,
        family=sm.families.Poisson(),
        offset=np.log(train_cells["Policies_Exposed"]),
    ).fit()
    bundle = {
        "model_key": model_key,
        "model_type": "statsmodels_glm",
        "label": model_key,
        "formula": formula,
        "features": features,
        "result": result,
        "baseline_ratio": baseline_ratio,
        "baseline_deviance": baseline_dev,
        "parameter_count": int(len(result.params)),
        "likelihood_comparable": "Yes",
        "aic": float(result.aic),
        "bic": infer_glm_bic(result),
        "null_deviance": float(getattr(result, "null_deviance", np.nan)),
    }

    scored_train = score_with_bundle(bundle, train_df)
    scored_score = score_with_bundle(bundle, score_df)
    metrics = pd.DataFrame([
        evaluate_predictions(scored_train[ACTUAL_CNT_COL], scored_train[EXPECTED_CNT_COL], scored_train["Predicted_Deaths"], "Train"),
        evaluate_predictions(scored_score[ACTUAL_CNT_COL], scored_score[EXPECTED_CNT_COL], scored_score["Predicted_Deaths"], "Scored_Data"),
    ])
    bundle["metrics"] = metrics
    bundle["train_cells"] = train_cells
    return bundle

# Fits a sklearn PoissonRegressor bundle.
def fit_poisson_regressor_bundle(
    model_key: str,
    train_df: pd.DataFrame,
    score_df: pd.DataFrame,
    features: list[str],
    alpha: float = 0.001,
) -> dict:
    require_sklearn()
    needed = features + [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"]
    train_model_df = prepare_model_frame(train_df, needed)
    score_model_df = prepare_model_frame(score_df, needed)

    train_cells = aggregate_cells(train_model_df, features)
    baseline_ratio, baseline_dev = make_intercept_baseline(train_cells)
    category_levels = build_category_levels(train_cells, features)
    X_train = encode_features(train_cells, features, category_levels=category_levels)
    rate_train = safe_divide(train_cells[ACTUAL_CNT_COL], train_cells["Policies_Exposed"])
    w_train = np.clip(train_cells["Policies_Exposed"].to_numpy(dtype=float), 1e-12, None)
    est = PoissonRegressor(alpha=alpha, max_iter=1000)
    est.fit(X_train, rate_train, sample_weight=w_train)
    bundle = {
        "model_key": model_key,
        "model_type": "poisson_regressor",
        "label": model_key,
        "features": features,
        "result": est,
        "category_levels": category_levels,
        "design_columns": X_train.columns.tolist(),
        "baseline_ratio": baseline_ratio,
        "baseline_deviance": baseline_dev,
        "parameter_count": int(X_train.shape[1] + 1),
        "likelihood_comparable": "No",
    }
    scored_train = score_with_bundle(bundle, train_df)
    scored_score = score_with_bundle(bundle, score_df)
    metrics = pd.DataFrame([
        evaluate_predictions(scored_train[ACTUAL_CNT_COL], scored_train[EXPECTED_CNT_COL], scored_train["Predicted_Deaths"], "Train"),
        evaluate_predictions(scored_score[ACTUAL_CNT_COL], scored_score[EXPECTED_CNT_COL], scored_score["Predicted_Deaths"], "Scored_Data"),
    ])
    bundle["metrics"] = metrics
    bundle["train_cells"] = train_cells
    return bundle

# Fits a sklearn estimator to smoothed log mortality rates and packages it for consistent scoring.
def fit_log_ratio_bundle(
    model_key: str,
    estimator,
    train_df: pd.DataFrame,
    score_df: pd.DataFrame,
    features: list[str],
    smooth: float = 0.25,
) -> dict:
    require_sklearn()
    needed = features + [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"]
    train_model_df = prepare_model_frame(train_df, needed)
    score_model_df = prepare_model_frame(score_df, needed)
    train_cells = aggregate_cells(train_model_df, features)
    baseline_ratio, baseline_dev = make_intercept_baseline(train_cells)
    category_levels = build_category_levels(train_cells, features)
    X_train = encode_features(train_cells, features, category_levels=category_levels)
    y_train = np.log((train_cells[ACTUAL_CNT_COL] + smooth) / (train_cells["Policies_Exposed"] + smooth))
    w_train = np.clip(train_cells["Policies_Exposed"].to_numpy(dtype=float), 1e-12, None)
    estimator.fit(X_train, y_train, sample_weight=w_train)
    bundle = {
        "model_key": model_key,
        "model_type": "log_ratio_model",
        "label": model_key,
        "features": features,
        "result": estimator,
        "category_levels": category_levels,
        "design_columns": X_train.columns.tolist(),
        "smooth": smooth,
        "baseline_ratio": baseline_ratio,
        "baseline_deviance": baseline_dev,
        "parameter_count": int(X_train.shape[1]),
        "likelihood_comparable": "No",
    }
    scored_train = score_with_bundle(bundle, train_df)
    scored_score = score_with_bundle(bundle, score_df)
    metrics = pd.DataFrame([
        evaluate_predictions(scored_train[ACTUAL_CNT_COL], scored_train[EXPECTED_CNT_COL], scored_train["Predicted_Deaths"], "Train"),
        evaluate_predictions(scored_score[ACTUAL_CNT_COL], scored_score[EXPECTED_CNT_COL], scored_score["Predicted_Deaths"], "Scored_Data"),
    ])
    bundle["metrics"] = metrics
    bundle["train_cells"] = train_cells
    return bundle

# Keeps a backwards-compatible alias for fit_log_ratio_bundle().
def fit_log_rate_bundle(*args, **kwargs):
    return fit_log_ratio_bundle(*args, **kwargs)


# SESSION 1.5.12 CALIBRATION AND VALIDATION HELPERS
# Aggregates actual, expected, predicted, A/P, A/E, rates, and review flags by calibration segment.
def calibration_table(df: pd.DataFrame, group_cols: str | list[str], pred_col: str = "Predicted_Deaths") -> pd.DataFrame:
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    out = (
        df.groupby(group_cols, dropna=False)[
            [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", pred_col]
        ]
        .sum()
        .reset_index()
    )
    out["Actual_to_Predicted"] = safe_divide(out[ACTUAL_CNT_COL], out[pred_col])
    out["Actual_to_VBT"] = safe_divide(out[ACTUAL_CNT_COL], out[EXPECTED_CNT_COL])
    out["Predicted_to_VBT"] = safe_divide(out[pred_col], out[EXPECTED_CNT_COL])
    out["Amount_AE_to_VBT"] = safe_divide(out[ACTUAL_AMT_COL], out[EXPECTED_AMT_COL])
    out["Actual_Rate_per_1000"] = safe_divide(out[ACTUAL_CNT_COL] * 1000.0, out["Policies_Exposed"])
    out["Expected_Rate_per_1000"] = safe_divide(out[EXPECTED_CNT_COL] * 1000.0, out["Policies_Exposed"])
    out["Predicted_Rate_per_1000"] = safe_divide(out[pred_col] * 1000.0, out["Policies_Exposed"])
    out["Gap_from_1_on_A/P"] = (out["Actual_to_Predicted"] - 1.0).abs()

    out["Flag"] = np.select(
        [
            out[EXPECTED_CNT_COL] < MIN_EXPECTED_FOR_REVIEW,
            out["Gap_from_1_on_A/P"] > SEGMENT_TOLERANCE,
        ],
        [
            "Thin",
            "Review",
        ],
        default="OK",
    )
    out["Flag_Order"] = out["Flag"].map({"Review": 2, "Thin": 1, "OK": 0}).fillna(0)
    # Sort robustly: convert interval categories to string for sorting,
    # which handles NaN groups and pandas version differences cleanly.
    sort_keys = [group_cols] if isinstance(group_cols, str) else list(group_cols)
    return out.sort_values(
        sort_keys,
        key=lambda col: col.astype(str),
        na_position="last",
    ).reset_index(drop=True)

# Monotonicity review.
def monotonicity_review(
    df: pd.DataFrame,
    x_col: str,
    group_cols: str | list[str],
    rate_col: str = "Predicted_Rate_per_1000",
    min_expected: float = 3.0,
) -> pd.DataFrame:
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    base = calibration_table(df, group_cols + [x_col]).copy()
    base = base.sort_values(group_cols + [x_col]).reset_index(drop=True)
    rows = []
    for keys, sub in base.groupby(group_cols, dropna=False, sort=False):
        sub = sub.sort_values(x_col).reset_index(drop=True)
        credible = sub[sub[EXPECTED_CNT_COL] >= min_expected].copy().reset_index(drop=True)

        if len(group_cols) == 1:
            keys = (keys,)
        elif not isinstance(keys, tuple):
            keys = tuple(keys)

        series_key = " | ".join([f"{col}={val}" for col, val in zip(group_cols, keys)])
        if len(credible) == 0:
            row = {col: val for col, val in zip(group_cols, keys)}
            row.update({
                "Series_Key": series_key,
                "Credible_Points": 0,
                "Overall_Direction": "Insufficient",
                "x_col": x_col,
            })
            rows.append(row)
            continue

        overall_direction = "Flat"
        if len(credible) >= 2:
            overall_delta = credible[rate_col].iloc[-1] - credible[rate_col].iloc[0]
            if overall_delta > 1e-12:
                overall_direction = "Up"
            elif overall_delta < -1e-12:
                overall_direction = "Down"
        prev_rate = None
        for _, point in credible.iterrows():
            delta = np.nan if prev_rate is None else point[rate_col] - prev_rate
            violation = False
            if prev_rate is not None:
                if overall_direction == "Up" and delta < -1e-12:
                    violation = True
                elif overall_direction == "Down" and delta > 1e-12:
                    violation = True
            row = {col: val for col, val in zip(group_cols, keys)}
            row.update({
                "Series_Key": series_key,
                "x_col": x_col,
                "x_value": point[x_col],
                "Overall_Direction": overall_direction,
                "Credible_Points": len(credible),
                EXPECTED_CNT_COL: point[EXPECTED_CNT_COL],
                "Predicted_Rate_per_1000": point["Predicted_Rate_per_1000"],
                "Actual_Rate_per_1000": point["Actual_Rate_per_1000"],
                "Expected_Rate_per_1000": point["Expected_Rate_per_1000"],
                "Delta_from_Prior": delta,
                "Monotonicity_Flag": "Review" if violation else "OK",
            })
            rows.append(row)
            prev_rate = point[rate_col]
    out = pd.DataFrame(rows)
    sort_cols = [c for c in group_cols if c in out.columns] + (["x_value"] if "x_value" in out.columns else [])
    if sort_cols:
        out = out.sort_values(sort_cols).reset_index(drop=True)
    return out


# SESSION 1.5.13 PLOTTING AND GRAPH REVIEW HELPERS
def plot_rate_curves(
    df: pd.DataFrame,
    x_col: str,
    split_col: str,
    rate_col: str | list[str] = "Predicted_Rate_per_1000",
    min_expected: float = 0.0,
    title: str | None = None,
):
    base = calibration_table(df, [split_col, x_col]).copy()
    base = base[base[EXPECTED_CNT_COL] >= min_expected].copy()
    if len(base) == 0:
        print("No rows passed the credibility floor for this plot.")
        return

    rate_cols = [rate_col] if isinstance(rate_col, str) else list(rate_col)

    marker_map = {
        "Actual_Rate_per_1000": "o",
        "Predicted_Rate_per_1000": "s",
        "Expected_Rate_per_1000": "x",
    }
    linestyle_map = {
        "Actual_Rate_per_1000": "-",
        "Predicted_Rate_per_1000": "--",
        "Expected_Rate_per_1000": ":",
    }
    label_map = {
        "Actual_Rate_per_1000": "Actual",
        "Predicted_Rate_per_1000": "Predicted",
        "Expected_Rate_per_1000": "VBT",
    }

    plt.figure(figsize=(10, 6))
    for level, sub in base.groupby(split_col, dropna=False):
        sub = sub.sort_values(x_col)
        for rc in rate_cols:
            if rc not in sub.columns:
                continue
            plt.plot(
                sub[x_col],
                sub[rc],
                marker=marker_map.get(rc, "o"),
                linestyle=linestyle_map.get(rc, "-"),
                label=f"{level} | {label_map.get(rc, rc)}",
            )

    plt.xlabel(x_col)
    plt.ylabel("Rate per 1000")
    if len(rate_cols) == 1:
        default_title = f"{rate_cols[0]} by {x_col} split by {split_col}"
    else:
        default_title = f"Rate comparison by {x_col} split by {split_col}"
    plt.title(title or default_title)

    plt.grid(True, alpha=0.3)
    plt.legend(title=split_col, bbox_to_anchor=(1.02, 1), loc="upper left")
    ax = plt.gca()
    force_plain_axis(ax, axis="both")
    plt.tight_layout()
    plt.show()

# Plots actual, VBT expected, and predicted rates.
def plot_rate_comparison(
    df: pd.DataFrame,
    x_col: str,
    split_col: str,
    min_expected: float = 0.0,
    title: str | None = None,
):
    base = calibration_table(df, [split_col, x_col]).copy()
    base = base[base[EXPECTED_CNT_COL] >= min_expected].copy()
    if len(base) == 0:
        print("No rows passed the credibility floor for this plot.")
        return

    plt.figure(figsize=(10, 6))
    for level, sub in base.groupby(split_col, dropna=False):
        sub = sub.sort_values(x_col)
        plt.plot(sub[x_col], sub["Actual_Rate_per_1000"], marker="o", linestyle="-", label=f"{level} | Actual")
        plt.plot(sub[x_col], sub["Expected_Rate_per_1000"], marker="x", linestyle="--", label=f"{level} | VBT")
        plt.plot(sub[x_col], sub["Predicted_Rate_per_1000"], marker="s", linestyle="-.", label=f"{level} | Pred")

    plt.xlabel(x_col)
    plt.ylabel("Rate per 1000")
    plt.title(title or f"Actual vs VBT vs Predicted by {x_col} split by {split_col}")
    plt.grid(True, alpha=0.3)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", ncol=1)
    ax = plt.gca()
    force_plain_axis(ax, axis="both")
    plt.tight_layout()
    plt.show()


def residual_heatmap(df: pd.DataFrame, row_col: str, col_col: str, value_col: str = "Actual_to_Predicted"):
    heat = calibration_table(df, [row_col, col_col]).pivot(index=row_col, columns=col_col, values=value_col)

    plt.figure(figsize=(10, 6))
    plt.imshow(heat.values, aspect="auto")
    plt.colorbar(label=value_col)
    plt.xticks(range(len(heat.columns)), [str(c) for c in heat.columns], rotation=45, ha="right")
    plt.yticks(range(len(heat.index)), [str(i) for i in heat.index])
    plt.title(f"{value_col}: {row_col} × {col_col}")
    plt.tight_layout()
    plt.show()

# Plots numerator-to-denominator rate ratios.
def plot_ratio_curves(
    df: pd.DataFrame,
    x_col: str,
    split_col: str,
    numerator_col: str = "Actual_Rate_per_1000",
    denominator_col: str = "Predicted_Rate_per_1000",
    min_expected: float = 0.0,
    title: str | None = None,
    y_label: str = "Ratio",
):
    base = calibration_table(df, [split_col, x_col]).copy()
    base = base[base[EXPECTED_CNT_COL] >= min_expected].copy()
    if len(base) == 0:
        print("No rows passed the credibility floor for this plot.")
        return

    base = base[base[denominator_col] > 0].copy()
    base["Ratio"] = base[numerator_col] / base[denominator_col]

    plt.figure(figsize=(10, 6))
    for level, sub in base.groupby(split_col, dropna=False):
        sub = sub.sort_values(x_col)
        plt.plot(
            sub[x_col],
            sub["Ratio"],
            marker="o",
            label=f"{level}",
        )

    plt.axhline(1.0, linestyle="--")
    plt.xlabel(x_col)
    plt.ylabel(y_label)
    plt.title(title or f"{y_label} by {x_col} split by {split_col}")
    plt.grid(True, alpha=0.3)
    plt.legend(title=split_col, bbox_to_anchor=(1.02, 1), loc="upper left")
    ax = plt.gca()
    force_plain_axis(ax, axis="both")
    plt.tight_layout()
    plt.show()


# SESSION 1.5.14 SHEET-STYLE OUTPUT AND BENCHMARK HELPERS
# Builds sheet-style mortality-rate tables by issue age and duration.
def build_sheet_style_table(
    df: pd.DataFrame,
    value_col: str,
    age_ind_value: str,
    sex_value: str,
    select_period: int = 25,
    ult_duration: int = 26,
    duration_cols: list[int] | None = None,
    rate_per: float = 1000.0,
    issue_age_min: int = 0,
    issue_age_max: int = 17,
) -> pd.DataFrame:
    if duration_cols is None:
        duration_cols = list(range(1, 26))

    work = df.copy()
    work["Age_Ind"] = work["Age_Ind"].astype("string")
    work["Sex"] = work["Sex"].astype("string")

    work = work[
        (work["Age_Ind"] == str(age_ind_value)) &
        (work["Sex"] == str(sex_value)) &
        (work["Issue_Age"].between(issue_age_min, issue_age_max, inclusive="both"))
    ].copy()

    rows = []
    for issue_age in range(issue_age_min, issue_age_max + 1):
        row = {"Iss. Age": issue_age}

        if str(age_ind_value) == "1":
            issue_age_mod_value = issue_age + 0.5
        else:
            issue_age_mod_value = float(issue_age)

        for dur in duration_cols:
            sub = work[(work["Issue_Age"] == issue_age) & (work["Duration"] == dur)].copy()
            row[str(dur)] = safe_divide(rate_per * sub[value_col].sum(), sub["Policies_Exposed"].sum())

        ult_sub = work[(work["Issue_Age"] == issue_age)].copy()

        preferred_ult = ult_sub[(ult_sub["Duration"] >= ult_duration)].copy()

        if "Slct_Ult_Ind" in ult_sub.columns and len(preferred_ult) == 0:
            preferred_ult = ult_sub[
                ult_sub["Slct_Ult_Ind"].astype("string").str.upper().isin(["U", "ULT", "ULTIMATE"])
            ].copy()

        attained_age_col = ATTAINED_AGE_MOD_COL if ATTAINED_AGE_MOD_COL in preferred_ult.columns else "Attained_Age"
        if len(preferred_ult) > 0 and attained_age_col in preferred_ult.columns:
            exact_age = preferred_ult[np.isclose(preferred_ult[attained_age_col], issue_age_mod_value + select_period)].copy()
            if len(exact_age) > 0:
                preferred_ult = exact_age

        row["Ult."] = safe_divide(rate_per * preferred_ult[value_col].sum(), preferred_ult["Policies_Exposed"].sum())
        row["Att. Age"] = issue_age_mod_value + select_period
        rows.append(row)

    col_order = ["Iss. Age"] + [str(d) for d in duration_cols] + ["Ult.", "Att. Age"]
    return pd.DataFrame(rows)[col_order]


def to_long_sheet_table(sheet_df: pd.DataFrame) -> pd.DataFrame:
    duration_cols = [c for c in sheet_df.columns if c not in ["Iss. Age", "Ult.", "Att. Age"]]
    long = sheet_df.melt(
        id_vars=["Iss. Age", "Ult.", "Att. Age"],
        value_vars=duration_cols,
        var_name="Duration_Col",
        value_name="Rate",
    )
    long["Duration_Col"] = long["Duration_Col"].astype(str)
    return long.sort_values(["Iss. Age", "Duration_Col"]).reset_index(drop=True)

# What: Reads and standardizes an Excel benchmark sheet for the expected issue-age range.
def read_reference_sheet(path: Path, sheet_name: str, max_issue_age: int = 17) -> pd.DataFrame:
    ref = pd.read_excel(path, sheet_name=sheet_name)
    ref = ref.iloc[: max_issue_age + 1].copy()
    ref.columns = [str(c).strip() for c in ref.columns]
    return ref

# Compares modeled sheet-style rates against a reference sheet and ranks difference in the optional Session 4 comparison cell.
def compare_to_reference(modeled_sheet: pd.DataFrame, reference_sheet: pd.DataFrame) -> pd.DataFrame:
    modeled_long = to_long_sheet_table(modeled_sheet).rename(columns={"Rate": "Modeled_Rate"})
    reference_long = to_long_sheet_table(reference_sheet).rename(columns={"Rate": "Reference_Rate"})
    out = modeled_long.merge(
        reference_long,
        on=["Iss. Age", "Att. Age", "Duration_Col"],
        how="inner",
    )
    out["Difference"] = out["Modeled_Rate"] - out["Reference_Rate"]
    out["Absolute_Difference"] = out["Difference"].abs()
    return out.sort_values(["Absolute_Difference", "Iss. Age"], ascending=[False, True]).reset_index(drop=True)


In [10]:
# SESSION 1.6 LOAD DATA, APPLY SCOPE, AND REVIEW OVERVIEW
raw_df = read_parquet_data(DATA_PATH, columns=BASE_COLUMNS)
scoped_df, scope_log = apply_scope_filters(raw_df)

screening_df = apply_optional_development_sample(
    scoped_df,
    fraction=DEVELOPMENT_SAMPLE_FRACTION,
    strata=DEVELOPMENT_SAMPLE_STRATA,
    random_state=DEVELOPMENT_SAMPLE_RANDOM_STATE,
)

print("Scope log")
display(scope_log)

print("\nOverall scoped metrics")
display(overall_metrics(scoped_df))

print("\nScreening base metrics")
display(overall_metrics(screening_df))


if DEVELOPMENT_SAMPLE_FRACTION is None:
    pass
else:
    print(
        f"\nDevelopment sample note: Session 2 screening is using an optional stratified sample with fraction="
        f"{DEVELOPMENT_SAMPLE_FRACTION:.2f}. Session 3 and Session 4 remain on full scoped data by default."
    )

# # Quick structure and sample rows
# print("Columns loaded:")
# print(list(scoped_df.columns))
#
# display(scoped_df.head(10))

Scope log


,step,rows
0,raw_input,3598960
1,Issue_Age between 0 and 17,3598960
2,Attained_Age >= 1,3592746
3,Attained_Age <= 67,3459328
4,Smoker_Status in ['U'],1553885
5,Insurance_Plan not in ['Other'],1458411
6,SOA_Post_Lvl_Ind not in ['PLT'],1428368
7,ExpDth_VBT2015_Cnt > 0,1395607
8,Applied 1M face cap rule,1395607
9,Added Issue_Age_mod / Attained_Age_mod (ALB = ...,1395607



Overall scoped metrics


,Metric,Value
0,Rows,"1,395,607.000000"
1,Policies Exposed,"64,140,893.991038"
2,Amount Exposed,"2,783,723,289,786.646484"
3,Death Count,"134,148.000000"
4,Expected Death Count,"104,256.949952"
5,A/E Count,1.286706
6,Death Claim Amount,"1,924,916,720.000000"
7,Expected Death Amount,"1,620,753,609.911697"
8,A/E Amount,1.187668
9,Actual Rate per 1000,2.091458



Screening base metrics


,Metric,Value
0,Rows,"1,395,607.000000"
1,Policies Exposed,"64,140,893.991038"
2,Amount Exposed,"2,783,723,289,786.646484"
3,Death Count,"134,148.000000"
4,Expected Death Count,"104,256.949952"
5,A/E Count,1.286706
6,Death Claim Amount,"1,924,916,720.000000"
7,Expected Death Amount,"1,620,753,609.911697"
8,A/E Amount,1.187668
9,Actual Rate per 1000,2.091458


---

# Session 2 · Model screening, dependence review, and method understanding


In [14]:
# SESSION 2.1 SCREENING CONTROL PANEL

SCREEN_FEATURES = [
    "Age_Ind",
    "Sex",
    "Insurance_Plan",
    "Insurance_Plan_Group",
    "Face_Amount_Band",
    "Face_Band_Group",
    "Slct_Ult_Ind",
    "Preferred_Indicator",
    "Number_of_Pfd_Classes",
    ISSUE_AGE_MOD_COL,
    ATTAINED_AGE_MOD_COL,
    "Duration",
    "Observation_Year",
    "SOA_Guar_Lvl_TP",
    "SOA_Antp_Lvl_TP",
    "SOA_Post_Lvl_Ind",
]
NUMERIC_SCREEN_FEATURES = [ISSUE_AGE_MOD_COL, ATTAINED_AGE_MOD_COL, "Duration", "Observation_Year"]
SCREEN_SPLINE_DF = 6
SCREEN_TARGET_SMOOTH = 0.25
SCREEN_RANDOM_STATE = 42

# Shared spline controls used by Session 2 screening and Session 3 modeling.
GAM_ISSUE_AGE_KNOTS = [10, 15]
GAM_DURATION_KNOTS = [1, 14, 25]
GAM_DURATION_UPPER_BOUND = 26
GAM_ATTAINED_AGE_KNOTS = [5, 25, 45]
ENFORCE_CONTINUITY = True

CORRELATION_WEIGHT_COL = "Policies_Exposed"

NUMERIC_CORR_FEATURES = [ISSUE_AGE_MOD_COL, ATTAINED_AGE_MOD_COL, "Duration", "Observation_Year"]
CATEGORICAL_ASSOC_FEATURES = [
    "Age_Ind",
    "Sex",
    "Insurance_Plan",
    "Insurance_Plan_Group",
    "Face_Amount_Band",
    "Face_Band_Group",
    "Slct_Ult_Ind",
]

TWO_WAY_AE_PAIRS = [
    ("Issue_Age_Band", "Duration_Band"),
    ("Attained_Age_Band", "Duration_Band"),
    ("Insurance_Plan_Group", "Face_Band_Group"),
    ("Age_Ind", "Sex"),
]

# Manual shortlist after screening.
SHORTLIST_FEATURES = [
    "Age_Ind",
    "Sex",
    "Issue_Age",
    "Duration",
    "Insurance_Plan_Group",
    "Face_Band_Group",
]

GAM_ISSUE_AGE_KNOTS = [10, 15]
GAM_DURATION_KNOTS = [1, 14, 25]
GAM_DURATION_UPPER_BOUND = 26
GAM_ATTAINED_AGE_KNOTS = [5, 25, 45]
ENFORCE_CONTINUITY = True

In [16]:
# SESSION 2.3 GBM FEATURE IMPORTANCE

require_sklearn()

gbm_screen_results = run_tree_feature_importance(
    screening_df,
    SCREEN_FEATURES,
    estimator=GradientBoostingRegressor(random_state=SCREEN_RANDOM_STATE),
    smooth=SCREEN_TARGET_SMOOTH,
    random_state=SCREEN_RANDOM_STATE,
)

display(gbm_screen_results)
export_csv(gbm_screen_results, "session2_gbm_feature_importance.csv")


,Feature,Importance,Rank
0,Attained_Age_mod,0.519870,1
1,Insurance_Plan,0.132146,2
2,Face_Amount_Band,0.098663,3
3,Issue_Age_mod,0.079611,4
4,Duration,0.055314,5
5,Age_Ind,0.042276,6
6,Face_Band_Group,0.029211,7
7,Sex,0.022714,8
8,Observation_Year,0.013679,9
9,SOA_Post_Lvl_Ind,0.003907,10


Saved: script\model_outputs\session2_gbm_feature_importance.csv


In [17]:
# SESSION 2.4 EXTRATREES FEATURE IMPORTANCE

require_sklearn()

extratrees_screen_results = run_tree_feature_importance(
    screening_df,
    SCREEN_FEATURES,
    estimator=ExtraTreesRegressor(
        n_estimators=300,
        random_state=SCREEN_RANDOM_STATE,
        n_jobs=1,
        min_samples_leaf=5,
    ),
    smooth=SCREEN_TARGET_SMOOTH,
    random_state=SCREEN_RANDOM_STATE,
)

display(extratrees_screen_results)
export_csv(extratrees_screen_results, "session2_extratrees_feature_importance.csv")


,Feature,Importance,Rank
0,Attained_Age_mod,0.372574,1
1,Face_Amount_Band,0.129410,2
2,Insurance_Plan,0.126335,3
3,Duration,0.113478,4
4,Issue_Age_mod,0.082354,5
5,Age_Ind,0.064196,6
6,Observation_Year,0.034357,7
7,Sex,0.025255,8
8,Face_Band_Group,0.023634,9
9,Slct_Ult_Ind,0.013142,10


Saved: script\model_outputs\session2_extratrees_feature_importance.csv


In [18]:
# SESSION 2.5 COMBINED SCREENING SCOREBOARD

available_univariate = screen_results if "screen_results" in globals() else None
available_gbm = gbm_screen_results if "gbm_screen_results" in globals() else None
available_extra = extratrees_screen_results if "extratrees_screen_results" in globals() else None

screen_scoreboard = combine_screen_ranks(
    univariate_df=available_univariate,
    gbm_df=available_gbm,
    extra_df=available_extra,
)

display(screen_scoreboard)
export_csv(screen_scoreboard, "session2_combined_screen_scoreboard.csv")


,Feature,Deviance_Reduction_%,Univariate_Rank,GBM_Importance,GBM_Rank,ExtraTrees_Importance,ExtraTrees_Rank,Average_Rank
0,Attained_Age_mod,NaN,NaN,0.519870,1.000000,0.372574,1.000000,1.000000
1,Face_Amount_Band,NaN,NaN,0.098663,3.000000,0.129410,2.000000,2.500000
2,Insurance_Plan,NaN,NaN,0.132146,2.000000,0.126335,3.000000,2.500000
3,Duration,NaN,NaN,0.055314,5.000000,0.113478,4.000000,4.500000
4,Issue_Age_mod,NaN,NaN,0.079611,4.000000,0.082354,5.000000,4.500000
5,Age_Ind,NaN,NaN,0.042276,6.000000,0.064196,6.000000,6.000000
6,Face_Band_Group,NaN,NaN,0.029211,7.000000,0.023634,9.000000,8.000000
7,Observation_Year,NaN,NaN,0.013679,9.000000,0.034357,7.000000,8.000000
8,Sex,NaN,NaN,0.022714,8.000000,0.025255,8.000000,8.000000
9,SOA_Post_Lvl_Ind,NaN,NaN,0.003907,10.000000,0.006555,11.000000,10.500000


Saved: script\model_outputs\session2_combined_screen_scoreboard.csv


In [19]:
# SESSION 2.6 NUMERIC DEPENDENCE / CORRELATION REVIEW

numeric_dependence = weighted_corr_matrix(
    screening_df,
    NUMERIC_CORR_FEATURES,
    weight_col=CORRELATION_WEIGHT_COL,
)

display(numeric_dependence)
export_csv(numeric_dependence.reset_index().rename(columns={"index": "Feature"}), "session2_numeric_dependence_matrix.csv")

print(
    "Actuarial note: Issue_Age_mod, Attained_Age_mod, and Duration are structurally linked. "
    "This table is for dependence review, not for casual inclusion of all three variables in one final model."
)


,Issue_Age_mod,Attained_Age_mod,Duration,Observation_Year
Issue_Age_mod,1.000000,0.165067,-0.103306,0.002119
Attained_Age_mod,0.165067,1.000000,0.949012,-0.125311
Duration,-0.103306,0.949012,1.000000,-0.137668
Observation_Year,0.002119,-0.125311,-0.137668,1.000000


Saved: script\model_outputs\session2_numeric_dependence_matrix.csv
Actuarial note: Issue_Age_mod, Attained_Age_mod, and Duration are structurally linked. This table is for dependence review, not for casual inclusion of all three variables in one final model.


In [20]:
# SESSION 2.7 CATEGORICAL ASSOCIATION REVIEW

categorical_association = cramers_v_matrix(
    screening_df,
    CATEGORICAL_ASSOC_FEATURES,
    weight_col=CORRELATION_WEIGHT_COL,
)

display(categorical_association)
export_csv(categorical_association.reset_index().rename(columns={"index": "Feature"}), "session2_categorical_association_matrix.csv")


,Age_Ind,Sex,Insurance_Plan,Insurance_Plan_Group,Face_Amount_Band,Face_Band_Group,Slct_Ult_Ind
Age_Ind,1.000000,0.011250,0.208825,0.082528,0.209068,0.132422,0.080452
Sex,0.011250,1.000000,0.023868,0.006703,0.065495,0.024283,0.015300
Insurance_Plan,0.208825,0.023868,1.000000,1.000000,0.239491,0.342080,0.189014
Insurance_Plan_Group,0.082528,0.006703,1.000000,1.000000,0.088698,0.054764,0.016227
Face_Amount_Band,0.209068,0.065495,0.239491,0.088698,1.000000,1.000000,0.419491
Face_Band_Group,0.132422,0.024283,0.342080,0.054764,1.000000,1.000000,0.166218
Slct_Ult_Ind,0.080452,0.015300,0.189014,0.016227,0.419491,0.166218,1.000000


Saved: script\model_outputs\session2_categorical_association_matrix.csv


In [21]:
# SESSION 2.8 TWO-WAY A/E DIAGNOSTIC TABLES

for row_col, col_col in TWO_WAY_AE_PAIRS:
    ae_pivot, exp_pivot = two_way_ae_tables(screening_df, row_col, col_col)

    print(f"\n=== A/E Count : {row_col} × {col_col} ===")
    display(ae_pivot)

    print(f"\n=== Expected Death Count : {row_col} × {col_col} ===")
    display(exp_pivot)

    export_csv(ae_pivot.reset_index(), f"session2_ae_{row_col}_by_{col_col}.csv")
    export_csv(exp_pivot.reset_index(), f"session2_expected_{row_col}_by_{col_col}.csv")



=== A/E Count : Issue_Age_Band × Duration_Band ===


Duration_Band,0-1,2-5,6-10,11-15,16-20,21-26+
Issue_Age_Band,,,,,,
0,NaN,1.193113,1.297556,1.213776,0.954307,1.189026
1-5,1.045837,0.960008,1.087580,0.934632,1.162020,1.232456
6-10,0.571412,0.923032,1.049744,1.161225,1.538691,1.324033
11-15,0.944023,0.937142,1.189540,1.484810,1.644742,1.429738
16-17,0.868168,0.927505,1.320975,1.622801,1.493693,1.409642



=== Expected Death Count : Issue_Age_Band × Duration_Band ===


Duration_Band,0-1,2-5,6-10,11-15,16-20,21-26+
Issue_Age_Band,,,,,,
0,0.000000,119.854562,103.271041,152.416958,638.159585,"28,625.114062"
1-5,58.326478,152.082009,158.149298,421.556320,901.017007,"24,107.968741"
6-10,29.750846,101.838303,287.689319,531.335280,561.516178,"14,483.023495"
11-15,43.431151,281.707684,508.599753,504.441608,521.662290,"17,922.160469"
16-17,56.440687,232.882840,260.413670,253.881967,302.605767,"11,935.652585"


Saved: script\model_outputs\session2_ae_Issue_Age_Band_by_Duration_Band.csv
Saved: script\model_outputs\session2_expected_Issue_Age_Band_by_Duration_Band.csv

=== A/E Count : Attained_Age_Band × Duration_Band ===


...

Saved: script\model_outputs\session2_ae_Insurance_Plan_Group_by_Face_Band_Group.csv
Saved: script\model_outputs\session2_expected_Insurance_Plan_Group_by_Face_Band_Group.csv

=== A/E Count : Age_Ind × Sex ===


Sex,F,M
Age_Ind,,
ALB,1.347506,1.474198
ANB,1.043428,1.224755



=== Expected Death Count : Age_Ind × Sex ===


Sex,F,M
Age_Ind,,
ALB,"12,830.367488","33,026.083965"
ANB,"18,498.648645","39,901.849854"


Saved: script\model_outputs\session2_ae_Age_Ind_by_Sex.csv
Saved: script\model_outputs\session2_expected_Age_Ind_by_Sex.csv


---

# Session 3 · Model fitting, GAM-style candidates, and model comparison


In [24]:
# SESSION 3.1 MODELING CONTROL PANEL

# 80/20 split controls on both Policies_Exposed and Death_Count, not row count alone.
TRAIN_FRACTION = 0.80
SPLIT_RANDOM_STATE = 42
SPLIT_MAX_SEED_TRIES = 200
SPLIT_MIN_RATIO = 0.75
SPLIT_MAX_RATIO = 0.85

# Thin cells are allowed to fall outside the 75/25 to 85/15 range.
SPLIT_MIN_EXPOSURE_FOR_STRICT_CHECK = 100.0
SPLIT_MIN_DEATHS_FOR_STRICT_CHECK = 5.0

SPLIT_REVIEW_COLUMNS = [
    "Observation_Year",
    "Attained_Age",
    "Issue_Age",
    "Duration",
    "Face_Amount_Band",
    "Insurance_Plan",
    "Sex",
    "Slct_Ult_Ind",
    "Age_Ind",
]

# Use coarser fields for the actual stratification key.
SPLIT_STRATA_COLUMNS = [
    "Observation_Year",
    "Age_Ind",
    "Sex",
    "Slct_Ult_Ind",
    "Issue_Age_Band",
    "Duration_Band",
    "Face_Band_Group",
    "Insurance_Plan_Group",
]


# Session 3: GAM-style candidate controls
GAM_ISSUE_AGE_KNOTS = [10, 15]
GAM_DURATION_KNOTS = [1, 14, 25]
GAM_DURATION_UPPER_BOUND = 26
GAM_ATTAINED_AGE_KNOTS = [5, 25, 45]

# Continuity toggle
#   True  -> cubic B-spline basis (production default)
#   False -> raw piecewise linear basis (testing only)
ENFORCE_CONTINUITY = True

GLM_FORMULA_FACTOR_SPLINE = (
    f"{ACTUAL_CNT_COL} ~ C(Age_Ind) + C(Sex) + C(Insurance_Plan) + C(Face_Amount_Band) "
    f"+ C(Slct_Ult_Ind) "
    f"+ bs(Issue_Age_mod, df=6, degree=3, include_intercept=False) "
    f"+ bs(Duration, df=6, degree=3, include_intercept=False)"
)

GAM_FORMULA_CORE = (
    f"{ACTUAL_CNT_COL} ~ C(Sex) "
    f"+ bs(Attained_Age_mod_spline, knots={tuple(GAM_ATTAINED_AGE_KNOTS)}, degree=3, include_intercept=False, lower_bound={ATTAINED_AGE_MIN}, upper_bound={ATTAINED_AGE_MOD_MAX})"
    f"+ C(Sex):bs(Attained_Age_mod_spline, knots={tuple(GAM_ATTAINED_AGE_KNOTS)}, degree=3, include_intercept=False, lower_bound={ATTAINED_AGE_MIN}, upper_bound={ATTAINED_AGE_MOD_MAX})"
)

GAM_FORMULA_SELECT = (
    f"{ACTUAL_CNT_COL} ~ C(Age_Ind) + C(Sex) + C(Slct_Ult_Ind) "
    f"+ bs(Issue_Age_mod_spline, knots={tuple(GAM_ISSUE_AGE_KNOTS)}, degree=3, include_intercept=False, lower_bound={ISSUE_AGE_MIN}, upper_bound={ISSUE_AGE_MOD_MAX}) "
    f"+ bs(Duration_spline, knots={tuple(GAM_DURATION_KNOTS)}, degree=3, include_intercept=False, lower_bound=0, upper_bound={GAM_DURATION_UPPER_BOUND})"
)

GAM_FORMULA_PLAN_FACE = (
    f"{ACTUAL_CNT_COL} ~ C(Sex) + C(Insurance_Plan_Group) + C(Face_Band_Group) "
    f"+ bs(Attained_Age_mod_spline, knots={tuple(GAM_ATTAINED_AGE_KNOTS)}, degree=3, include_intercept=False, lower_bound={ATTAINED_AGE_MIN}, upper_bound={ATTAINED_AGE_MOD_MAX})"
)

# GAM_FORMULA_ATTAINED is built dynamically below so it always respects ENFORCE_CONTINUITY.
ML_FEATURES = SHORTLIST_FEATURES.copy()
MODEL_TARGET_SMOOTH = 0.25
MODEL_RANDOM_STATE = 42


In [ ]:
# SESSION 3.2 BUILD TRAIN / HOLDOUT SETS

train_raw, score_raw, split_balance, split_info = make_balanced_train_score_split(
    scoped_df,
    train_fraction=TRAIN_FRACTION,
    random_state=SPLIT_RANDOM_STATE,
    review_cols=SPLIT_REVIEW_COLUMNS,
    strata_cols=SPLIT_STRATA_COLUMNS,
    min_ratio=SPLIT_MIN_RATIO,
    max_ratio=SPLIT_MAX_RATIO,
    min_full_exposure=SPLIT_MIN_EXPOSURE_FOR_STRICT_CHECK,
    min_full_deaths=SPLIT_MIN_DEATHS_FOR_STRICT_CHECK,
    max_seed_tries=SPLIT_MAX_SEED_TRIES,
)

print(
    f"Train / holdout split: {TRAIN_FRACTION:.0%} / {1 - TRAIN_FRACTION:.0%}  "
    f"selected_seed={split_info['Selected_Seed']}  "
    f"seeds_tried={split_info['Seeds_Tried']}  "
    f"balance_penalty={split_info['Balance_Penalty']:.6f}"
)
print(
    "Strict segment target: train share should stay inside "
    f"{SPLIT_MIN_RATIO:.0%}-{SPLIT_MAX_RATIO:.0%} for both exposure and deaths. "
    "Rows marked Thin data or No deaths are acceptable outliers."
)

display(split_overall_summary(train_raw, score_raw, scoped_df))

strict_reviews = split_balance[split_balance["Any_Strict_Review"]].copy()
if len(strict_reviews) == 0:
    print("All credible segment checks are inside the requested range.")
else:
    print("Credible segment checks outside the requested range. Review these first:")
    display(strict_reviews.head(80))

print("Full split balance table")
display(split_balance)
export_csv(split_balance, "session3_train_holdout_split_balance.csv")

# Session bookkeeping. Each model cell below saves a bundle and one comparison row.
MODEL_BUNDLES = {}
MODEL_ROWS = []

def save_model_bundle(bundle: dict):
    MODEL_BUNDLES[bundle["model_key"]] = bundle
    MODEL_ROWS.append(bundle_comparison_row(bundle))
    print(f"Saved model: {bundle['model_key']}")
    display(bundle["metrics"])


In [ ]:
# SESSION 3.3 KEEP THE HOLDOUT SPLIT ACTIVE

print(f"Training rows: {len(train_raw):,}")
print(f"Holdout rows:  {len(score_raw):,}")


In [23]:
# SESSION 3.4 MODEL 2.5 PER-SEX PENALISED GAM
# All knot search parameters independent per sex

import warnings
from concurrent.futures import ThreadPoolExecutor

try:
    from statsmodels.gam.api import GLMGam, BSplines

    _CORE_FEATURES = ["Attained_Age_mod_spline", "Sex"]
    _CORE_REQUIRED = _CORE_FEATURES + [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL,
        ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed",
    ]

    # ── training cells ───────────────────────────────────────
    _train_mf    = prepare_model_frame(train_raw, _CORE_REQUIRED)
    _score_mf    = prepare_model_frame(score_raw, _CORE_REQUIRED)
    _train_cells = aggregate_cells(_train_mf, _CORE_FEATURES)
    _base_ratio, _base_dev = make_intercept_baseline(_train_cells)
    _train_cells = _train_cells[
        _train_cells["Policies_Exposed"] > 0
    ].reset_index(drop=True)

    _sex_cats    = sorted(_train_cells["Sex"].astype(str).unique().tolist())
    _lower_bound = float(ATTAINED_AGE_MIN)
    _upper_bound = float(ATTAINED_AGE_MOD_MAX)

    # PER-SEX CONTROL PANEL
    _SEX_MIN_SEP = {
        "F": 7,
        "M": 7,
    }
    _SEX_BOUNDARY_GAP = {
        "F": 4,
        "M": 8,
    }
    _SEX_MAX_KNOTS = {
        "F": 3,
        "M": 5,
    }
    _MIN_KNOTS        = 1       # shared floor for all sexes
    _DEFAULT_MIN_SEP      = 10
    _DEFAULT_BOUNDARY_GAP = 4
    _DEFAULT_MAX_KNOTS    = 5

    # ── constraint ────────────────────────────────────────────
    _Z_THRESH = 2.0
    _MIN_EXP  = float(MIN_EXPECTED_FOR_REVIEW)

    # ── alpha grids ──────────────────────────────────────────
    _ALPHA_MAX     = 100.0
    _search_alphas = list(np.logspace(-1, 2, 3))   # [0.1, 3.16, 100]
    _final_alphas  = list(np.logspace(-2, 2, 15))  # 0.01 → 100

    _N_WORKERS = 4

    # ── raw knot candidate pool ───────────────────────────────
    _raw_cands = sorted(list(range(1, 35, 2)) + list(range(35, 67, 5)))

    # ── print configuration ───────────────────────────────────
    print("Per-sex configuration:")
    for _sv in _sex_cats:
        _bg  = _SEX_BOUNDARY_GAP.get(_sv, _DEFAULT_BOUNDARY_GAP)
        _ms  = _SEX_MIN_SEP.get(_sv, _DEFAULT_MIN_SEP)
        _mk  = _SEX_MAX_KNOTS.get(_sv, _DEFAULT_MAX_KNOTS)
        _cds = [k for k in _raw_cands
                if k > _lower_bound + _bg and k < _upper_bound - _bg]
        print(f"  Sex={_sv}: max_knots={_mk}  min_sep={_ms}  "
              f"boundary_gap={_bg}  "
              f"candidates={len(_cds)} (age {min(_cds)}–{max(_cds)})")
    print(f"Constraint: z_thresh={_Z_THRESH}  "
          f"(|A/P-1|>0.20 at expected=100)\n")

    # ── helpers ───────────────────────────────────────────────
    def _sep_ok(new_k, existing, min_sep):
        return all(abs(new_k - k) >= min_sep for k in existing)

    def _fit_one_sex(sv, knots, alpha_grid):
        cells = _train_cells[
            _train_cells["Sex"].astype(str) == sv
        ].reset_index(drop=True)
        if len(cells) < 5:
            return None
        y_sv   = cells[ACTUAL_CNT_COL].astype(float)
        off_sv = np.log(cells["Policies_Exposed"].astype(float).clip(1e-10))
        Xp_sv  = pd.DataFrame({"const": np.ones(len(cells))})
        bs_sv  = BSplines(
            cells[["Attained_Age_mod_spline"]],
            df=[len(knots) + 4],
            degree=[3],
            include_intercept=False,
            knot_kwds=[{
                "knots":       list(knots),
                "lower_bound": _lower_bound,
                "upper_bound": _upper_bound,
            }],
        )
        best_a, best_aic, best_res = None, np.inf, None
        for a in alpha_grid:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    g = GLMGam(y_sv, exog=Xp_sv, smoother=bs_sv,
                               alpha=[a], family=sm.families.Poisson(),
                               offset=off_sv)
                    r = g.fit(disp=False)
                if np.isfinite(r.aic) and r.aic < best_aic:
                    best_a, best_aic, best_res = a, r.aic, r
            except Exception:
                continue
        if best_res is None:
            return None
        return {
            "params":   np.asarray(best_res.params, dtype=float),
            "smoother": bs_sv,
            "alpha":    best_a,
            "aic":      best_aic,
        }

    def _z_score_sex(sv, mi):
        cells = _train_cells[
            _train_cells["Sex"].astype(str) == sv
        ].reset_index(drop=True)
        age_m  = cells[["Attained_Age_mod_spline"]].to_numpy(dtype=float)
        X_spl  = mi["smoother"].transform(age_m)
        X_full = np.column_stack([np.ones((len(cells), 1)), X_spl])
        off    = np.log(cells["Policies_Exposed"].astype(float).clip(1e-10))
        pred   = np.exp(X_full @ mi["params"] + off.values)
        agg = pd.DataFrame({
            "age":     cells["Attained_Age_mod_spline"].values,
            "actual":  cells[ACTUAL_CNT_COL].values.astype(float),
            "exp_vbt": cells[EXPECTED_CNT_COL].values.astype(float),
            "pred":    pred,
        }).groupby("age").sum()
        credible = agg[agg["exp_vbt"] >= _MIN_EXP]
        if len(credible) == 0:
            return (np.inf, np.inf, np.inf)
        ap     = credible["actual"] / credible["pred"].clip(1e-10)
        z      = (ap - 1.0).abs() * np.sqrt(credible["exp_vbt"].clip(1.0))
        n_viol = int((z > _Z_THRESH).sum())
        return (n_viol, float(z.max()), float((np.maximum(z - _Z_THRESH, 0) ** 2).sum()))

    # ── per-sex greedy search ─────────────────────────────────
    _sex_models = {}

    for _sv in _sex_cats:
        _sv_min_sep  = _SEX_MIN_SEP.get(_sv, _DEFAULT_MIN_SEP)
        _sv_bg       = _SEX_BOUNDARY_GAP.get(_sv, _DEFAULT_BOUNDARY_GAP)
        _sv_max_k    = _SEX_MAX_KNOTS.get(_sv, _DEFAULT_MAX_KNOTS)

        # Sex-specific candidate pool
        _sv_cands = [
            k for k in _raw_cands
            if k > _lower_bound + _sv_bg
            and k < _upper_bound - _sv_bg
        ]

        print(f"{'─'*58}")
        _sv_deaths = _train_cells[
            _train_cells["Sex"].astype(str) == _sv
        ][ACTUAL_CNT_COL].sum()
        print(f"Searching knots for Sex={_sv}  "
              f"(trying {_MIN_KNOTS}–{_sv_max_k} knots  "
              f"min_sep={_sv_min_sep}  boundary_gap={_sv_bg})")
        print(f"  Training deaths: {_sv_deaths:,.0f}  "
              f"  Candidates: {len(_sv_cands)} (age {min(_sv_cands)}–{max(_sv_cands)})")

        _best_sv_score  = (np.inf, np.inf, np.inf)
        _best_sv_knots  = None
        _best_sv_nknots = None

        for _nk in range(_MIN_KNOTS, _sv_max_k + 1):
            _knots_sv = []
            _rem_sv   = list(_sv_cands)
            _score_sv = (np.inf, np.inf, np.inf)

            for _step in range(_nk):
                _snap = list(_knots_sv)

                def _try_add_sv(cand, _s=_snap, _sv_=_sv, _ms=_sv_min_sep):
                    if not _sep_ok(cand, _s, _ms):
                        return None
                    trial = tuple(sorted(_s + [cand]))
                    mi = _fit_one_sex(_sv_, trial, _search_alphas)
                    if mi is None:
                        return None
                    sc = _z_score_sex(_sv_, mi)
                    return sc, cand, mi

                with ThreadPoolExecutor(max_workers=_N_WORKERS) as pool:
                    _step_res = [r for r in pool.map(_try_add_sv, _rem_sv)
                                 if r is not None]

                if not _step_res:
                    break
                _step_res.sort(key=lambda x: x[0])
                _sc, _cand, _ = _step_res[0]
                _knots_sv = sorted(_knots_sv + [_cand])
                _rem_sv.remove(_cand)
                _score_sv = _sc

            if not _knots_sv:
                continue

            # ── local swap ────────────────────────────────────
            _swap_imp = True
            while _swap_imp:
                _swap_imp = False
                for _ok in list(_knots_sv):
                    _rem_k = [k for k in _knots_sv if k != _ok]
                    _spool = [c for c in _sv_cands
                              if c not in _knots_sv
                              and _sep_ok(c, _rem_k, _sv_min_sep)]

                    def _try_swap_sv(nk, _o=_ok, _b=list(_knots_sv),
                                     _sv_=_sv, _ms=_sv_min_sep):
                        rem = [k for k in _b if k != _o]
                        if not _sep_ok(nk, rem, _ms):
                            return None
                        trial = tuple(sorted(rem + [nk]))
                        mi = _fit_one_sex(_sv_, trial, _search_alphas)
                        if mi is None:
                            return None
                        sc = _z_score_sex(_sv_, mi)
                        return sc, _o, nk, sorted(rem + [nk])

                    with ThreadPoolExecutor(max_workers=_N_WORKERS) as pool:
                        for r in pool.map(_try_swap_sv, _spool):
                            if r and r[0] < _score_sv:
                                _score_sv, _, _, _knots_sv = r
                                _swap_imp = True

            _gaps = [_knots_sv[i+1]-_knots_sv[i]
                     for i in range(len(_knots_sv)-1)]
            print(f"  n_knots={_nk}: knots={_knots_sv}  "
                  f"gaps={_gaps}  "
                  f"z_viol={_score_sv[0]}  max_z={_score_sv[1]:.2f}")

            if _score_sv < _best_sv_score:
                _best_sv_score  = _score_sv
                _best_sv_knots  = _knots_sv
                _best_sv_nknots = _nk

            if _best_sv_score[0] == 0 and _best_sv_score[1] < 1.5:
                print(f"  → 0 violations — stopping at {_nk} knots")
                break

        if _best_sv_knots is None:
            raise RuntimeError(f"No valid knot set found for Sex={_sv}")

        # ── final refit ───────────────────────────────────────
        _best_sv_mi = _fit_one_sex(_sv, _best_sv_knots, _final_alphas)
        if _best_sv_mi is None:
            raise RuntimeError(f"Final refit failed for Sex={_sv}")

        _alpha_note = ""
        if _best_sv_mi["alpha"] >= _ALPHA_MAX:
            _alpha_note = (
                "  (at ceiling — data supports near-linear curve; "
                "predictions are still correct)"
            )
        elif _best_sv_mi["alpha"] >= _ALPHA_MAX * 0.5:
            _alpha_note = "  (moderate-high)"

        _sv_final_sc = _z_score_sex(_sv, _best_sv_mi)
        print(f"\n  Sex={_sv} selected: {_best_sv_nknots} knots = {_best_sv_knots}")
        print(f"  Final alpha={_best_sv_mi['alpha']:.3g}  "
              f"AIC={_best_sv_mi['aic']:.2f}{_alpha_note}")
        print(f"  Final z_violations={_sv_final_sc[0]}  "
              f"max_z={_sv_final_sc[1]:.3f}")

        _sex_models[_sv] = _best_sv_mi
        _sex_models[_sv]["knots"]   = _best_sv_knots
        _sex_models[_sv]["n_knots"] = _best_sv_nknots

    # ── wrapper ──────────────────────────────────────────────
    class _GAMCoreWrapper:
        def __init__(self, sex_models, sex_cats):
            self._sex_models = sex_models
            self._sex_cats   = sex_cats
            self.params = np.concatenate(
                [m["params"] for m in sex_models.values()]
            )
        def predict(self, df, offset=None):
            df_r = df.reset_index(drop=True)
            off  = np.asarray(offset, dtype=float)
            pred = np.zeros(len(df_r))
            for sv, mi in self._sex_models.items():
                mask   = (df_r["Sex"].astype(str) == sv).values
                if not mask.any():
                    continue
                age_m  = df_r.loc[mask, ["Attained_Age_mod_spline"]
                                  ].to_numpy(dtype=float)
                X_spl  = mi["smoother"].transform(age_m)
                X_full = np.column_stack([np.ones((mask.sum(), 1)), X_spl])
                pred[mask] = np.exp(X_full @ mi["params"] + off[mask])
            return pred

    _wrapper = _GAMCoreWrapper(_sex_models, _sex_cats)

    # ── model summary ─────────────────────────────────────────
    print(f"\n{'═'*58}")
    print("Model summary:")
    for _sv, _mi in _sex_models.items():
        _gaps = [_mi["knots"][i+1]-_mi["knots"][i]
                 for i in range(len(_mi["knots"])-1)]
        print(f"  Sex={_sv}: {_mi['n_knots']} knots {_mi['knots']}  "
              f"gaps={_gaps}  "
              f"alpha={_mi['alpha']:.3g}  "
              f"params={len(_mi['params'])}")
    print(f"  Total params: {len(_wrapper.params)}")

    # ── bundle ───────────────────────────────────────────────
    bundle_gam_core = {
        "model_key":   "poisson_gam_core_per_sex",
        "model_type":  "statsmodels_glm",
        "label":       "poisson_gam_core",
        "formula": (
            "GLMGam per-sex: "
            + "  ".join(f"Sex={sv}: {mi['n_knots']}k{mi['knots']}"
                        for sv, mi in _sex_models.items())
        ),
        "features":              _CORE_FEATURES,
        "result":                _wrapper,
        "is_pygam_core":         True,
        "_sex_models_meta": {
            sv: {"alpha":   mi["alpha"],
                 "knots":   mi["knots"],
                 "n_knots": mi["n_knots"]}
            for sv, mi in _sex_models.items()
        },
        "_bs_df":        max(mi["n_knots"] for mi in _sex_models.values()) + 4,
        "_lower_bound":  _lower_bound,
        "_upper_bound":  _upper_bound,
        "_sex_cats":     _sex_cats,
        "category_levels":       {},
        "design_columns":        [],
        "baseline_ratio":        _base_ratio,
        "baseline_deviance":     _base_dev,
        "parameter_count":       int(len(_wrapper.params)),
        "likelihood_comparable": "Yes",
        "aic":  float(sum(mi["aic"] for mi in _sex_models.values())),
        "bic":  np.nan,
        "null_deviance":         np.nan,
        "train_cells":           _train_cells,
    }

    _scored_train = score_with_bundle(bundle_gam_core, train_raw)
    _scored_score = score_with_bundle(bundle_gam_core, score_raw)
    bundle_gam_core["metrics"] = pd.DataFrame([
        evaluate_predictions(
            _scored_train[ACTUAL_CNT_COL],
            _scored_train[EXPECTED_CNT_COL],
            _scored_train["Predicted_Deaths"],
            "Train",
        ),
        evaluate_predictions(
            _scored_score[ACTUAL_CNT_COL],
            _scored_score[EXPECTED_CNT_COL],
            _scored_score["Predicted_Deaths"],
            "Scored_Data",
        ),
    ])

    save_model_bundle(bundle_gam_core)
    display(bundle_gam_core["metrics"])
    print(f"\nSaved: poisson_gam_core  "
          f"(per-sex knots: "
          + ", ".join(f"{sv}={mi['knots']}" for sv, mi in _sex_models.items())
          + ")")

except Exception as exc:
    print(f"Model failed: poisson_gam_core -> {exc}")
    import traceback; traceback.print_exc()


Model failed: poisson_gam_core -> name 'train_raw' is not defined


Traceback (most recent call last):
  File "C:\Users\sober\AppData\Local\Temp\ipykernel_3104\3221446039.py", line 18, in <module>
    _train_mf    = prepare_model_frame(train_raw, _CORE_REQUIRED)
                                       ^^^^^^^^^
NameError: name 'train_raw' is not defined


In [ ]:
# SESSION 3.5 MODEL 1 FACTOR + SPLINE BENCHMARK GLM

try:
    bundle_factor_spline = fit_formula_glm_bundle(
        model_key="poisson_glm_factor_spline",
        formula=GLM_FORMULA_FACTOR_SPLINE,
        train_df=train_raw,
        score_df=score_raw,
    )
    save_model_bundle(bundle_factor_spline)
except Exception as exc:
    print(f"Model failed: poisson_glm_factor_spline -> {exc}")


In [ ]:
# SESSION 3.6 MODEL 2 COMBINED-SEX PENALISED GAM
# Single model; shared knots; Sex × spline interaction;

import warnings
from concurrent.futures import ThreadPoolExecutor
from scipy.special import gammaln

try:
    from statsmodels.gam.api import BSplines

    # ── features ─────────────────────────────────────────────
    _CORE_FEATURES = ["Attained_Age_mod_spline", "Sex"]
    _CORE_REQUIRED = _CORE_FEATURES + [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL,
        ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed",
    ]

    _train_mf    = prepare_model_frame(train_raw, _CORE_REQUIRED)
    _train_cells = aggregate_cells(_train_mf, _CORE_FEATURES)
    _base_ratio, _base_dev = make_intercept_baseline(_train_cells)
    _train_cells = _train_cells[
        _train_cells["Policies_Exposed"] > 0
    ].reset_index(drop=True)

    _lower_bound = float(ATTAINED_AGE_MIN)
    _upper_bound = float(ATTAINED_AGE_MOD_MAX)

    # ============================================================
    # COMBINED-SEX CONTROL PANEL
    # ============================================================
    _COMBINED_MIN_SEP      = 8
    _COMBINED_BOUNDARY_GAP = 6
    _COMBINED_MAX_KNOTS    = 6
    _MIN_KNOTS             = 1

    _Z_THRESH = 2.0
    _MIN_EXP  = float(MIN_EXPECTED_FOR_REVIEW)

    _ALPHA_MAX     = 500.0
    _search_alphas = list(np.logspace(-1, 2.7, 3))   # [0.1, 5.6, 500]
    _final_alphas  = list(np.logspace(-2, 2.7, 15))  # 0.01 → 500

    _N_WORKERS = 4

    _raw_cands = sorted(list(range(1, 35, 1)) + list(range(35, 67, 5)))
    _cands = [
        k for k in _raw_cands
        if k > _lower_bound + _COMBINED_BOUNDARY_GAP
        and k < _upper_bound - _COMBINED_BOUNDARY_GAP
    ]

    print("Combined-sex penalised GAM configuration:")
    print(f"  max_knots={_COMBINED_MAX_KNOTS}  min_sep={_COMBINED_MIN_SEP}  "
          f"boundary_gap={_COMBINED_BOUNDARY_GAP}")
    print(f"  Candidates: {len(_cands)} (age {min(_cands)}–{max(_cands)})")
    print(f"  Training deaths: {_train_cells[ACTUAL_CNT_COL].sum():,.0f}  "
          f"  Cells: {len(_train_cells)}")
    print(f"Constraint: z_thresh={_Z_THRESH}  "
          f"(|A/P-1|>0.20 at expected≥{_MIN_EXP})\n")

    # ── penalised PIRLS ───────────────────────────────────────
    def _fit_penalized_interaction(cells, knots, alpha):
        y   = cells[ACTUAL_CNT_COL].astype(float).values
        off = np.log(cells["Policies_Exposed"].astype(float).clip(1e-10).values)
        sex = cells["Sex"].astype(str).values

        is_M = (sex == "M").astype(float)
        is_F = 1.0 - is_M
        n    = len(y)

        bs = BSplines(
            cells[["Attained_Age_mod_spline"]],
            df=[len(knots) + 4],   # hint; actual width read below
            degree=[3],
            include_intercept=False,
            knot_kwds=[{"knots":       list(knots),
                         "lower_bound": _lower_bound,
                         "upper_bound": _upper_bound}],
        )
        B  = bs.basis                   # (n, nb_actual)
        nb = B.shape[1]                 # ← actual width after include_intercept=False

        B_F = B * is_F[:, None]
        B_M = B * is_M[:, None]
        X   = np.column_stack([np.ones(n), is_M, B_F, B_M])
        p   = X.shape[1]               # = 2 + 2*nb

        S = bs.penalty_matrices[0]     # (nb, nb) — now guaranteed to match
        P = np.zeros((p, p))
        P[2:2+nb,   2:2+nb]   = alpha * S
        P[2+nb:p,   2+nb:p]   = alpha * S

        # PIRLS
        beta = np.zeros(p)
        for _ in range(50):
            eta = X @ beta + off
            mu  = np.exp(np.clip(eta, -30, 30))
            W   = mu
            XtWX = (X * W[:, None]).T @ X
            XtWz = (X * W[:, None]).T @ (
                eta - off + (y - mu) / W.clip(1e-15)
            )
            try:
                beta_new = np.linalg.solve(XtWX + P, XtWz)
            except np.linalg.LinAlgError:
                return None
            if np.max(np.abs(beta_new - beta)) < 1e-7:
                beta = beta_new
                break
            beta = beta_new

        mu_f  = np.exp(np.clip(X @ beta + off, -30, 30))
        ll    = float(np.sum(y * np.log(mu_f.clip(1e-15)) - mu_f - gammaln(y + 1)))
        XtWXf = (X * mu_f[:, None]).T @ X
        try:
            edf = float(np.trace(np.linalg.solve(XtWXf + P, XtWXf)))
        except np.linalg.LinAlgError:
            edf = float(p)

        return {
            "beta":    beta,
            "bs":      bs,
            "aic":     -2.0 * ll + 2.0 * edf,
            "edf":     edf,
            "ll":      ll,
            "n_basis": nb,
            "alpha":   alpha,
        }

    # ── predict wrapper for score_with_bundle ─────────────────
    class _PenalizedInteractionGAM:
        """
        Wrapper with the same predict(df, offset) interface as a
        statsmodels GLM result, so score_with_bundle works unchanged.
        """
        def __init__(self, beta, bs, n_basis):
            self.params  = beta
            self._bs     = bs
            self._nb     = n_basis

        def predict(self, df, offset=None):
            df_r = df.reset_index(drop=True)
            n    = len(df_r)
            sex  = df_r["Sex"].astype(str).values
            age  = df_r["Attained_Age_mod_spline"].astype(float).values
            is_M = (sex == "M").astype(float)
            is_F = 1.0 - is_M
            B    = self._bs.transform(
                df_r[["Attained_Age_mod_spline"]].astype(float).values
            )
            B_F  = B * is_F[:, None]
            B_M  = B * is_M[:, None]
            X    = np.column_stack([np.ones(n), is_M, B_F, B_M])
            eta  = X @ self.params
            if offset is not None:
                eta = eta + np.asarray(offset, dtype=float)
            return np.exp(eta)

    # ── helpers ───────────────────────────────────────────────
    def _sep_ok(new_k, existing, min_sep):
        return all(abs(new_k - k) >= min_sep for k in existing)

    def _fit_one(knots, alpha_grid):
        """Try alpha_grid, return best AIC fit for this knot set."""
        best_aic, best_mi = np.inf, None
        for a in alpha_grid:
            mi = _fit_penalized_interaction(_train_cells, list(knots), a)
            if mi is not None and np.isfinite(mi["aic"]) and mi["aic"] < best_aic:
                best_aic = mi["aic"]
                best_mi  = mi
        return best_mi

    def _z_score(mi):
        """Z-score violations on combined (M+F) cells, aggregated by age."""
        wrapper = _PenalizedInteractionGAM(mi["beta"], mi["bs"], mi["n_basis"])
        off  = np.log(_train_cells["Policies_Exposed"].astype(float).clip(1e-10))
        pred = wrapper.predict(_train_cells, offset=off.values)
        agg  = pd.DataFrame({
            "age":     _train_cells["Attained_Age_mod_spline"].values,
            "actual":  _train_cells[ACTUAL_CNT_COL].values.astype(float),
            "exp_vbt": _train_cells[EXPECTED_CNT_COL].values.astype(float),
            "pred":    pred,
        }).groupby("age").sum()
        credible = agg[agg["exp_vbt"] >= _MIN_EXP]
        if len(credible) == 0:
            return (np.inf, np.inf, np.inf)
        ap    = credible["actual"] / credible["pred"].clip(1e-10)
        z     = (ap - 1.0).abs() * np.sqrt(credible["exp_vbt"].clip(1.0))
        n_viol = int((z > _Z_THRESH).sum())
        return (n_viol, float(z.max()),
                float((np.maximum(z - _Z_THRESH, 0) ** 2).sum()))

    # ── greedy forward search ─────────────────────────────────
    _best_score  = (np.inf, np.inf, np.inf)
    _best_knots  = None
    _best_nknots = None

    for _nk in range(_MIN_KNOTS, _COMBINED_MAX_KNOTS + 1):
        _knots = []
        _rem   = list(_cands)
        _score = (np.inf, np.inf, np.inf)

        for _step in range(_nk):
            _snap = list(_knots)

            def _try_add(cand, _s=_snap, _ms=_COMBINED_MIN_SEP):
                if not _sep_ok(cand, _s, _ms):
                    return None
                trial = tuple(sorted(_s + [cand]))
                mi = _fit_one(trial, _search_alphas)
                if mi is None:
                    return None
                return _z_score(mi), cand, mi

            with ThreadPoolExecutor(max_workers=_N_WORKERS) as pool:
                _step_res = [r for r in pool.map(_try_add, _rem)
                             if r is not None]
            if not _step_res:
                break
            _step_res.sort(key=lambda x: x[0])
            _sc, _cand, _ = _step_res[0]
            _knots = sorted(_knots + [_cand])
            _rem.remove(_cand)
            _score = _sc

        if not _knots:
            continue

        # ── local swap ─────────────────────────────────────────
        _swap_imp = True
        while _swap_imp:
            _swap_imp = False
            for _ok in list(_knots):
                _rem_k = [k for k in _knots if k != _ok]
                _spool = [c for c in _cands
                          if c not in _knots
                          and _sep_ok(c, _rem_k, _COMBINED_MIN_SEP)]

                def _try_swap(nk, _o=_ok, _b=list(_knots),
                              _ms=_COMBINED_MIN_SEP):
                    rem = [k for k in _b if k != _o]
                    if not _sep_ok(nk, rem, _ms):
                        return None
                    trial = tuple(sorted(rem + [nk]))
                    mi = _fit_one(trial, _search_alphas)
                    if mi is None:
                        return None
                    sc = _z_score(mi)
                    return sc, _o, nk, sorted(rem + [nk])

                with ThreadPoolExecutor(max_workers=_N_WORKERS) as pool:
                    for r in pool.map(_try_swap, _spool):
                        if r and r[0] < _score:
                            _score, _, _, _knots = r
                            _swap_imp = True

        _gaps = [_knots[i+1]-_knots[i] for i in range(len(_knots)-1)]
        print(f"  n_knots={_nk}: knots={_knots}  gaps={_gaps}  "
              f"z_viol={_score[0]}  max_z={_score[1]:.2f}")

        if _score < _best_score:
            _best_score  = _score
            _best_knots  = _knots
            _best_nknots = _nk

        if _best_score[0] == 0 and _best_score[1] < 1.5:
            print(f"  → 0 violations — stopping at {_nk} knots")
            break

    if _best_knots is None:
        raise RuntimeError("No valid knot set found for combined model")

    # ── final refit with precise alpha grid ───────────────────
    _final_mi = _fit_one(_best_knots, _final_alphas)
    if _final_mi is None:
        raise RuntimeError("Final penalised fit failed")

    _alpha_note = ""
    if _final_mi["alpha"] >= _ALPHA_MAX:
        _alpha_note = ("  (at ceiling — near-linear curve; "
                       "predictions still correct)")
    elif _final_mi["alpha"] >= _ALPHA_MAX * 0.5:
        _alpha_note = "  (moderate-high)"

    _final_sc   = _z_score(_final_mi)
    _gaps_final = [_best_knots[i+1]-_best_knots[i]
                   for i in range(len(_best_knots)-1)]

    print(f"\n{'═'*58}")
    print(f"Selected: {_best_nknots} knots = {_best_knots}  "
          f"gaps={_gaps_final}")
    print(f"Final alpha={_final_mi['alpha']:.3g}{_alpha_note}")
    print(f"AIC={_final_mi['aic']:.2f}  edf={_final_mi['edf']:.2f}  "
          f"params={len(_final_mi['beta'])}")
    print(f"Final z_violations={_final_sc[0]}  "
          f"max_z={_final_sc[1]:.3f}")

    # ── bundle ────────────────────────────────────────────────
    _wrapper = _PenalizedInteractionGAM(
        _final_mi["beta"], _final_mi["bs"], _final_mi["n_basis"]
    )

    bundle_gam_core = {
        "model_key":              "poisson_gam_core",
        "model_type":             "statsmodels_glm",   # uses score_with_bundle's GLM path
        "label":                  "poisson_gam_core",
        "formula":                (
            f"penalised Sex×BSpline interaction  "
            f"knots={_best_knots}  alpha={_final_mi['alpha']:.3g}"
        ),
        "features":               _CORE_FEATURES,
        "result":                 _wrapper,
        "is_pygam_core":          False,
        "_penalized_interaction": True,   # signals cell 40 refit path
        "_knots":                 _best_knots,
        "_n_knots":               _best_nknots,
        "_alpha":                 _final_mi["alpha"],
        "_lower_bound":           _lower_bound,
        "_upper_bound":           _upper_bound,
        "category_levels":        {},
        "design_columns":         [],
        "baseline_ratio":         _base_ratio,
        "baseline_deviance":      _base_dev,
        "parameter_count":        int(len(_final_mi["beta"])),
        "likelihood_comparable":  "Yes",
        "aic":                    float(_final_mi["aic"]),
        "bic":                    np.nan,
        "null_deviance":          np.nan,
        "train_cells":            _train_cells,
    }

    _scored_train = score_with_bundle(bundle_gam_core, train_raw)
    _scored_score = score_with_bundle(bundle_gam_core, score_raw)
    bundle_gam_core["metrics"] = pd.DataFrame([
        evaluate_predictions(
            _scored_train[ACTUAL_CNT_COL],
            _scored_train[EXPECTED_CNT_COL],
            _scored_train["Predicted_Deaths"],
            "Train",
        ),
        evaluate_predictions(
            _scored_score[ACTUAL_CNT_COL],
            _scored_score[EXPECTED_CNT_COL],
            _scored_score["Predicted_Deaths"],
            "Scored_Data",
        ),
    ])

    save_model_bundle(bundle_gam_core)
    display(bundle_gam_core["metrics"])
    print(f"\nSaved: poisson_gam_core  "
          f"(combined-sex penalised, knots={_best_knots}, "
          f"alpha={_final_mi['alpha']:.3g})")

except Exception as exc:
    print(f"Model failed: poisson_gam_core -> {exc}")
    import traceback; traceback.print_exc()


In [ ]:
# SESSION 3.7 MODEL 2.6 ADD ISSUE_AGE_MOD LINEARLY
#              Sex × Attained_Age_mod B-spline (knots from GAM_ATTAINED_AGE_KNOTS)
#              + Issue_Age_mod as a single linear predictor.

try:
    _formula_26 = (
        f"{ACTUAL_CNT_COL} ~ C(Sex) + Issue_Age_mod_spline"
        f" + bs(Attained_Age_mod_spline,"
        f" knots={tuple(GAM_ATTAINED_AGE_KNOTS)}, degree=3,"
        f" include_intercept=False,"
        f" lower_bound={ATTAINED_AGE_MIN},"
        f" upper_bound={ATTAINED_AGE_MOD_MAX})"
        f" + C(Sex):bs(Attained_Age_mod_spline,"
        f" knots={tuple(GAM_ATTAINED_AGE_KNOTS)}, degree=3,"
        f" include_intercept=False,"
        f" lower_bound={ATTAINED_AGE_MIN},"
        f" upper_bound={ATTAINED_AGE_MOD_MAX})"
    )
    print(f"Model 2.6 formula:\n  {_formula_26}\n")

    bundle_gam_core_26 = fit_formula_glm_bundle(
        model_key="poisson_gam_core_26",
        formula=_formula_26,
        train_df=train_raw,
        score_df=score_raw,
    )
    save_model_bundle(bundle_gam_core_26)

except Exception as exc:
    print(f"Model failed: poisson_gam_core_26 -> {exc}")
    import traceback; traceback.print_exc()


In [ ]:
# SESSION 3.8 MODEL 2.7 ADD ISSUE_AGE_MOD AS A SPLINE

try:
    _formula_27 = (
        f"{ACTUAL_CNT_COL} ~ C(Sex)"
        f" + bs(Issue_Age_mod_spline,"
        f" knots={tuple(GAM_ISSUE_AGE_KNOTS)}, degree=3,"
        f" include_intercept=False,"
        f" lower_bound={ISSUE_AGE_MIN},"
        f" upper_bound={ISSUE_AGE_MOD_MAX})"
        f" + bs(Attained_Age_mod_spline,"
        f" knots={tuple(GAM_ATTAINED_AGE_KNOTS)}, degree=3,"
        f" include_intercept=False,"
        f" lower_bound={ATTAINED_AGE_MIN},"
        f" upper_bound={ATTAINED_AGE_MOD_MAX})"
        f" + C(Sex):bs(Attained_Age_mod_spline,"
        f" knots={tuple(GAM_ATTAINED_AGE_KNOTS)}, degree=3,"
        f" include_intercept=False,"
        f" lower_bound={ATTAINED_AGE_MIN},"
        f" upper_bound={ATTAINED_AGE_MOD_MAX})"
    )
    print(f"Model 2.7 formula:\n  {_formula_27}\n")

    bundle_gam_core_27 = fit_formula_glm_bundle(
        model_key="poisson_gam_core_27",
        formula=_formula_27,
        train_df=train_raw,
        score_df=score_raw,
    )
    save_model_bundle(bundle_gam_core_27)

except Exception as exc:
    print(f"Model failed: poisson_gam_core_27 -> {exc}")
    import traceback; traceback.print_exc()


In [ ]:
# SESSION 3.9 MODEL 3 ? WHITTAKER-HENDERSON GRADUATED RATE MODEL
#
# The WH smoother is a classical actuarial graduation method.
# It minimises:
#   H = Σᵢ wᵢ(yᵢ − γᵢ)² + λ Σᵢ (Δᵈγᵢ)²
# where wᵢ = Policies_Exposed, yᵢ = crude rate = Deaths / Exposure,
# γᵢ = graduated (smoothed) rate, d = order of differences (d=2),
# λ = smoothing parameter.
#
# Matrix form: γ = (W + λ DᵀD)⁻¹ W y
#
# Applied per sex over a regular integer Attained_Age_mod grid
# [ATTAINED_AGE_MIN, ATTAINED_AGE_MOD_MAX].  Predictions for any
# record are looked up from the nearest graduated age node.
#
# Actuarial note: WH is a graduation on the marginal age curve.
# It does not incorporate issue-age or duration covariates.
# It is included here as a classical benchmark for smooth age curves.
# ============================================================

try:
    _WH_LAMBDA = 100.0   # smoothing strength (higher = smoother)
    _WH_D      = 2       # order of differences (2 = penalise curvature)
    _WH_FEATURES = ["Attained_Age_mod_spline", "Sex"]
    _WH_REQUIRED  = _WH_FEATURES + [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL,
        ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed",
    ]

    # ── training cells ───────────────────────────────────────────
    _wh_train_mf    = prepare_model_frame(train_raw, _WH_REQUIRED)
    _wh_train_cells = aggregate_cells(_wh_train_mf, _WH_FEATURES)
    _wh_base_ratio, _wh_base_dev = make_intercept_baseline(_wh_train_cells)
    _wh_train_cells = _wh_train_cells[
        _wh_train_cells["Policies_Exposed"] > 0
    ].reset_index(drop=True)

    # ── difference matrix of order d ─────────────────────────────
    def _diff_matrix(n: int, d: int) -> np.ndarray:
        D = np.eye(n)
        for _ in range(d):
            D = np.diff(D, axis=0)
        return D

    # ── WH graduation per sex ────────────────────────────────────
    _wh_sex_cats = sorted(_wh_train_cells["Sex"].astype(str).unique().tolist())
    _wh_tables   = {}   # sex → (ages_array, graduated_rates_array)

    for _sv in _wh_sex_cats:
        _cells_sv = (
            _wh_train_cells[_wh_train_cells["Sex"].astype(str) == _sv]
            .copy()
            .sort_values("Attained_Age_mod_spline")
            .reset_index(drop=True)
        )
        _ages   = _cells_sv["Attained_Age_mod_spline"].to_numpy(dtype=float)
        _deaths = _cells_sv[ACTUAL_CNT_COL].to_numpy(dtype=float)
        _exp    = _cells_sv["Policies_Exposed"].to_numpy(dtype=float).clip(1e-10)
        _crude  = _deaths / _exp

        n = len(_ages)
        _W  = np.diag(_exp)
        _Dd = _diff_matrix(n, _WH_D)
        _A  = _W + _WH_LAMBDA * (_Dd.T @ _Dd)
        _b  = _W @ _crude
        _gamma = np.linalg.solve(_A, _b)
        _gamma = np.clip(_gamma, 1e-10, None)   # rates must be positive

        _wh_tables[_sv] = (_ages, _gamma)
        print(f"  Sex={_sv}: n_nodes={n}  "
              f"age=[{_ages[0]:.1f}, {_ages[-1]:.1f}]  "
              f"lambda={_WH_LAMBDA}  d={_WH_D}  "
              f"crude_min={_crude.min():.6f}  "
              f"crude_max={_crude.max():.6f}  "
              f"grad_min={_gamma.min():.6f}  "
              f"grad_max={_gamma.max():.6f}")

    # ── predict wrapper ──────────────────────────────────────────
    class _WHWrapper:
        """
        Lookup-based wrapper with the same predict(df, offset) interface
        as a statsmodels GLM result, so score_with_bundle works unchanged.
        The graduated rate for any record is obtained by linear interpolation
        from the per-sex WH graduation table; Policies_Exposed is the offset.
        """
        def __init__(self, tables: dict, sex_cats: list):
            self._tables   = tables
            self._sex_cats = sex_cats
            # Expose a params attribute so bundle_comparison_row doesn't crash
            n_params = sum(len(g) for _, g in tables.values())
            self.params = np.zeros(n_params)

        def predict(self, df, offset=None):
            df_r   = df.reset_index(drop=True)
            n      = len(df_r)
            pred   = np.zeros(n)
            age_v  = df_r["Attained_Age_mod_spline"].to_numpy(dtype=float)
            sex_v  = df_r["Sex"].astype(str).values
            exp_v  = df_r["Policies_Exposed"].to_numpy(dtype=float)

            for sv, (ages, gamma) in self._tables.items():
                mask = (sex_v == sv)
                if not mask.any():
                    continue
                # Linear interpolation; extrapolate flat beyond endpoints
                rate_hat = np.interp(age_v[mask], ages, gamma)
                pred[mask] = rate_hat * exp_v[mask]

            return pred

    _wh_wrapper = _WHWrapper(_wh_tables, _wh_sex_cats)

    # ── build bundle ─────────────────────────────────────────────
    _scored_train_wh = _wh_train_cells.copy()
    # Manually compute predictions on train cells for metrics
    _wh_pred_train = _wh_wrapper.predict(
        prepare_model_frame(train_raw, _WH_REQUIRED)
    )
    _wh_scored_train = score_with_bundle(
        {
            "model_key":   "whittaker_henderson",
            "model_type":  "statsmodels_glm",
            "features":    _WH_FEATURES,
            "result":      _wh_wrapper,
            "category_levels": {},
            "design_columns":  [],
        },
        train_raw,
    )
    _wh_scored_score = score_with_bundle(
        {
            "model_key":   "whittaker_henderson",
            "model_type":  "statsmodels_glm",
            "features":    _WH_FEATURES,
            "result":      _wh_wrapper,
            "category_levels": {},
            "design_columns":  [],
        },
        score_raw,
    )

    bundle_wh = {
        "model_key":             "whittaker_henderson",
        "model_type":            "statsmodels_glm",
        "label":                 "whittaker_henderson",
        "formula":               f"WH grad: lambda={_WH_LAMBDA}  d={_WH_D}  by Sex",
        "features":              _WH_FEATURES,
        "result":                _wh_wrapper,
        "is_pygam_core":         False,
        "_wh_tables":            _wh_tables,
        "_wh_lambda":            _WH_LAMBDA,
        "_wh_d":                 _WH_D,
        "_wh_sex_cats":          _wh_sex_cats,
        "category_levels":       {},
        "design_columns":        [],
        "baseline_ratio":        _wh_base_ratio,
        "baseline_deviance":     _wh_base_dev,
        "parameter_count":       int(sum(len(g) for _, g in _wh_tables.values())),
        "likelihood_comparable": "No",
        "aic":                   np.nan,
        "bic":                   np.nan,
        "null_deviance":         np.nan,
        "train_cells":           _wh_train_cells,
        "metrics": pd.DataFrame([
            evaluate_predictions(
                _wh_scored_train[ACTUAL_CNT_COL],
                _wh_scored_train[EXPECTED_CNT_COL],
                _wh_scored_train["Predicted_Deaths"],
                "Train",
            ),
            evaluate_predictions(
                _wh_scored_score[ACTUAL_CNT_COL],
                _wh_scored_score[EXPECTED_CNT_COL],
                _wh_scored_score["Predicted_Deaths"],
                "Scored_Data",
            ),
        ]),
    }

    save_model_bundle(bundle_wh)

except Exception as exc:
    print(f"Model failed: whittaker_henderson -> {exc}")
    import traceback; traceback.print_exc()


In [ ]:
# SESSION 3.10 MODEL 4 ? PROJECTION PURSUIT REGRESSION
#
# PPR models the response as a sum of smooth ridge functions:
#   log(μ/E) = Σ_m f_m(αₘᵀ X)
# where each αₘ is a learned projection direction and f_m is a
# smooth function estimated by spline smoothing (backfitting).
#
# Python implementation: pyGAM's PoissonGAM provides an additive
# smooth model that shares the core mathematical spirit of PPR —
# each term f(Xⱼ) is a smooth function of a single feature, and
# the features here are chosen to be meaningful projections:
#   • s(Attained_Age_mod_spline)  — principal age projection
#   • s(Issue_Age_mod_spline)     — issue-age projection
#   • f(Sex)                      — binary projection
#
# Offset: log(Policies_Exposed) is passed via pyGAM's exposure arg.
# The model is fitted directly on the aggregated cell data.
#
# Actuarial note: PPR/pyGAM is included as a data-driven nonlinear
# challenger.  Its smooth terms are not penalised by the actuary's
# explicit knot choices; the number of spline basis functions (n_splines)
# controls smoothness.
# ============================================================

try:
    from pygam import PoissonGAM, s, f as gam_f

    _PPR_N_SPLINES  = 20   # basis width per smooth term
    _PPR_LAM_GRID   = np.logspace(-3, 3, 20)   # lambda search grid

    _PPR_FEATURES = ["Attained_Age_mod_spline", "Issue_Age_mod_spline", "Sex"]
    _PPR_REQUIRED = _PPR_FEATURES + [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL,
        ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed",
    ]

    # ── training cells ───────────────────────────────────────────
    _ppr_train_mf    = prepare_model_frame(train_raw, _PPR_REQUIRED)
    _ppr_score_mf    = prepare_model_frame(score_raw, _PPR_REQUIRED)
    _ppr_train_cells = aggregate_cells(_ppr_train_mf, _PPR_FEATURES)
    _ppr_base_ratio, _ppr_base_dev = make_intercept_baseline(_ppr_train_cells)
    _ppr_train_cells = _ppr_train_cells[
        _ppr_train_cells["Policies_Exposed"] > 0
    ].reset_index(drop=True)

    # ── encode Sex as integer (pyGAM factor term) ────────────────
    _ppr_sex_cats = sorted(_ppr_train_cells["Sex"].astype(str).unique())
    _sex_map      = {sv: i for i, sv in enumerate(_ppr_sex_cats)}

    def _encode_ppr(cells: pd.DataFrame) -> np.ndarray:
        aa  = cells["Attained_Age_mod_spline"].to_numpy(dtype=float)
        ia  = cells["Issue_Age_mod_spline"].to_numpy(dtype=float)
        sex = cells["Sex"].astype(str).map(_sex_map).fillna(0).to_numpy(dtype=float)
        return np.column_stack([aa, ia, sex])

    _X_train = _encode_ppr(_ppr_train_cells)
    _y_train = _ppr_train_cells[ACTUAL_CNT_COL].to_numpy(dtype=float)
    _E_train = _ppr_train_cells["Policies_Exposed"].to_numpy(dtype=float).clip(1e-10)

    # ── pyGAM PoissonGAM: s() for continuous, f() for Sex ───────
    # Column 0: Attained_Age_mod_spline (smooth)
    # Column 1: Issue_Age_mod_spline    (smooth)
    # Column 2: Sex                     (factor)
    _gam_spec = (
        s(0, n_splines=_PPR_N_SPLINES)
        + s(1, n_splines=_PPR_N_SPLINES)
        + gam_f(2)
    )

    _ppr_model = PoissonGAM(_gam_spec)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _ppr_model.gridsearch(
            _X_train,
            _y_train,
            exposure=_E_train,
            lam=_PPR_LAM_GRID,
            progress=False,
        )

    print(f"PPR / PoissonGAM fitted:")
    print(f"  AIC={_ppr_model.statistics_['AIC']:.2f}  "
          f"GCV={_ppr_model.statistics_['GCV']:.6f}  "
          f"n_train_cells={len(_ppr_train_cells)}")

    # ── predict wrapper ──────────────────────────────────────────
    class _PPRWrapper:
        """
        Wrapper with predict(df, offset) interface compatible with
        score_with_bundle's statsmodels_glm branch.
        """
        def __init__(self, gam, sex_map, sex_cats):
            self._gam      = gam
            self._sex_map  = sex_map
            self._sex_cats = sex_cats
            self.params    = np.zeros(gam.statistics_["edof"])

        def predict(self, df, offset=None):
            df_r   = df.reset_index(drop=True)
            aa     = df_r["Attained_Age_mod_spline"].to_numpy(dtype=float)
            ia     = df_r["Issue_Age_mod_spline"].to_numpy(dtype=float)
            sex    = df_r["Sex"].astype(str).map(self._sex_map).fillna(0).to_numpy(dtype=float)
            X      = np.column_stack([aa, ia, sex])
            exp_v  = df_r["Policies_Exposed"].to_numpy(dtype=float).clip(1e-10)
            # pyGAM predict returns expected count when exposure is supplied
            return self._gam.predict(X, exposure=exp_v)

    _ppr_wrapper = _PPRWrapper(_ppr_model, _sex_map, _ppr_sex_cats)

    # ── score on train / test ────────────────────────────────────
    _ppr_scored_train = score_with_bundle(
        {
            "model_key":       "poisson_ppr",
            "model_type":      "statsmodels_glm",
            "features":        _PPR_FEATURES,
            "result":          _ppr_wrapper,
            "category_levels": {},
            "design_columns":  [],
        },
        train_raw,
    )
    _ppr_scored_score = score_with_bundle(
        {
            "model_key":       "poisson_ppr",
            "model_type":      "statsmodels_glm",
            "features":        _PPR_FEATURES,
            "result":          _ppr_wrapper,
            "category_levels": {},
            "design_columns":  [],
        },
        score_raw,
    )

    bundle_ppr = {
        "model_key":             "poisson_ppr",
        "model_type":            "statsmodels_glm",
        "label":                 "poisson_ppr",
        "formula":               (
            f"PoissonGAM: s(Attained_Age) + s(Issue_Age_mod) + f(Sex)  "
            f"n_splines={_PPR_N_SPLINES}  "
            f"lam={[round(float(l), 4) for l in _ppr_model.lam]}"
        ),
        "features":              _PPR_FEATURES,
        "result":                _ppr_wrapper,
        "_sex_map":              _sex_map,
        "_sex_cats":             _ppr_sex_cats,
        "is_pygam_core":         False,
        "_is_ppr":               True,
        "category_levels":       {},
        "design_columns":        [],
        "baseline_ratio":        _ppr_base_ratio,
        "baseline_deviance":     _ppr_base_dev,
        "parameter_count":       int(round(_ppr_model.statistics_["edof"])),
        "likelihood_comparable": "Yes",
        "aic":                   float(_ppr_model.statistics_["AIC"]),
        "bic":                   np.nan,
        "null_deviance":         np.nan,
        "train_cells":           _ppr_train_cells,
        "metrics": pd.DataFrame([
            evaluate_predictions(
                _ppr_scored_train[ACTUAL_CNT_COL],
                _ppr_scored_train[EXPECTED_CNT_COL],
                _ppr_scored_train["Predicted_Deaths"],
                "Train",
            ),
            evaluate_predictions(
                _ppr_scored_score[ACTUAL_CNT_COL],
                _ppr_scored_score[EXPECTED_CNT_COL],
                _ppr_scored_score["Predicted_Deaths"],
                "Scored_Data",
            ),
        ]),
    }

    save_model_bundle(bundle_ppr)

except Exception as exc:
    print(f"Model failed: poisson_ppr -> {exc}")
    import traceback; traceback.print_exc()


In [ ]:
# SESSION 3.11 MODEL 5 ? GAM-STYLE ATTAINED-AGE CHALLENGER
# ============================================================

# Build the attained-age formula dynamically so it always reflects the
# current ENFORCE_CONTINUITY toggle and AA_BP1 / AA_BP2 breakpoints.
GAM_FORMULA_ATTAINED = build_attained_age_gam_formula(
    enforce_continuity=ENFORCE_CONTINUITY,
    knots=GAM_ATTAINED_AGE_KNOTS,
    attained_age_min=ATTAINED_AGE_MIN,
    attained_age_mod_max=ATTAINED_AGE_MOD_MAX,
    actual_cnt_col=ACTUAL_CNT_COL,
    duration_knots=GAM_DURATION_KNOTS,
    duration_upper_bound=GAM_DURATION_UPPER_BOUND,
)

print(
    f"[Model 5] ENFORCE_CONTINUITY={ENFORCE_CONTINUITY}  "
    f"breakpoints=({AA_BP1}, {AA_BP2})\n"
    f"Formula: {GAM_FORMULA_ATTAINED}"
)

try:
    bundle_gam_attained = fit_formula_glm_bundle(
        model_key="poisson_gam_attained",
        formula=GAM_FORMULA_ATTAINED,
        train_df=train_raw,
        score_df=score_raw,
    )
    save_model_bundle(bundle_gam_attained)
except Exception as exc:
    print(f"Model failed: poisson_gam_attained -> {exc}")


In [ ]:
# SESSION 3.12 MODEL 6 ? SKLEARN POISSON REGRESSOR
# ============================================================

try:
    bundle_poisson_reg = fit_poisson_regressor_bundle(
        model_key="poisson_regressor",
        train_df=train_raw,
        score_df=score_raw,
        features=ML_FEATURES,
        alpha=0.001,
    )
    save_model_bundle(bundle_poisson_reg)
except Exception as exc:
    print(f"Model failed: poisson_regressor -> {exc}")


In [ ]:
# SESSION 3.13 MODEL 7 ? RANDOM FOREST LOG-RATIO CHALLENGER
# ============================================================

try:
    bundle_random_forest = fit_log_ratio_bundle(
        model_key="random_forest_log_ratio",
        estimator=RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=5,
            random_state=MODEL_RANDOM_STATE,
            n_jobs=1,
        ),
        train_df=train_raw,
        score_df=score_raw,
        features=ML_FEATURES,
        smooth=MODEL_TARGET_SMOOTH,
    )
    save_model_bundle(bundle_random_forest)
except Exception as exc:
    print(f"Model failed: random_forest_log_ratio -> {exc}")


In [ ]:
# SESSION 3.14 MODEL 8 ? GRADIENT BOOSTING LOG-RATIO CHALLENGER
# ============================================================

try:
    bundle_gbm = fit_log_ratio_bundle(
        model_key="gradient_boosting_log_ratio",
        estimator=GradientBoostingRegressor(random_state=MODEL_RANDOM_STATE),
        train_df=train_raw,
        score_df=score_raw,
        features=ML_FEATURES,
        smooth=MODEL_TARGET_SMOOTH,
    )
    save_model_bundle(bundle_gbm)
except Exception as exc:
    print(f"Model failed: gradient_boosting_log_ratio -> {exc}")


In [ ]:
# SESSION 3.15 MODEL 9 ? HIST GRADIENT BOOSTING LOG-RATIO CHALLENGER
# ============================================================

try:
    bundle_hist_gbm = fit_log_ratio_bundle(
        model_key="hist_gradient_boosting_log_ratio",
        estimator=HistGradientBoostingRegressor(
            max_depth=6,
            learning_rate=0.05,
            max_iter=300,
            random_state=MODEL_RANDOM_STATE,
        ),
        train_df=train_raw,
        score_df=score_raw,
        features=ML_FEATURES,
        smooth=MODEL_TARGET_SMOOTH,
    )
    save_model_bundle(bundle_hist_gbm)
except Exception as exc:
    print(f"Model failed: hist_gradient_boosting_log_ratio -> {exc}")


In [ ]:
# SESSION 3.16 MODEL 10 ? EXTRATREES LOG-RATIO CHALLENGER
# ============================================================

try:
    bundle_extratrees = fit_log_ratio_bundle(
        model_key="extratrees_log_ratio",
        estimator=ExtraTreesRegressor(
            n_estimators=400,
            min_samples_leaf=5,
            random_state=MODEL_RANDOM_STATE,
            n_jobs=1,
        ),
        train_df=train_raw,
        score_df=score_raw,
        features=ML_FEATURES,
        smooth=MODEL_TARGET_SMOOTH,
    )
    save_model_bundle(bundle_extratrees)
except Exception as exc:
    print(f"Model failed: extratrees_log_ratio -> {exc}")


In [ ]:
# SESSION 3.17 MODEL CARDS
# ============================================================

model_cards = pd.DataFrame([
    {
        "Model_Key": "poisson_glm_factor_spline",
        "Role": "Benchmark",
        "Response_Basis": "Death count",
        "Offset_or_Weight_Usage": "Offset = log(Policies_Exposed)",
        "Interpretability": "High",
        "Comments": "Broad fixed-effect benchmark with raw plan, raw face band, issue-age spline, and duration spline.",
    },
    {
        "Model_Key": "poisson_gam_core_per_sex",
        "Role": "GAM-style challenger",
        "Response_Basis": "Death count",
        "Offset_or_Weight_Usage": "Offset = log(Policies_Exposed)",
        "Interpretability": "High",
        "Comments": "Per-sex attained-age smoothing. Kept separate from the combined-sex model so keys do not overwrite each other.",
    },
    {
        "Model_Key": "poisson_gam_core",
        "Role": "Primary GAM-style candidate",
        "Response_Basis": "Death count",
        "Offset_or_Weight_Usage": "Offset = log(Policies_Exposed)",
        "Interpretability": "High",
        "Comments": "Combined-sex attained-age smooth with sex interaction. This remains the default Session 4 selected key.",
    },
    {
        "Model_Key": "poisson_gam_core_26",
        "Role": "GAM-style challenger",
        "Response_Basis": "Death count",
        "Offset_or_Weight_Usage": "Offset = log(Policies_Exposed)",
        "Interpretability": "High",
        "Comments": "Model 2 plus Issue_Age_mod as a linear term.",
    },
    {
        "Model_Key": "poisson_gam_core_27",
        "Role": "GAM-style challenger",
        "Response_Basis": "Death count",
        "Offset_or_Weight_Usage": "Offset = log(Policies_Exposed)",
        "Interpretability": "High",
        "Comments": "Model 2.6 with Issue_Age_mod entering through a spline.",
    },
    {
        "Model_Key": "whittaker_henderson",
        "Role": "Actuarial graduation challenger",
        "Response_Basis": "Death count",
        "Offset_or_Weight_Usage": "Exposure enters the graduated crude rate calculation.",
        "Interpretability": "High",
        "Comments": "Classical WH graduation by sex and attained age.",
    },
    {
        "Model_Key": "poisson_ppr",
        "Role": "Nonlinear challenger",
        "Response_Basis": "Death count",
        "Offset_or_Weight_Usage": "pyGAM exposure argument",
        "Interpretability": "Medium",
        "Comments": "Projection-pursuit-style smooth model using attained age, issue age, and sex.",
    },
    {
        "Model_Key": "poisson_gam_attained",
        "Role": "GAM-style challenger",
        "Response_Basis": "Death count",
        "Offset_or_Weight_Usage": "Offset = log(Policies_Exposed)",
        "Interpretability": "High",
        "Comments": "Attained-age alternative with select/duration structure.",
    },
    {
        "Model_Key": "poisson_regressor",
        "Role": "Regularized challenger",
        "Response_Basis": "Death rate",
        "Offset_or_Weight_Usage": "Policies_Exposed used as sample weight",
        "Interpretability": "Medium",
        "Comments": "Sklearn PoissonRegressor on the selected feature set.",
    },
    {
        "Model_Key": "random_forest_log_ratio",
        "Role": "Signal finder",
        "Response_Basis": "log death rate",
        "Offset_or_Weight_Usage": "Policies_Exposed used as sample weight",
        "Interpretability": "Low",
        "Comments": "Useful nonlinear challenger, not usually the final actuarial form.",
    },
    {
        "Model_Key": "gradient_boosting_log_ratio",
        "Role": "Signal finder",
        "Response_Basis": "log death rate",
        "Offset_or_Weight_Usage": "Policies_Exposed used as sample weight",
        "Interpretability": "Low",
        "Comments": "Gradient boosting challenger for nonlinear pattern checks.",
    },
    {
        "Model_Key": "hist_gradient_boosting_log_ratio",
        "Role": "Signal finder",
        "Response_Basis": "log death rate",
        "Offset_or_Weight_Usage": "Policies_Exposed used as sample weight",
        "Interpretability": "Low",
        "Comments": "Histogram gradient boosting challenger.",
    },
    {
        "Model_Key": "extratrees_log_ratio",
        "Role": "Signal finder",
        "Response_Basis": "log death rate",
        "Offset_or_Weight_Usage": "Policies_Exposed used as sample weight",
        "Interpretability": "Low",
        "Comments": "ExtraTrees challenger and high-variance signal finder.",
    },
])

display(model_cards)
export_csv(model_cards, "session3_model_cards.csv")


In [ ]:
# SESSION 3.18 MODEL COMPARISON TABLE
# ============================================================

if len(MODEL_ROWS) == 0:
    model_comparison = pd.DataFrame()
    print("No model results are available yet. Run one or more model cells above.")
else:
    model_comparison = pd.DataFrame(MODEL_ROWS).sort_values(
        ["Scored_Deviance", "Scored_Weighted_AE_Error", "Train_Deviance_Explained_vs_Intercept"],
        ascending=[True, True, False],
    ).reset_index(drop=True)

    display(model_comparison)
    export_csv(model_comparison, "session3_model_comparison.csv")

print(
    "Actuarial note: AIC and BIC are only directly comparable across the likelihood-based GLM / GAM-style models. "
    "They are shown for tree challengers only as blank by design."
)


In [ ]:
# SESSION 3.19 CONCLUSION
if "model_comparison" not in globals() or len(model_comparison) == 0:
    print("Session 3 conclusion: no model comparison table is available yet. Run at least one model cell above.")
else:
    best_key = str(model_comparison.loc[0, "Model_Key"])
    session3_conclusion = (
        f"Session 3 conclusion: the current in-scope comparison ranks {best_key} first on the scoring table. "
        f"The GAM-style candidates should be reviewed first because they retain actuarial interpretability, explicit age-basis handling, "
        f"and smooth issue-age / duration structure with explicit knot choices. The grouped plan / face challenger is included to test "
        f"whether those variables add stable value rather than just sparse-cell noise."
    )
    print(session3_conclusion)


---

# Session 4 · Testing, monotonicity review, graph review, and sheet-style output


In [ ]:
# SESSION 4.1 VALIDATION CONTROL PANEL
# ============================================================

# ------------------------------
# Session 4: validation / review controls
# ------------------------------
SELECTED_MODEL_KEY = "poisson_gam_core"  # Change only this key to rerun Session 4 for a different Session 3 model
SEGMENT_TOLERANCE = 0.10
MIN_EXPECTED_FOR_REVIEW = 3.0
MIN_EXPECTED_FOR_MONOTONICITY = 3.0
MONOTONICITY_RATE_COL = "Predicted_Rate_per_1000"

# ------------------------------
# Sheet-style mortality output
# ------------------------------
OUTPUT_AGE_IND = "0"              # "0" = ANB, "1" = ALB
OUTPUT_SEX = "M"                  # "M" or "F"
OUTPUT_SELECT_PERIOD = 25
OUTPUT_ULT_DURATION = 26
OUTPUT_DURATION_COLUMNS = list(range(1, 26))
OUTPUT_RATE_PER = 1000
OUTPUT_ISSUE_AGE_MIN = 0
OUTPUT_ISSUE_AGE_MAX = 17

# Optional benchmark comparison
BENCHMARK_XLSX_PATH = None        # Example: Path("2015-vbt-unismoke-alb-anb.xlsx")
BENCHMARK_SHEET_NAME = None       # Example: "2015 Male Unismoke ANB"

EXPORT_DIR = Path("script/model_outputs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# SESSION 4.2 REFIT SELECTED MODEL ON FULL SCOPED DATA
# ============================================================

if SELECTED_MODEL_KEY not in MODEL_BUNDLES:
    raise KeyError(
        f"SELECTED_MODEL_KEY={SELECTED_MODEL_KEY!r} not in MODEL_BUNDLES."
    )

selected_session3_bundle = MODEL_BUNDLES[SELECTED_MODEL_KEY]

refit_df = scoped_df[
    pd.to_numeric(scoped_df[ATTAINED_AGE_MOD_COL], errors="coerce") <= ATTAINED_AGE_MOD_MAX
].copy()
n_dropped = len(scoped_df) - len(refit_df)
if n_dropped:
    print(f"Refit cap: dropped {n_dropped:,} rows "
          f"where {ATTAINED_AGE_MOD_COL} > {ATTAINED_AGE_MOD_MAX}")

# ── Model 2: combined-sex penalised interaction refit ────────
if selected_session3_bundle.get("_penalized_interaction"):
    from statsmodels.gam.api import BSplines

    _r_knots  = selected_session3_bundle["_knots"]
    _r_alpha  = selected_session3_bundle["_alpha"]
    _r_lower  = selected_session3_bundle.get("_lower_bound", float(ATTAINED_AGE_MIN))
    _r_upper  = selected_session3_bundle.get("_upper_bound", float(ATTAINED_AGE_MOD_MAX))

    _CORE_FEATURES = ["Attained_Age_mod_spline", "Sex"]
    _CORE_REQUIRED = _CORE_FEATURES + [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL,
        ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed",
    ]

    _r_mf    = prepare_model_frame(refit_df, _CORE_REQUIRED)
    _r_cells = aggregate_cells(_r_mf, _CORE_FEATURES)
    _r_ratio, _r_dev = make_intercept_baseline(_r_cells)
    _r_cells = _r_cells[_r_cells["Policies_Exposed"] > 0].reset_index(drop=True)

    print(f"Refitting penalised interaction GAM on full scope "
          f"(knots={_r_knots}  alpha={_r_alpha:.3g}  "
          f"bounds=[{_r_lower}, {_r_upper}]):")

    # ── PIRLS on full scope ───────────────────────────────────
    _y   = _r_cells[ACTUAL_CNT_COL].astype(float).values
    _off = np.log(_r_cells["Policies_Exposed"].astype(float).clip(1e-10).values)
    _age = _r_cells["Attained_Age_mod_spline"].astype(float).values
    _sex = _r_cells["Sex"].astype(str).values

    _is_M = (_sex == "M").astype(float)
    _is_F = 1.0 - _is_M
    _n    = len(_y)
    _r_bs = BSplines(
        _r_cells[["Attained_Age_mod_spline"]],
        df=[len(_r_knots) + 4], degree=[3], include_intercept=False,
        knot_kwds=[{"knots": list(_r_knots),
                    "lower_bound": _r_lower,
                    "upper_bound": _r_upper}],
    )
    _nb = _r_bs.basis.shape[1]   # ← actual width after include_intercept=False
    _B   = _r_bs.basis
    _B_F = _B * _is_F[:, None]
    _B_M = _B * _is_M[:, None]
    _X   = np.column_stack([np.ones(_n), _is_M, _B_F, _B_M])
    _p   = _X.shape[1]

    _S = _r_bs.penalty_matrices[0]
    _P = np.zeros((_p, _p))
    _P[2:2+_nb, 2:2+_nb]       = _r_alpha * _S
    _P[2+_nb:_p, 2+_nb:_p]     = _r_alpha * _S

    _beta = np.zeros(_p)
    for _ in range(50):
        _eta = _X @ _beta + _off
        _mu  = np.exp(np.clip(_eta, -30, 30))
        _XtWX = (_X * _mu[:, None]).T @ _X
        _XtWz = (_X * _mu[:, None]).T @ (
            _eta - _off + (_y - _mu) / _mu.clip(1e-15)
        )
        _beta_new = np.linalg.solve(_XtWX + _P, _XtWz)
        if np.max(np.abs(_beta_new - _beta)) < 1e-7:
            _beta = _beta_new
            break
        _beta = _beta_new

    _mu_f  = np.exp(np.clip(_X @ _beta + _off, -30, 30))
    _ll    = float(np.sum(_y * np.log(_mu_f.clip(1e-15)) - _mu_f - gammaln(_y + 1)))
    _XtWXf = (_X * _mu_f[:, None]).T @ _X
    _edf   = float(np.trace(np.linalg.solve(_XtWXf + _P, _XtWXf)))
    _aic_r = -2.0 * _ll + 2.0 * _edf

    print(f"  Full-scope refit: cells={len(_r_cells)}  "
          f"edf={_edf:.2f}  AIC={_aic_r:.2f}")

    # ── wrapper ───────────────────────────────────────────────
    class _RPenalisedWrapper:
        def __init__(self, beta, bs, nb):
            self.params = beta
            self._bs = bs
            self._nb = nb

        def predict(self, df, offset=None):
            df_r = df.reset_index(drop=True)
            n    = len(df_r)
            sex  = df_r["Sex"].astype(str).values
            age  = df_r["Attained_Age_mod_spline"].astype(float).values
            is_M = (sex == "M").astype(float)
            is_F = 1.0 - is_M
            B    = self._bs.transform(
                df_r[["Attained_Age_mod_spline"]].astype(float).values
            )
            B_F  = B * is_F[:, None]
            B_M  = B * is_M[:, None]
            X    = np.column_stack([np.ones(n), is_M, B_F, B_M])
            eta  = X @ self.params
            if offset is not None:
                eta += np.asarray(offset, dtype=float)
            return np.exp(eta)

    _r_wrapper = _RPenalisedWrapper(_beta, _r_bs, _nb)

    final_bundle = {
        **selected_session3_bundle,
        "model_key":         SELECTED_MODEL_KEY,
        "result":            _r_wrapper,
        "train_cells":       _r_cells,
        "baseline_ratio":    _r_ratio,
        "baseline_deviance": _r_dev,
        "parameter_count":   int(_p),
        "aic":               _aic_r,
        "bic":               np.nan,
    }

# ── Model 2.5: per-sex GLMGam refit ──────────────────────────
elif selected_session3_bundle.get("is_pygam_core"):
    from statsmodels.gam.api import GLMGam, BSplines

    _r_lower    = selected_session3_bundle.get("_lower_bound", float(ATTAINED_AGE_MIN))
    _r_upper    = selected_session3_bundle.get("_upper_bound", float(ATTAINED_AGE_MOD_MAX))
    _r_sex_meta = selected_session3_bundle.get("_sex_models_meta", {})
    _r_sex_cats = selected_session3_bundle.get("_sex_cats",
                    sorted(refit_df["Sex"].astype(str).dropna().unique()))

    _CORE_FEATURES = ["Attained_Age_mod_spline", "Sex"]
    _CORE_REQUIRED = _CORE_FEATURES + [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL,
        ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed",
    ]

    _r_mf    = prepare_model_frame(refit_df, _CORE_REQUIRED)
    _r_cells = aggregate_cells(_r_mf, _CORE_FEATURES)
    _r_ratio, _r_dev = make_intercept_baseline(_r_cells)
    _r_cells = _r_cells[_r_cells["Policies_Exposed"] > 0].reset_index(drop=True)

    _r_sex_models = {}
    print(f"Refitting per-sex GLMGam on full scope "
          f"(bounds=[{_r_lower}, {_r_upper}]):")

    for _sv in _r_sex_cats:
        _meta    = _r_sex_meta.get(_sv, {})
        _r_knots = _meta.get("knots",
                    selected_session3_bundle.get("_knots",
                    list(GAM_ATTAINED_AGE_KNOTS)))
        _r_alpha = _meta.get("alpha", 1.0)
        _r_bs_df = len(_r_knots) + 4

        _cells_sv = _r_cells[
            _r_cells["Sex"].astype(str) == _sv
        ].reset_index(drop=True)
        if len(_cells_sv) < 5:
            print(f"  Sex={_sv}: too few cells — skipped")
            continue

        _y_sv   = _cells_sv[ACTUAL_CNT_COL].astype(float)
        _off_sv = np.log(_cells_sv["Policies_Exposed"].astype(float).clip(1e-10))
        _Xp_sv  = pd.DataFrame({"const": np.ones(len(_cells_sv))})
        _bs_sv  = BSplines(
            _cells_sv[["Attained_Age_mod_spline"]],
            df=[_r_bs_df], degree=[3], include_intercept=False,
            knot_kwds=[{"knots": _r_knots,
                        "lower_bound": _r_lower,
                        "upper_bound": _r_upper}],
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _r_result = GLMGam(
                _y_sv, exog=_Xp_sv, smoother=_bs_sv,
                alpha=[_r_alpha], family=sm.families.Poisson(),
                offset=_off_sv,
            ).fit(disp=False)

        _r_sex_models[_sv] = {
            "params":  np.asarray(_r_result.params, dtype=float),
            "smoother": _bs_sv,
            "alpha":   _r_alpha,
            "aic":     float(_r_result.aic),
            "knots":   _r_knots,
            "n_knots": len(_r_knots),
        }
        print(f"  Sex={_sv}: knots={_r_knots}  "
              f"alpha={_r_alpha:.3g}  AIC={_r_result.aic:.2f}  "
              f"cells={len(_cells_sv)}")
        print(_r_result.summary())

    class _RWrapper:
        def __init__(self, sex_models, sex_cats):
            self._sex_models = sex_models
            self._sex_cats   = sex_cats
            self.params = np.concatenate(
                [m["params"] for m in sex_models.values()]
            )
        def predict(self, df, offset=None):
            df_r = df.reset_index(drop=True)
            off  = np.asarray(offset, dtype=float)
            pred = np.zeros(len(df_r))
            for sv, mi in self._sex_models.items():
                mask  = (df_r["Sex"].astype(str) == sv).values
                if not mask.any():
                    continue
                age_m  = df_r.loc[mask, ["Attained_Age_mod_spline"]
                                  ].to_numpy(dtype=float)
                X_spl  = mi["smoother"].transform(age_m)
                X_full = np.column_stack([np.ones((mask.sum(), 1)), X_spl])
                pred[mask] = np.exp(X_full @ mi["params"] + off[mask])
            return pred

    _r_wrapper = _RWrapper(_r_sex_models, _r_sex_cats)

    final_bundle = {
        **selected_session3_bundle,
        "model_key":         SELECTED_MODEL_KEY,
        "result":            _r_wrapper,
        "train_cells":       _r_cells,
        "baseline_ratio":    _r_ratio,
        "baseline_deviance": _r_dev,
        "parameter_count":   int(len(_r_wrapper.params)),
        "aic":  float(sum(mi["aic"] for mi in _r_sex_models.values())),
        "bic":  np.nan,
    }

# ── Model 3 WH: re-graduate on full scope ────────────────────
elif selected_session3_bundle.get("_wh_tables") is not None:
    _r_wh_lambda  = selected_session3_bundle["_wh_lambda"]
    _r_wh_d       = selected_session3_bundle["_wh_d"]
    _r_wh_sex_cats = selected_session3_bundle["_wh_sex_cats"]

    _CORE_FEATURES = ["Attained_Age_mod_spline", "Sex"]
    _CORE_REQUIRED = _CORE_FEATURES + [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL,
        ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed",
    ]

    _r_mf    = prepare_model_frame(refit_df, _CORE_REQUIRED)
    _r_cells = aggregate_cells(_r_mf, _CORE_FEATURES)
    _r_ratio, _r_dev = make_intercept_baseline(_r_cells)
    _r_cells = _r_cells[_r_cells["Policies_Exposed"] > 0].reset_index(drop=True)

    def _diff_matrix_r(n: int, d: int) -> np.ndarray:
        D = np.eye(n)
        for _ in range(d):
            D = np.diff(D, axis=0)
        return D

    _r_wh_tables = {}
    print(f"Re-graduating WH on full scope (lambda={_r_wh_lambda}  d={_r_wh_d}):")
    for _sv in _r_wh_sex_cats:
        _cells_sv = (
            _r_cells[_r_cells["Sex"].astype(str) == _sv]
            .copy().sort_values("Attained_Age_mod_spline").reset_index(drop=True)
        )
        _ages   = _cells_sv["Attained_Age_mod_spline"].to_numpy(dtype=float)
        _deaths = _cells_sv[ACTUAL_CNT_COL].to_numpy(dtype=float)
        _exp    = _cells_sv["Policies_Exposed"].to_numpy(dtype=float).clip(1e-10)
        _crude  = _deaths / _exp
        n = len(_ages)
        _W  = np.diag(_exp)
        _Dd = _diff_matrix_r(n, _r_wh_d)
        _gamma = np.linalg.solve(_W + _r_wh_lambda * (_Dd.T @ _Dd), _W @ _crude)
        _gamma = np.clip(_gamma, 1e-10, None)
        _r_wh_tables[_sv] = (_ages, _gamma)
        print(f"  Sex={_sv}: n_nodes={n}  grad_min={_gamma.min():.6f}  grad_max={_gamma.max():.6f}")

    class _RWHWrapper:
        def __init__(self, tables, sex_cats):
            self._tables   = tables
            self._sex_cats = sex_cats
            self.params    = np.zeros(sum(len(g) for _, g in tables.values()))
        def predict(self, df, offset=None):
            df_r  = df.reset_index(drop=True)
            n     = len(df_r)
            pred  = np.zeros(n)
            age_v = df_r["Attained_Age_mod_spline"].to_numpy(dtype=float)
            sex_v = df_r["Sex"].astype(str).values
            exp_v = df_r["Policies_Exposed"].to_numpy(dtype=float)
            for sv, (ages, gamma) in self._tables.items():
                mask = (sex_v == sv)
                if not mask.any():
                    continue
                pred[mask] = np.interp(age_v[mask], ages, gamma) * exp_v[mask]
            return pred

    _r_wh_wrapper = _RWHWrapper(_r_wh_tables, _r_wh_sex_cats)

    final_bundle = {
        **selected_session3_bundle,
        "model_key":         SELECTED_MODEL_KEY,
        "result":            _r_wh_wrapper,
        "train_cells":       _r_cells,
        "baseline_ratio":    _r_ratio,
        "baseline_deviance": _r_dev,
        "parameter_count":   int(sum(len(g) for _, g in _r_wh_tables.values())),
        "_wh_tables":        _r_wh_tables,
        "aic":               np.nan,
        "bic":               np.nan,
    }

# ── Model 4 PPR / pyGAM: refit on full scope ─────────────────
elif selected_session3_bundle.get("_is_ppr"):
    from pygam import PoissonGAM, s, f as gam_f

    _r_ppr_sex_cats = selected_session3_bundle["_sex_cats"]
    _r_sex_map      = selected_session3_bundle["_sex_map"]
    _r_ppr_lam      = selected_session3_bundle["result"]._gam.lam

    _PPR_FEATURES = ["Attained_Age_mod_spline", "Issue_Age_mod_spline", "Sex"]
    _PPR_REQUIRED = _PPR_FEATURES + [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL,
        ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed",
    ]

    _r_ppr_mf    = prepare_model_frame(refit_df, _PPR_REQUIRED)
    _r_ppr_cells = aggregate_cells(_r_ppr_mf, _PPR_FEATURES)
    _r_ppr_ratio, _r_ppr_dev = make_intercept_baseline(_r_ppr_cells)
    _r_ppr_cells = _r_ppr_cells[
        _r_ppr_cells["Policies_Exposed"] > 0
    ].reset_index(drop=True)

    def _encode_ppr_r(cells):
        aa  = cells["Attained_Age_mod_spline"].to_numpy(dtype=float)
        ia  = cells["Issue_Age_mod_spline"].to_numpy(dtype=float)
        sex = cells["Sex"].astype(str).map(_r_sex_map).fillna(0).to_numpy(dtype=float)
        return np.column_stack([aa, ia, sex])

    _X_r   = _encode_ppr_r(_r_ppr_cells)
    _y_r   = _r_ppr_cells[ACTUAL_CNT_COL].to_numpy(dtype=float)
    _E_r   = _r_ppr_cells["Policies_Exposed"].to_numpy(dtype=float).clip(1e-10)

    _r_ppr_model = PoissonGAM(
        s(0, n_splines=20) + s(1, n_splines=20) + gam_f(2),
        lam=_r_ppr_lam,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _r_ppr_model.fit(_X_r, _y_r, exposure=_E_r)

    print(f"PPR full-scope refit: AIC={_r_ppr_model.statistics_['AIC']:.2f}  "
          f"cells={len(_r_ppr_cells)}")

    class _RPPRWrapper:
        def __init__(self, gam, sex_map, sex_cats):
            self._gam      = gam
            self._sex_map  = sex_map
            self._sex_cats = sex_cats
            self.params    = np.zeros(int(round(gam.statistics_["edof"])))
        def predict(self, df, offset=None):
            df_r   = df.reset_index(drop=True)
            aa     = df_r["Attained_Age_mod_spline"].to_numpy(dtype=float)
            ia     = df_r["Issue_Age_mod_spline"].to_numpy(dtype=float)
            sex    = df_r["Sex"].astype(str).map(self._sex_map).fillna(0).to_numpy(dtype=float)
            X      = np.column_stack([aa, ia, sex])
            exp_v  = df_r["Policies_Exposed"].to_numpy(dtype=float).clip(1e-10)
            return self._gam.predict(X, exposure=exp_v)

    _r_ppr_wrapper = _RPPRWrapper(_r_ppr_model, _r_sex_map, _r_ppr_sex_cats)

    final_bundle = {
        **selected_session3_bundle,
        "model_key":         SELECTED_MODEL_KEY,
        "result":            _r_ppr_wrapper,
        "train_cells":       _r_ppr_cells,
        "baseline_ratio":    _r_ppr_ratio,
        "baseline_deviance": _r_ppr_dev,
        "parameter_count":   int(round(_r_ppr_model.statistics_["edof"])),
        "aic":               float(_r_ppr_model.statistics_["AIC"]),
        "bic":               np.nan,
    }

# ── other formula GLMs (Models 1, 2.6, 2.7, 5, …) ───────────
elif selected_session3_bundle["model_type"] == "statsmodels_glm":
    _refit_formula = selected_session3_bundle["formula"]
    print(f"Refitting statsmodels GLM on full scope:")
    print(f"  Formula: {_refit_formula}")
    final_bundle = fit_formula_glm_bundle(
        model_key=SELECTED_MODEL_KEY,
        formula=_refit_formula,
        train_df=refit_df, score_df=refit_df,
    )
    print(final_bundle["result"].summary())

elif selected_session3_bundle["model_type"] == "poisson_regressor":
    final_bundle = fit_poisson_regressor_bundle(
        model_key=SELECTED_MODEL_KEY,
        train_df=refit_df, score_df=refit_df,
        features=selected_session3_bundle["features"],
        alpha=selected_session3_bundle["result"].alpha,
    )
elif selected_session3_bundle["model_type"] == "log_ratio_model":
    final_estimator = selected_session3_bundle["result"].__class__(
        **selected_session3_bundle["result"].get_params()
    )
    final_bundle = fit_log_ratio_bundle(
        model_key=SELECTED_MODEL_KEY,
        estimator=final_estimator,
        train_df=refit_df, score_df=refit_df,
        features=selected_session3_bundle["features"],
        smooth=selected_session3_bundle["smooth"],
    )
else:
    raise ValueError(
        f"Unsupported model type: {selected_session3_bundle['model_type']}"
    )

# ── score: train, holdout, full scope ────────────────────────
_sc_train   = score_with_bundle(final_bundle, train_raw)
_sc_holdout = score_with_bundle(final_bundle, score_raw)

scored_df = score_with_bundle(final_bundle, refit_df)
scored_df = add_validation_bands(add_grouped_features(scored_df))

def _val_row(label, df_s):
    act = df_s[ACTUAL_CNT_COL].sum()
    exp = df_s[EXPECTED_CNT_COL].sum()
    prd = df_s["Predicted_Deaths"].sum()
    return {
        "Split":               label,
        "Rows":                len(df_s),
        "Actual_Deaths":       act,
        "Expected_VBT":        exp,
        "Predicted_Deaths":    prd,
        "Actual_to_Predicted": safe_divide(act, prd),
        "Actual_to_VBT":       safe_divide(act, exp),
        "Amount_AE_to_VBT":    safe_divide(df_s[ACTUAL_AMT_COL].sum(),
                                           df_s[EXPECTED_AMT_COL].sum()),
    }

overall_validation = pd.DataFrame([
    _val_row(f"Train ({TRAIN_FRACTION:.0%})", _sc_train),
    _val_row(f"Holdout ({1 - TRAIN_FRACTION:.0%})", _sc_holdout),
    _val_row("Full scope (refit)",   scored_df),
])
display(overall_validation)
export_csv(overall_validation, "session4_overall_validation_full_refit.csv")
if HAS_PYARROW:
    _export_df = scored_df.copy()
    for _col in _export_df.select_dtypes(include="category").columns:
        _export_df[_col] = _export_df[_col].astype(str)
    _export_df.to_parquet(EXPORT_DIR / "session4_scored_full_refit.parquet", index=False)
    print(f"Saved: {EXPORT_DIR / 'session4_scored_full_refit.parquet'}")
_gap = abs(
    safe_divide(_sc_holdout[ACTUAL_CNT_COL].sum(),
                _sc_holdout["Predicted_Deaths"].sum()) -
    safe_divide(_sc_train[ACTUAL_CNT_COL].sum(),
                _sc_train["Predicted_Deaths"].sum())
)
print(f"\nOverfitting check: train A/P vs holdout A/P gap = {_gap:.4f} "
      f"({'OK < 0.05' if _gap < 0.05 else 'REVIEW >= 0.05'})")


In [ ]:
# SESSION 4.3 HOLDOUT AND FINAL-REFIT NOTE
# ============================================================

validation_basis_note = pd.DataFrame({
    "Item": [
        "Validation basis",
        "Holdout logic",
        "Current final refit",
        "Interpretation note",
    ],
    "Value": [
        f"Train/holdout split is {TRAIN_FRACTION:.0%}/{1 - TRAIN_FRACTION:.0%}",
        "Split is selected on exposure and death-count balance, then checked by key segments",
        "The selected model is refit on full scoped data after Session 3 comparison",
        "Use train/holdout metrics for overfitting; use full refit for final calibration, monotonicity, and sheet output",
    ],
})

display(validation_basis_note)
export_csv(validation_basis_note, "session4_validation_basis_note.csv")


In [ ]:
# SESSION 4.4 CROSS-VALIDATION ? K-FOLD POISSON DEVIANCE
# ============================================================

CV_K_FOLDS = 5
CV_RANDOM_STATE = 42

if selected_session3_bundle["model_type"] == "statsmodels_glm" and "formula" in selected_session3_bundle:
    formula  = selected_session3_bundle["formula"]
    features = selected_session3_bundle["features"]
    required_cols = features + [ACTUAL_CNT_COL, EXPECTED_CNT_COL, "Policies_Exposed"]

    model_df = prepare_model_frame(scoped_df, required_cols)
    cells    = aggregate_cells(model_df, features).reset_index(drop=True)

    kf = KFold(n_splits=CV_K_FOLDS, shuffle=True, random_state=CV_RANDOM_STATE)

    cv_rows = []
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(cells), start=1):
        train_cells = cells.iloc[train_idx].copy()
        val_cells   = cells.iloc[val_idx].copy()
        try:
            res = smf.glm(
                formula=formula,
                data=train_cells,
                family=sm.families.Poisson(),
                offset=np.log(train_cells["Policies_Exposed"]),
            ).fit(disp=False)

            pred_train = res.predict(train_cells, offset=np.log(train_cells["Policies_Exposed"]))
            pred_val   = res.predict(val_cells,   offset=np.log(val_cells["Policies_Exposed"]))

            cv_rows.append({
                "Fold":                    fold_idx,
                "Train_Cells":             len(train_cells),
                "Val_Cells":               len(val_cells),
                "Train_Deviance":          safe_poisson_deviance(train_cells[ACTUAL_CNT_COL], pred_train),
                "Val_Deviance":            safe_poisson_deviance(val_cells[ACTUAL_CNT_COL],   pred_val),
                "Deviance_Gap (Val-Train)": safe_poisson_deviance(val_cells[ACTUAL_CNT_COL], pred_val)
                                           - safe_poisson_deviance(train_cells[ACTUAL_CNT_COL], pred_train),
                "Train_A/P": safe_divide(train_cells[ACTUAL_CNT_COL].sum(), pred_train.sum()),
                "Val_A/P":   safe_divide(val_cells[ACTUAL_CNT_COL].sum(),   pred_val.sum()),
            })
        except Exception as exc:
            cv_rows.append({"Fold": fold_idx, "Note": str(exc)[:200]})

    cv_results = pd.DataFrame(cv_rows)

    # Append mean summary row
    numeric_cv_cols = ["Train_Deviance", "Val_Deviance", "Deviance_Gap (Val-Train)", "Train_A/P", "Val_A/P"]
    summary_row = {c: cv_results[c].mean() for c in numeric_cv_cols if c in cv_results.columns}
    summary_row.update({"Fold": "Mean", "Train_Cells": "", "Val_Cells": ""})
    cv_results = pd.concat([cv_results, pd.DataFrame([summary_row])], ignore_index=True)

    print(f"\n=== {CV_K_FOLDS}-Fold Cross-Validation · Model: {SELECTED_MODEL_KEY} ===")
    print("CV runs on aggregated cells (the model's natural training unit).")
    print("A small Val-Train deviance gap indicates no material overfitting.\n")
    display(cv_results)
    export_csv(cv_results, "session4_cross_validation.csv")

else:
    print(f"CV cell is implemented for formula-based statsmodels GLM bundles only. The Session 4 refit and calibration tables still work for this selected model.")
    print(f"Selected model type: {selected_session3_bundle['model_type']}")


In [ ]:
# SESSION 4.5 INFORMATION CRITERIA AND HOLDOUT DIAGNOSTICS
# ============================================================
# AIC/BIC are only directly comparable for likelihood-based GLM-style models.
# Holdout deviance and A/P are available for every bundle that reached Session 3.

ic_rows = []
for key, bundle in MODEL_BUNDLES.items():
    metrics = bundle["metrics"].set_index("Dataset")
    train_dev = float(metrics.loc["Train", "Poisson_Deviance"]) if "Train" in metrics.index else np.nan
    holdout_dev = float(metrics.loc["Scored_Data", "Poisson_Deviance"]) if "Scored_Data" in metrics.index else np.nan
    train_ap = float(metrics.loc["Train", "Actual_to_Predicted"]) if "Train" in metrics.index else np.nan
    holdout_ap = float(metrics.loc["Scored_Data", "Actual_to_Predicted"]) if "Scored_Data" in metrics.index else np.nan

    null_dev = bundle.get("null_deviance", np.nan)
    k = bundle.get("parameter_count", np.nan)
    comparable = bundle.get("likelihood_comparable", "No")

    train_pseudo_r2 = safe_divide(null_dev - train_dev, null_dev) if np.isfinite(null_dev) else np.nan
    holdout_pseudo_r2 = safe_divide(null_dev - holdout_dev, null_dev) if np.isfinite(null_dev) else np.nan

    ic_rows.append({
        "Model_Key": key,
        "Model_Type": bundle.get("model_type", "Unknown"),
        "Likelihood_Comparable": comparable,
        "Parameter_Count": k,
        "AIC": bundle.get("aic", np.nan) if comparable == "Yes" else np.nan,
        "BIC": bundle.get("bic", np.nan) if comparable == "Yes" else np.nan,
        "Null_Deviance": null_dev,
        "Train_Deviance": train_dev,
        "Holdout_Deviance": holdout_dev,
        "Train_Pseudo_R2": train_pseudo_r2,
        "Holdout_Pseudo_R2": holdout_pseudo_r2,
        "Generalisation_Gap": train_pseudo_r2 - holdout_pseudo_r2 if np.isfinite(train_pseudo_r2) and np.isfinite(holdout_pseudo_r2) else np.nan,
        "Train_A/P": train_ap,
        "Holdout_A/P": holdout_ap,
        "A/P_Gap": abs(train_ap - holdout_ap) if np.isfinite(train_ap) and np.isfinite(holdout_ap) else np.nan,
    })

ic_df = (
    pd.DataFrame(ic_rows)
    .sort_values(["AIC", "Holdout_Deviance"], ascending=[True, True], na_position="last")
    .reset_index(drop=True)
)

ic_df["Overfit_Flag"] = np.select(
    [
        pd.to_numeric(ic_df["A/P_Gap"], errors="coerce") > 0.05,
        pd.to_numeric(ic_df["Generalisation_Gap"], errors="coerce") > 0.05,
    ],
    ["Review A/P gap", "Review deviance gap"],
    default="OK",
)

print("AIC/BIC: compare only rows where Likelihood_Comparable == 'Yes'.")
print("Holdout_Deviance and Holdout_A/P are the cross-model overfitting checks.")
display(ic_df)
export_csv(ic_df, "session4_information_criteria.csv")


In [ ]:
# SESSION 4.6 BOOTSTRAP VALIDATION ? DEVIANCE AND A/P STABILITY
# ============================================================

N_BOOTSTRAP       = 200
BOOTSTRAP_SEED    = 42
rng               = np.random.default_rng(BOOTSTRAP_SEED)

if selected_session3_bundle["model_type"] == "statsmodels_glm" and "formula" in selected_session3_bundle:
    formula  = selected_session3_bundle["formula"]
    features = selected_session3_bundle["features"]
    required_cols = features + [ACTUAL_CNT_COL, EXPECTED_CNT_COL, "Policies_Exposed"]

    model_df = prepare_model_frame(scoped_df, required_cols)
    cells    = aggregate_cells(model_df, features).reset_index(drop=True)
    n_cells  = len(cells)

    boot_in_dev, boot_oob_dev = [], []
    boot_in_ap,  boot_oob_ap  = [], []

    for _ in range(N_BOOTSTRAP):
        idx      = rng.integers(0, n_cells, size=n_cells)
        oob_mask = np.ones(n_cells, dtype=bool)
        oob_mask[np.unique(idx)] = False

        in_bag = cells.iloc[idx].copy()
        oob    = cells[oob_mask].copy()
        if len(oob) == 0:
            continue

        try:
            res = smf.glm(
                formula=formula,
                data=in_bag,
                family=sm.families.Poisson(),
                offset=np.log(in_bag["Policies_Exposed"]),
            ).fit(disp=False)

            pred_in  = res.predict(in_bag, offset=np.log(in_bag["Policies_Exposed"]))
            pred_oob = res.predict(oob,    offset=np.log(oob["Policies_Exposed"]))

            boot_in_dev.append(safe_poisson_deviance(in_bag[ACTUAL_CNT_COL], pred_in))
            boot_oob_dev.append(safe_poisson_deviance(oob[ACTUAL_CNT_COL],   pred_oob))
            boot_in_ap.append(safe_divide(in_bag[ACTUAL_CNT_COL].sum(), pred_in.sum()))
            boot_oob_ap.append(safe_divide(oob[ACTUAL_CNT_COL].sum(),   pred_oob.sum()))
        except Exception:
            continue

    gap = np.array(boot_oob_dev) - np.array(boot_in_dev)

    boot_summary = pd.DataFrame({
        "Metric": [
            "In-Bag Deviance — Mean",     "In-Bag Deviance — Std",
            "OOB Deviance — Mean",         "OOB Deviance — Std",
            "OOB minus In-Bag Gap — Mean", "OOB minus In-Bag Gap — Std",
            "In-Bag A/P — Mean",           "In-Bag A/P — Std",
            "OOB A/P — Mean",              "OOB A/P — Std",
            "Successful Resamples",
        ],
        "Value": [
            np.mean(boot_in_dev),  np.std(boot_in_dev),
            np.mean(boot_oob_dev), np.std(boot_oob_dev),
            np.mean(gap),          np.std(gap),
            np.mean(boot_in_ap),   np.std(boot_in_ap),
            np.mean(boot_oob_ap),  np.std(boot_oob_ap),
            len(boot_in_dev),
        ],
    })

    print(f"\n=== Bootstrap Validation · Model: {SELECTED_MODEL_KEY} · {N_BOOTSTRAP} resamples ===")
    print("A small OOB–In-Bag deviance gap signals low overfitting risk.")
    print("Low OOB A/P std signals a stable, consistent model.\n")
    display(boot_summary)
    export_csv(boot_summary, "session4_bootstrap_validation.csv")

    # Distribution plots
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(boot_in_dev,  bins=30, alpha=0.7, label="In-Bag")
    axes[0].hist(boot_oob_dev, bins=30, alpha=0.7, label="OOB")
    axes[0].set_title("Bootstrap Poisson Deviance")
    axes[0].set_xlabel("Deviance")
    axes[0].legend()

    axes[1].hist(boot_in_ap,  bins=30, alpha=0.7, label="In-Bag A/P")
    axes[1].hist(boot_oob_ap, bins=30, alpha=0.7, label="OOB A/P")
    axes[1].axvline(1.0, linestyle="--", color="k", linewidth=1)
    axes[1].set_title("Bootstrap A/P Ratio")
    axes[1].set_xlabel("Actual / Predicted")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

else:
    print(f"Bootstrap cell is implemented for formula-based statsmodels GLM bundles only. The Session 4 refit and calibration tables still work for this selected model.")
    print(f"Selected model type: {selected_session3_bundle['model_type']}")


In [ ]:
# SESSION 4.7 CORE CALIBRATION TABLES
# ============================================================

diagnostic_df = scored_df.copy()
diagnostic_label = "full_refit"

validation_tables = {
    "by_year": calibration_table(diagnostic_df, "Observation_Year"),
    "by_age_ind": calibration_table(diagnostic_df, "Age_Ind"),
    "by_sex": calibration_table(diagnostic_df, "Sex"),
    "by_plan": calibration_table(diagnostic_df, "Insurance_Plan"),
    "by_plan_group": calibration_table(diagnostic_df, "Insurance_Plan_Group"),
    "by_face_band": calibration_table(diagnostic_df, "Face_Amount_Band"),
    "by_face_group": calibration_table(diagnostic_df, "Face_Band_Group"),
    "by_issue_age_band": calibration_table(diagnostic_df, "Issue_Age_Band"),
    "by_attained_age_band": calibration_table(diagnostic_df, "Attained_Age_Band"),
    "by_duration_band": calibration_table(diagnostic_df, "Duration_Band"),
    "male_by_attained_age_band":  calibration_table(
        diagnostic_df[diagnostic_df["Sex"].astype("string") == "M"].copy(),
        "Attained_Age_Band",
    ),
    "female_by_attained_age_band": calibration_table(
        diagnostic_df[diagnostic_df["Sex"].astype("string") == "F"].copy(),
        "Attained_Age_Band",
    ),
}

for name, table in validation_tables.items():
    print(f"\n=== {name} ({diagnostic_label}) ===")
    display(table.drop(columns=["Flag_Order"]))
    export_csv(table.drop(columns=["Flag_Order"]), f"session4_{diagnostic_label}_{name}.csv")


In [ ]:
# SESSION 4.8 WORST SEGMENTS TO REVIEW
# ============================================================

worst_segments = pd.concat(
    [
        validation_tables["by_age_ind"].assign(Segment_Type="Age_Ind"),
        validation_tables["by_sex"].assign(Segment_Type="Sex"),
        validation_tables["by_plan"].assign(Segment_Type="Insurance_Plan"),
        validation_tables["by_plan_group"].assign(Segment_Type="Insurance_Plan_Group"),
        validation_tables["by_face_band"].assign(Segment_Type="Face_Amount_Band"),
        validation_tables["by_face_group"].assign(Segment_Type="Face_Band_Group"),
        validation_tables["by_issue_age_band"].assign(Segment_Type="Issue_Age_Band"),
        validation_tables["by_attained_age_band"].assign(Segment_Type="Attained_Age_Band"),
        validation_tables["by_duration_band"].assign(Segment_Type="Duration_Band"),
    ],
    ignore_index=True,
).sort_values(["Flag_Order", "Gap_from_1_on_A/P"], ascending=[False, False]).reset_index(drop=True)

display(worst_segments.drop(columns=["Flag_Order"]).head(30))
export_csv(worst_segments.drop(columns=["Flag_Order"]), "session4_worst_segments.csv")


In [ ]:
# SESSION 4.9 MONOTONICITY REVIEW ? ISSUE AGE
# ============================================================

issue_age_monotonicity = monotonicity_review(
    diagnostic_df,
    x_col=ISSUE_AGE_MOD_COL,
    group_cols=["Age_Ind", "Sex"],
    rate_col=MONOTONICITY_RATE_COL,
    min_expected=MIN_EXPECTED_FOR_MONOTONICITY,
)

display(issue_age_monotonicity.head(50))
export_csv(issue_age_monotonicity, "session4_monotonicity_issue_age.csv")


In [ ]:
# SESSION 4.10 MONOTONICITY REVIEW ? DURATION WITHIN ISSUE-AGE BANDS
# ============================================================

duration_monotonicity = monotonicity_review(
    diagnostic_df,
    x_col="Duration",
    group_cols=["Age_Ind", "Sex", "Issue_Age_Band"],
    rate_col=MONOTONICITY_RATE_COL,
    min_expected=MIN_EXPECTED_FOR_MONOTONICITY,
)

display(duration_monotonicity.head(80))
export_csv(duration_monotonicity, "session4_monotonicity_duration.csv")


In [ ]:
# SESSION 4.11 GRAPH REVIEW ? PREDICTED RATE BY ISSUE AGE
# ============================================================

plot_rate_curves(
    diagnostic_df,
    x_col=ISSUE_AGE_MOD_COL,
    split_col="Sex",
    rate_col=["Actual_Rate_per_1000", "Predicted_Rate_per_1000"],
    min_expected=MIN_EXPECTED_FOR_MONOTONICITY,
    title="Actual vs predicted mortality rate by Issue_Age_mod split by Sex",
)


In [ ]:
# SESSION 4.12 GRAPH REVIEW ? ACTUAL VS PREDICTED BY ATTAINED AGE
# ============================================================

plot_rate_curves(
    diagnostic_df,
    x_col=ATTAINED_AGE_MOD_COL,
    split_col="Sex",
    rate_col=["Actual_Rate_per_1000", "Predicted_Rate_per_1000"],
    min_expected=MIN_EXPECTED_FOR_MONOTONICITY,
    title="Predicted mortality rate by Attained_Age_mod split by Sex",
)


In [ ]:
# SESSION 4.13 GRAPH REVIEW ? A/P BY ISSUE AGE
# ============================================================

plot_ratio_curves(
    diagnostic_df,
    x_col=ISSUE_AGE_MOD_COL,
    split_col="Sex",
    numerator_col="Actual_Rate_per_1000",
    denominator_col="Predicted_Rate_per_1000",
    min_expected=MIN_EXPECTED_FOR_MONOTONICITY,
    title="A/P by Issue_Age split by Sex",
    y_label="A/P",
)


In [ ]:
# SESSION 4.14 GRAPH REVIEW ? P/E BY ATTAINED AGE
# ============================================================

plot_ratio_curves(
    diagnostic_df,
    x_col=ATTAINED_AGE_MOD_COL,
    split_col="Sex",
    numerator_col="Predicted_Rate_per_1000",
    denominator_col="Expected_Rate_per_1000",
    min_expected=MIN_EXPECTED_FOR_MONOTONICITY,
    title="P/E by Attained_Age_mod split by Sex",
    y_label="P/E",
)


In [ ]:
# SESSION 4.15 GRAPH REVIEW ? PREDICTED RATE BY OBSERVATION YEAR
# ============================================================

plot_rate_curves(
    diagnostic_df,
    x_col="Observation_Year",
    split_col="Sex",
    rate_col="Predicted_Rate_per_1000",
    min_expected=MIN_EXPECTED_FOR_MONOTONICITY,
    title="Predicted mortality rate by Observation_Year split by Sex",
)


In [ ]:
# SESSION 4.16 RESIDUAL HEATMAPS
# ============================================================

residual_heatmap(diagnostic_df, ISSUE_AGE_MOD_COL, ATTAINED_AGE_MOD_COL, value_col="Actual_to_Predicted")


In [ ]:
# SESSION 4.17 FINAL TESTING SUMMARY TABLE
# ============================================================

summary_points = pd.DataFrame({
    "Item": [
        "Source file",
        "Scoped rows",
        "Selected model",
        "Validation basis",
        "1M face cap applied",
        "Age_Ind filter",
        "Age-basis testing columns",
        "Attained_Age_min",
        "Attained_Age_max",
        "Overall A/P (full refit)",
        "Overall A/VBT (full refit)",
        "Optional development sample fraction",
        "Output Age_Ind",
        "Output Sex",
    ],
    "Value": [
        str(DATA_PATH),
        f"{len(scoped_df):,}",
        SELECTED_MODEL_KEY,
        diagnostic_label,
        str(CAP_FACE_AT_1M),
        str(AGE_IND_KEEP),
        "Issue_Age_mod / Attained_Age_mod used for age-related testing; Age_Ind still segmented explicitly",
        str(ATTAINED_AGE_MIN),
        str(ATTAINED_AGE_MAX),
        round(float(overall_validation.loc[0, "Actual_to_Predicted"]), 6),
        round(float(overall_validation.loc[0, "Actual_to_VBT"]), 6),
        str(DEVELOPMENT_SAMPLE_FRACTION),
        str(OUTPUT_AGE_IND),
        str(OUTPUT_SEX),
    ]
})

display(summary_points)
export_csv(summary_points, "session4_summary_points.csv")


In [ ]:
# SESSION 4.18 SHEET-STYLE MORTALITY OUTPUTS
# ============================================================

modeled_sheet_pred = build_sheet_style_table(
    scored_df,
    value_col="Predicted_Deaths",
    age_ind_value=OUTPUT_AGE_IND,
    sex_value=OUTPUT_SEX,
    select_period=OUTPUT_SELECT_PERIOD,
    ult_duration=OUTPUT_ULT_DURATION,
    duration_cols=OUTPUT_DURATION_COLUMNS,
    rate_per=OUTPUT_RATE_PER,
    issue_age_min=OUTPUT_ISSUE_AGE_MIN,
    issue_age_max=OUTPUT_ISSUE_AGE_MAX,
)

modeled_sheet_actual = build_sheet_style_table(
    scored_df,
    value_col=ACTUAL_CNT_COL,
    age_ind_value=OUTPUT_AGE_IND,
    sex_value=OUTPUT_SEX,
    select_period=OUTPUT_SELECT_PERIOD,
    ult_duration=OUTPUT_ULT_DURATION,
    duration_cols=OUTPUT_DURATION_COLUMNS,
    rate_per=OUTPUT_RATE_PER,
    issue_age_min=OUTPUT_ISSUE_AGE_MIN,
    issue_age_max=OUTPUT_ISSUE_AGE_MAX,
)

modeled_sheet_vbt = build_sheet_style_table(
    scored_df,
    value_col=EXPECTED_CNT_COL,
    age_ind_value=OUTPUT_AGE_IND,
    sex_value=OUTPUT_SEX,
    select_period=OUTPUT_SELECT_PERIOD,
    ult_duration=OUTPUT_ULT_DURATION,
    duration_cols=OUTPUT_DURATION_COLUMNS,
    rate_per=OUTPUT_RATE_PER,
    issue_age_min=OUTPUT_ISSUE_AGE_MIN,
    issue_age_max=OUTPUT_ISSUE_AGE_MAX,
)

print("Predicted mortality table (rate per 1000)")
display(modeled_sheet_pred)

print("\nActual mortality table (rate per 1000)")
display(modeled_sheet_actual)

print("\n2015 VBT expected table on the same data mix (rate per 1000)")
display(modeled_sheet_vbt)

sheet_stub = f"ageind_{OUTPUT_AGE_IND}_sex_{OUTPUT_SEX}"
export_csv(modeled_sheet_pred, f"session4_modeled_sheet_pred_{sheet_stub}.csv")
export_csv(modeled_sheet_actual, f"session4_modeled_sheet_actual_{sheet_stub}.csv")
export_csv(modeled_sheet_vbt, f"session4_modeled_sheet_vbt_{sheet_stub}.csv")


In [ ]:
# SESSION 4.19 OPTIONAL BENCHMARK COMPARISON TO A REFERENCE EXCEL SHEET
# ============================================================

if BENCHMARK_XLSX_PATH is not None and BENCHMARK_SHEET_NAME is not None:
    reference_sheet = read_reference_sheet(Path(BENCHMARK_XLSX_PATH), BENCHMARK_SHEET_NAME, max_issue_age=OUTPUT_ISSUE_AGE_MAX)
    benchmark_comparison = compare_to_reference(modeled_sheet_pred, reference_sheet)

    print("Reference sheet")
    display(reference_sheet)

    print("\nLargest modeled-minus-reference differences")
    display(benchmark_comparison.head(25))

    export_csv(reference_sheet, f"session4_reference_sheet_{sheet_stub}.csv")
    export_csv(benchmark_comparison, f"session4_benchmark_comparison_{sheet_stub}.csv")
else:
    print("No benchmark workbook / sheet supplied. Skip this cell unless you want an explicit table-to-table comparison.")


In [ ]:
# SESSION 4.20 CONCLUSION
n_review = int((worst_segments["Flag"] == "Review").sum())
n_issue_monotone_review = int((issue_age_monotonicity["Monotonicity_Flag"] == "Review").sum()) if len(issue_age_monotonicity) else 0
n_duration_monotone_review = int((duration_monotonicity["Monotonicity_Flag"] == "Review").sum()) if len(duration_monotonicity) else 0
overall_ap = float(overall_validation.loc[0, "Actual_to_Predicted"])

session4_conclusion = (
    f"Session 4 conclusion: the selected model {SELECTED_MODEL_KEY} produces an overall A/P of {overall_ap:.4f} "
    f"on the full-data refit. There are {n_review} segment tables currently flagged for review, "
    f"{n_issue_monotone_review} issue-age monotonicity points flagged, and {n_duration_monotone_review} duration monotonicity points flagged "
    f"after applying the credibility floor of {MIN_EXPECTED_FOR_MONOTONICITY:.1f} expected deaths. "
    f"The notebook now supports calibration review, monotonicity review, graph review, and direct sheet-style mortality table production "
    f"without converting Age_Ind to a single basis."
)

print(session4_conclusion)
